# Paper 2 — Goal 1 complete analysis

## Clinical information available in recording quality

**Scientific question.** Does the fixed recording-quality representation \(Q\) contain reproducible out-of-sample information about ALS diagnosis or contemporaneous ALSFRS-R bulbar function in unseen participants?

**Permitted interpretation.** Above-null performance means clinically structured information is available in \(Q\). It does not prove technical origin, spuriousness, confounding, or shortcut learning.

This notebook is the single authoritative Goal 1 implementation. It is designed to run from a fresh kernel from top to bottom and does not depend on variables or result files created by earlier failed Goal 1 notebooks.

It implements:

- the frozen diagnosis and ≤60-day severity index populations;
- the fixed participant 5-fold × 10-repeat outer split manifest;
- 5-fold participant-grouped inner tuning;
- fold-safe QCHAN reconstruction from training participants only;
- training-only transforms, QADD imputation/support handling, scaling, and hyperparameter tuning;
- Age, Support-only, QADD, QGAIN, QREV, QCHAN, Core-Q ridge, Age+Core-Q ridge, and Core-Q HGB;
- participant-cluster bootstrap uncertainty;
- full-pipeline participant-level permutation nulls for primary Core-Q ridge;
- native OOF calibration;
- prespecified Goal 1 sensitivities;
- machine-readable saved outputs and Nature/npj-style publication figures.

### Run modes

`RUN_MODE = "DEVELOPMENT"` executes the complete pipeline with reduced resampling counts so implementation can be verified.

`RUN_MODE = "FINAL"` uses the frozen inference counts:
- 2,000 participant bootstraps;
- 1,000 full-pipeline participant-level permutations.

Development results must never be copied into the manuscript as final inference.

## How to run this notebook

Use **Kernel → Restart Kernel and Run All Cells**. Do not mix cells from the earlier Goal 1 notebooks with this version. This rebuilt notebook writes to a new isolated output tree:

`outputs/goal1/goal1_complete_v1_1/<development|final>/`

The default is `RUN_MODE = "DEVELOPMENT"`. It executes the entire scientific workflow with reduced resampling counts and must end with `GOAL 1 DEVELOPMENT PIPELINE: PASS`. Only after that should `RUN_MODE` be changed to `"FINAL"` for the frozen 2,000-bootstrap / 1,000-permutation analysis.

`RESET_OUTPUTS = False` is intentionally restart-safe. On a first run there is nothing to reuse; after an interruption, only schema-validated checkpoints under the identical run signature are reused. Set it to `True` only when you deliberately want to discard this rebuilt notebook's checkpoints and recompute from zero.


In [1]:
from __future__ import annotations

import hashlib
import importlib.metadata as mdlib
import itertools
import json
import math
import os
import platform
import shutil
import subprocess
import sys
import tempfile
import warnings
from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from scipy import stats
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score,
)
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
from statsmodels.genmod.families import Binomial
from statsmodels.nonparametric.smoothers_lowess import lowess

# -------------------------------------------------------------------------
# USER-FACING EXECUTION CONFIGURATION
# -------------------------------------------------------------------------
RUN_MODE = "FINAL"   # change to "FINAL" only after development PASS
RESET_OUTPUTS = False      # restart-safe; set True only for a deliberate clean recomputation
RESUME_IF_VALID = True     # when RESET_OUTPUTS=False, reuse only schema-validated checkpoints
RUN_HGB = True
RUN_SENSITIVITIES = True
RUN_PERMUTATIONS = True
RUN_ELASTIC_NET = False    # optional in the frozen specification; not primary

BASE_SEED = 20260825
OUTER_FOLDS = 5
OUTER_REPEATS = 10
INNER_FOLDS = 5

if RUN_MODE not in {"DEVELOPMENT", "FINAL"}:
    raise ValueError("RUN_MODE must be 'DEVELOPMENT' or 'FINAL'.")

if RUN_MODE == "FINAL":
    N_BOOTSTRAPS = 2000
    N_PERMUTATIONS = 1000
    PERMUTATION_REPEATS = 10
else:
    # Complete code path, reduced resampling only.
    N_BOOTSTRAPS = 300
    N_PERMUTATIONS = 10
    PERMUTATION_REPEATS = 3

EXPECTED_PAPER1_COMMIT = "cb31fb6886df1b2b2fedba4ffbbf8624bd56d7e8"
ENGINE_VERSION = "goal1-complete-v1.1.0"
ENGINE_BUILD_SHA256 = "c6b7dcdea2ede265523549b745fcc3aaa4cfd6d6042cdf30af6b2cf3a1d6da4b"

EXPECTED = {
    "recordings": 519,
    "participants": 224,
    "als_participants": 158,
    "control_participants": 66,
    "als_recordings": 418,
    "control_recordings": 101,
    "severity_primary_participants": 145,
}

def find_project_root() -> Path:
    override = os.environ.get("PAPER2_ROOT", "").strip()
    if override:
        candidate = Path(override).expanduser().resolve()
        if (candidate / "data" / "processed").exists():
            return candidate
        raise FileNotFoundError(f"PAPER2_ROOT is invalid: {candidate}")

    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (
            (candidate / "data" / "processed").exists()
            and (candidate / "data" / "manifests").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "Could not locate the Paper 2 repository root. "
        "Open this notebook from the repository or notebooks/ folder."
    )

ROOT = find_project_root()
PROCESSED = ROOT / "data" / "processed"
MANIFESTS = ROOT / "data" / "manifests"
INTERIM = ROOT / "data" / "interim"
EXTERNAL = ROOT / "external" / "quality_framework_features"

RUN_TAG = RUN_MODE.lower()
OUT = ROOT / "outputs" / "goal1" / "goal1_complete_v1_1" / RUN_TAG
CHECKPOINTS = OUT / "checkpoints"
TABLES = OUT / "tables"
FIGURES = OUT / "figures"
OOF_DIR = OUT / "oof"
LOGS = OUT / "logs"

if RESET_OUTPUTS and OUT.exists():
    shutil.rmtree(OUT)

for directory in [OUT, CHECKPOINTS, TABLES, FIGURES, OOF_DIR, LOGS]:
    directory.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings("default")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

print("Paper 2 root:", ROOT)
print("Engine:", ENGINE_VERSION)
print("Run mode:", RUN_MODE)
print("Bootstrap replicates:", N_BOOTSTRAPS)
print("Permutation replicates:", N_PERMUTATIONS)
print("Permutation CV repeats:", PERMUTATION_REPEATS)

Paper 2 root: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code
Engine: goal1-complete-v1.1.0
Run mode: FINAL
Bootstrap replicates: 2000
Permutation replicates: 1000
Permutation CV repeats: 10


## 1. Atomic I/O and reproducibility utilities

A partial computation is never allowed to masquerade as a completed result. Tables are written to temporary files, validated, and atomically promoted only after successful completion.

In [2]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

def stable_hash(payload) -> str:
    text = json.dumps(payload, sort_keys=True, default=str, separators=(",", ":"))
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def atomic_write_json(payload, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
    os.replace(tmp, path)

def atomic_write_csv(
    frame: pd.DataFrame,
    path: Path,
    *,
    required_columns=None,
    allow_empty=False,
):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if not isinstance(frame, pd.DataFrame):
        raise TypeError(f"{path.name}: expected DataFrame.")
    if not allow_empty and len(frame) == 0:
        raise ValueError(f"{path.name}: refusing to write an empty result table.")
    if required_columns:
        missing = set(required_columns) - set(frame.columns)
        if missing:
            raise ValueError(f"{path.name}: missing required columns {sorted(missing)}")

    tmp = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(tmp, index=False)
    if tmp.stat().st_size == 0:
        raise IOError(f"{path.name}: temporary CSV is unexpectedly empty.")
    os.replace(tmp, path)

def safe_read_csv(path: Path, *, required_columns=None, allow_empty=False):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.stat().st_size == 0:
        raise IOError(f"{path} exists but is empty.")
    frame = pd.read_csv(path, low_memory=False)
    if not allow_empty and len(frame) == 0:
        raise ValueError(f"{path} contains no rows.")
    if required_columns:
        missing = set(required_columns) - set(frame.columns)
        if missing:
            raise ValueError(f"{path.name}: missing columns {sorted(missing)}")
    return frame

def package_version(name):
    try:
        return mdlib.version(name)
    except Exception:
        return "unknown"

def safe_name(text: str) -> str:
    return (
        str(text).lower()
        .replace(" + ", "_plus_")
        .replace("+", "plus")
        .replace("≤", "le")
        .replace(" ", "_")
        .replace("/", "_")
        .replace("–", "-")
    )

print("Atomic I/O utilities: READY")

Atomic I/O utilities: READY


## 2. Pre-modeling gates and canonical inputs

The denominator and leakage gates are rechecked in this notebook. Goal 1 does not proceed if the canonical 519-recording / 224-participant contract or the Phase 0.5 QCHAN reproduction gate is broken.

In [3]:
PATHS = {
    "recording_table": PROCESSED / "recording_table_phase0.csv",
    "participant_table": PROCESSED / "participant_table.csv",
    "diagnosis_index": PROCESSED / "goal1_diagnosis_index.csv",
    "severity_index": PROCESSED / "goal1_severity_index.csv",
    "severity_pairs": PROCESSED / "severity_pairs.csv",
    "split_manifest": MANIFESTS / "split_manifest.csv",
    "q_registry": MANIFESTS / "q_registry.csv",
    "phase0_report": MANIFESTS / "phase0_freeze_report.csv",
    "qchan_ready": MANIFESTS / "qchan_cache_ready.json",
    "qchan_cache_index": MANIFESTS / "qchan_spectrum_cache_index.csv",
}

missing_files = [name for name, path in PATHS.items() if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing Phase 0/0.5 artifacts: " + ", ".join(missing_files)
    )

recording_table = safe_read_csv(
    PATHS["recording_table"],
    required_columns=["participant_id", "logical_recording_id", "diagnosis"],
)
participant_table = safe_read_csv(
    PATHS["participant_table"],
    required_columns=["participant_id", "diagnosis"],
)
diagnosis_index = safe_read_csv(
    PATHS["diagnosis_index"],
    required_columns=[
        "participant_id", "logical_recording_id", "diagnosis",
        "age_at_recording_years",
    ],
)
severity_index = safe_read_csv(
    PATHS["severity_index"],
    required_columns=[
        "participant_id", "logical_recording_id", "bulbar_score",
        "abs_delta_days",
    ],
)
severity_pairs = safe_read_csv(
    PATHS["severity_pairs"],
    required_columns=[
        "participant_id", "logical_recording_id", "bulbar_score",
        "abs_delta_days", "within_60_days", "within_90_days",
    ],
)
split_manifest = safe_read_csv(
    PATHS["split_manifest"],
    required_columns=["participant_id", "repeat", "outer_fold"],
)
q_registry = safe_read_csv(
    PATHS["q_registry"],
    required_columns=["feature", "family", "paper2_role", "transform"],
)
phase0_report = safe_read_csv(PATHS["phase0_report"])
qchan_cache_index = safe_read_csv(
    PATHS["qchan_cache_index"],
    required_columns=[
        "logical_recording_id", "status", "spectrum_sha256", "cache_path"
    ],
)
qchan_ready = json.loads(PATHS["qchan_ready"].read_text(encoding="utf-8"))

# Canonical string identities.
for frame in [
    recording_table, participant_table, diagnosis_index,
    severity_index, severity_pairs, split_manifest, qchan_cache_index
]:
    if "participant_id" in frame.columns:
        frame["participant_id"] = frame["participant_id"].astype(str).str.strip()
    if "logical_recording_id" in frame.columns:
        frame["logical_recording_id"] = frame["logical_recording_id"].astype(str).str.strip()

# -------------------------------------------------------------------------
# Enrich frozen severity pair tables with canonical recording/Q predictors.
# -------------------------------------------------------------------------
# Phase 0 intentionally stores goal1_severity_index.csv and severity_pairs.csv
# as compact clinical-link tables. They contain the deterministic recording /
# assessment pairing, but not the Q feature columns. Every severity analysis
# therefore must join those pair rows back to recording_table_phase0.csv before
# any predictor construction, preflight, or model fit.

def enrich_severity_rows(pair_rows, *, validation):
    pair_rows = pair_rows.copy()

    required_pair = {
        "participant_id",
        "logical_recording_id",
        "bulbar_score",
        "abs_delta_days",
    }
    missing_pair = required_pair - set(pair_rows.columns)
    if missing_pair:
        raise ValueError(
            "Severity pair table is missing required columns: "
            + ", ".join(sorted(missing_pair))
        )

    # Preserve only pair-specific clinical-link fields from the compact table.
    pair_fields = [
        "participant_id",
        "logical_recording_id",
        "bulbar_score",
        "abs_delta_days",
    ]
    for optional in [
        "recording_date",
        "assessment_date",
        "alsfrs_total",
        "delta_days",
        "within_60_days",
        "within_90_days",
    ]:
        if optional in pair_rows.columns:
            pair_fields.append(optional)

    pair_meta = pair_rows[pair_fields].copy()

    # The canonical recording table supplies the acquisition/Q predictors.
    # Drop any historical clinical-link fields that could collide with the
    # frozen matched-pair outcome. The matched pair table remains authoritative
    # for bulbar_score, assessment date, and matching lag.
    clinical_collision_fields = {
        "bulbar_score",
        "alsfrs_total",
        "assessment_date",
        "delta_days",
        "abs_delta_days",
        "within_60_days",
        "within_90_days",
    }
    canonical = recording_table.drop(
        columns=[
            col for col in clinical_collision_fields
            if col in recording_table.columns
        ],
        errors="ignore",
    ).copy()

    # recording_date is already frozen in the pair table. Avoid suffixes.
    if "recording_date" in pair_meta.columns and "recording_date" in canonical.columns:
        canonical = canonical.drop(columns=["recording_date"])

    merged = pair_meta.merge(
        canonical,
        on=["participant_id", "logical_recording_id"],
        how="left",
        validate=validation,
        indicator="_recording_join",
    )

    if len(merged) != len(pair_meta):
        raise RuntimeError("Severity enrichment changed the number of pair rows.")
    if not merged["_recording_join"].eq("both").all():
        bad = merged.loc[
            ~merged["_recording_join"].eq("both"),
            ["participant_id", "logical_recording_id", "_recording_join"],
        ]
        raise RuntimeError(
            "Severity pair(s) did not match the canonical recording table:\n"
            + bad.head(20).to_string(index=False)
        )
    merged = merged.drop(columns=["_recording_join"])

    merged["y"] = pd.to_numeric(merged["bulbar_score"], errors="raise")
    if not merged["y"].between(0, 12).all():
        raise ValueError("Severity outcome lies outside the ALSFRS-R bulbar 0-12 range.")

    # Data-contract assertions: every enriched row must now expose every
    # Q feature that exists in the canonical recording table. This catches
    # schema regressions before nested CV begins.
    q_columns_in_recording = [
        feature
        for feature in q_registry["feature"].astype(str)
        if feature in recording_table.columns
    ]
    missing_q = [feature for feature in q_columns_in_recording if feature not in merged.columns]
    if missing_q:
        raise KeyError(
            "Enriched severity rows are missing canonical Q columns: "
            + ", ".join(missing_q)
        )

    if "age_at_recording_years" not in merged.columns:
        raise KeyError("Enriched severity rows are missing age_at_recording_years.")

    return merged

severity_index = enrich_severity_rows(
    severity_index,
    validation="one_to_one",
)
severity_pairs = enrich_severity_rows(
    severity_pairs,
    validation="many_to_one",
)

# Explicit regression guard for the failure reported in v1.0.1.
for required_feature in [
    "qadd_pause_ac_level_dbfs_median",
    "qadd_pause_level_iqr_db",
    "qadd_speech_pause_level_contrast_db",
    "qgain_typical_speech_level_dbfs",
    "qrev_srmr_norm",
]:
    if required_feature not in severity_index.columns:
        raise KeyError(
            f"Primary severity table lacks required predictor {required_feature!r} "
            "after canonical enrichment."
        )

# Diagnosis coding.
diagnosis_index["y"] = diagnosis_index["diagnosis"].map(
    {"ALS": 1, "CONTROLS": 0}
)
recording_table["y"] = recording_table["diagnosis"].map(
    {"ALS": 1, "CONTROLS": 0}
)
if diagnosis_index["y"].isna().any() or recording_table["y"].isna().any():
    raise ValueError("Unexpected diagnosis labels in the canonical data.")

# Hard denominators.
assert len(recording_table) == EXPECTED["recordings"]
assert recording_table["participant_id"].nunique() == EXPECTED["participants"]
assert len(diagnosis_index) == EXPECTED["participants"]
assert diagnosis_index["participant_id"].is_unique
assert int(diagnosis_index["y"].sum()) == EXPECTED["als_participants"]
assert int((diagnosis_index["y"] == 0).sum()) == EXPECTED["control_participants"]
assert int((recording_table["y"] == 1).sum()) == EXPECTED["als_recordings"]
assert int((recording_table["y"] == 0).sum()) == EXPECTED["control_recordings"]

severity_index["y"] = pd.to_numeric(
    severity_index["bulbar_score"], errors="raise"
)
assert len(severity_index) == EXPECTED["severity_primary_participants"]
assert severity_index["participant_id"].is_unique
assert severity_index["abs_delta_days"].le(60).all()
assert severity_index["y"].between(0, 12).all()

# Master split contract.
assert len(split_manifest) == EXPECTED["participants"] * OUTER_REPEATS
assert split_manifest["repeat"].nunique() == OUTER_REPEATS
assert split_manifest["outer_fold"].nunique() == OUTER_FOLDS
assert split_manifest.groupby(["participant_id", "repeat"]).size().eq(1).all()
assert set(split_manifest["participant_id"]) == set(diagnosis_index["participant_id"])

# QCHAN Phase 0.5 gate.
required_qchan_flags = [
    "all_media_paths_resolved",
    "all_media_hashes_verified",
    "all_spectra_measured",
    "paper1_release_reproduced",
]
for flag in required_qchan_flags:
    if qchan_ready.get(flag) is not True:
        raise RuntimeError(f"QCHAN readiness gate failed: {flag}")

if qchan_ready.get("paper1_repo_commit") != EXPECTED_PAPER1_COMMIT:
    raise RuntimeError("QCHAN cache was not generated from the pinned Paper 1 commit.")

if not (EXTERNAL / ".git").exists():
    raise FileNotFoundError(
        f"Pinned Paper 1 repository is missing at {EXTERNAL}"
    )

observed_paper1_commit = subprocess.check_output(
    ["git", "-C", str(EXTERNAL), "rev-parse", "HEAD"],
    text=True,
).strip()
if observed_paper1_commit != EXPECTED_PAPER1_COMMIT:
    raise RuntimeError(
        f"Paper 1 code commit mismatch. Expected {EXPECTED_PAPER1_COMMIT}, "
        f"observed {observed_paper1_commit}."
    )

paper1_src = EXTERNAL / "src"
if str(paper1_src) not in sys.path:
    sys.path.insert(0, str(paper1_src))

from paper1_qc_reviewed.qchan_v400 import (
    ANALYSIS_FEATURES as QCHAN_FEATURES_TUPLE,
    DEFAULT_PARAMETERS as QCHAN_PARAMETERS,
    build_subject_balanced_loso_references,
    compute_reference_relative_features,
)
from paper1_qc_reviewed.qchan_v400_cohort import load_recording_spectrum

QCHAN_FEATURES = list(QCHAN_FEATURES_TUPLE)

# Input hashes and run signature.
input_hashes = {
    name: sha256_file(path)
    for name, path in PATHS.items()
    if path.is_file()
}
run_contract = {
    "engine_version": ENGINE_VERSION,
    "engine_build_sha256": ENGINE_BUILD_SHA256,
    "run_mode": RUN_MODE,
    "base_seed": BASE_SEED,
    "outer_folds": OUTER_FOLDS,
    "outer_repeats": OUTER_REPEATS,
    "inner_folds": INNER_FOLDS,
    "n_bootstraps": N_BOOTSTRAPS,
    "n_permutations": N_PERMUTATIONS,
    "permutation_repeats": PERMUTATION_REPEATS,
    "paper1_commit": observed_paper1_commit,
    "input_hashes": input_hashes,
}
RUN_SIGNATURE = stable_hash(run_contract)

signature_path = OUT / "run_signature.json"
if signature_path.exists() and not RESET_OUTPUTS:
    existing = json.loads(signature_path.read_text(encoding="utf-8"))
    if existing.get("run_signature") != RUN_SIGNATURE:
        raise RuntimeError(
            "Existing output directory was produced under a different run contract. "
            "Set RESET_OUTPUTS=True or use a new output directory."
        )

atomic_write_json(
    {"run_signature": RUN_SIGNATURE, **run_contract},
    signature_path,
)

print("CANONICAL DATA / QCHAN / SPLIT GATES: PASS")
print("519 recordings | 224 participants | 158 ALS | 66 controls")
print("Primary severity participants:", len(severity_index))
print("Run signature:", RUN_SIGNATURE[:16])

CANONICAL DATA / QCHAN / SPLIT GATES: PASS
519 recordings | 224 participants | 158 ALS | 66 controls
Primary severity participants: 145
Run signature: cbcec778d60d573f


## 3. Freeze Core-Q, Extended-Q and demographic representations

Support indicators are generated only for the features that require them. This prevents the root failure in the previous notebook, where Age-only and Support-only models attempted to access QADD columns that had intentionally been omitted from a temporary frame.

In [4]:
QADD = [
    "qadd_pause_ac_level_dbfs_median",
    "qadd_pause_level_iqr_db",
    "qadd_speech_pause_level_contrast_db",
]
QGAIN = [
    "qgain_typical_speech_level_dbfs",
    "qgain_within_segment_iqr_db",
    "qgain_between_segment_mad_db",
    "qgain_abs_drift_db_per_min",
]
QREV = ["qrev_srmr_norm"]
QCHAN = list(QCHAN_FEATURES)
CORE_Q = QADD + QGAIN + QREV + QCHAN

EXTENDED_Q_EXTRA = [
    "qadd_pause_spectral_flatness",
    "qadd_mains_hum_comb_score_db",
    "qrev_tail_excess_100ms_db",
    "qrev_tail_persistence_median_sec",
    "qrev_downward_decay_rate_db_per_sec",
]
EXTENDED_Q = CORE_Q + EXTENDED_Q_EXTRA

AGE = "age_at_recording_years"
SEX_BINARY = "sex_binary"

TRANSFORM = dict(zip(q_registry["feature"], q_registry["transform"]))
for feature in CORE_Q + EXTENDED_Q_EXTRA:
    if feature not in TRANSFORM:
        raise ValueError(f"Frozen transform missing from q_registry.csv: {feature}")

# Validate source columns before modeling.
required_feature_columns = set(CORE_Q + EXTENDED_Q_EXTRA)
missing_feature_columns = required_feature_columns - set(recording_table.columns)
if missing_feature_columns:
    raise ValueError(
        "Canonical recording table is missing frozen Q columns: "
        + ", ".join(sorted(missing_feature_columns))
    )

# Participant-level sex source.
if "sex" not in participant_table.columns:
    if "Sex" in participant_table.columns:
        participant_table["sex"] = participant_table["Sex"]
    else:
        participant_table["sex"] = np.nan

def normalize_sex(value):
    if pd.isna(value):
        return np.nan
    token = str(value).strip().lower()
    if token in {"m", "male", "man", "1"}:
        return 1.0
    if token in {"f", "female", "woman", "0"}:
        return 0.0
    return np.nan

participant_table[SEX_BINARY] = participant_table["sex"].map(normalize_sex)

sex_lookup = participant_table[["participant_id", SEX_BINARY]].drop_duplicates()
if sex_lookup["participant_id"].duplicated().any():
    raise ValueError("Participant sex lookup is not one-to-one.")

def attach_sex(frame):
    out = frame.drop(columns=[SEX_BINARY], errors="ignore").merge(
        sex_lookup, on="participant_id", how="left", validate="many_to_one"
    )
    return out

diagnosis_index = attach_sex(diagnosis_index)
severity_index = attach_sex(severity_index)
recording_table = attach_sex(recording_table)
severity_pairs = attach_sex(severity_pairs)

# Persistence censor indicator required for Extended-Q.
PERSISTENCE_CENSOR = "qrev_persistence_recording_median_censored"
if PERSISTENCE_CENSOR not in recording_table.columns:
    status_col = "qrev_tail_persistence_median_sec_status"
    if status_col in recording_table.columns:
        recording_table[PERSISTENCE_CENSOR] = (
            recording_table[status_col]
            .astype(str)
            .str.contains("censor", case=False, na=False)
            .astype(float)
        )
    else:
        raise ValueError(
            "Extended-Q requires the QREV persistence censor indicator. "
            f"Neither {PERSISTENCE_CENSOR!r} nor {status_col!r} is present."
        )

# Propagate censor field to primary/sensitivity one-row frames by recording identity.
censor_lookup = recording_table[
    ["logical_recording_id", PERSISTENCE_CENSOR]
].drop_duplicates()

def attach_censor(frame):
    return frame.drop(columns=[PERSISTENCE_CENSOR], errors="ignore").merge(
        censor_lookup,
        on="logical_recording_id",
        how="left",
        validate="many_to_one",
    )

diagnosis_index = attach_censor(diagnosis_index)
severity_index = attach_censor(severity_index)
severity_pairs = attach_censor(severity_pairs)

# Model specification helper.
def model_spec(
    *,
    numeric,
    support_sources=(),
    binary_sources=(),
    qchan_presence=(),
    kind="ridge",
    requires_age=False,
    requires_sex=False,
):
    return {
        "numeric": list(numeric),
        "support_sources": list(support_sources),
        "binary_sources": list(binary_sources),
        "qchan_presence": list(qchan_presence),
        "kind": kind,
        "requires_age": bool(requires_age),
        "requires_sex": bool(requires_sex),
    }

PRIMARY_MODEL_SPECS = OrderedDict([
    ("Age", model_spec(numeric=[AGE], requires_age=True)),
    ("Support-only", model_spec(numeric=[], support_sources=QADD)),
    ("QADD", model_spec(numeric=QADD, support_sources=QADD)),
    ("QGAIN", model_spec(numeric=QGAIN)),
    ("QREV", model_spec(numeric=QREV)),
    ("QCHAN", model_spec(numeric=QCHAN)),
    ("Core-Q", model_spec(numeric=CORE_Q, support_sources=QADD)),
    (
        "Age + Core-Q",
        model_spec(
            numeric=[AGE] + CORE_Q,
            support_sources=QADD,
            requires_age=True,
        ),
    ),
    (
        "Core-Q HGB",
        model_spec(
            numeric=CORE_Q,
            support_sources=QADD,
            kind="hgb",
        ),
    ),
])

EXTENDED_SPEC = model_spec(
    numeric=EXTENDED_Q,
    support_sources=QADD + EXTENDED_Q_EXTRA,
    binary_sources=[PERSISTENCE_CENSOR],
)
QCHAN_TWO_PART_SPEC = model_spec(
    numeric=CORE_Q,
    support_sources=QADD,
    qchan_presence=[
        "qchan_rolloff95_deficit_hz",
        "qchan_highband_ratio_deficit",
        "qchan_tilt_steepening_db_per_oct",
    ],
)
AGE_SEX_SPEC = model_spec(
    numeric=[AGE],
    binary_sources=[SEX_BINARY],
    requires_age=True,
    requires_sex=True,
)
AGE_SEX_CORE_SPEC = model_spec(
    numeric=[AGE] + CORE_Q,
    support_sources=QADD,
    binary_sources=[SEX_BINARY],
    requires_age=True,
    requires_sex=True,
)

C_GRID = np.logspace(-4, 4, 9)
ALPHA_GRID = np.logspace(-4, 4, 9)
HGB_GRID = [
    {
        "max_depth": d,
        "learning_rate": lr,
        "min_samples_leaf": leaf,
        "l2_regularization": l2,
    }
    for d, lr, leaf, l2
    in itertools.product(
        [2, 3], [0.03, 0.10], [15, 30], [0.0, 1.0]
    )
]

display(pd.DataFrame([
    {
        "model": name,
        "numeric_n": len(spec["numeric"]),
        "support_n": len(spec["support_sources"]),
        "binary_n": len(spec["binary_sources"]),
        "estimator": spec["kind"],
    }
    for name, spec in PRIMARY_MODEL_SPECS.items()
]))

print("Q REPRESENTATION / MODEL CONTRACT: PASS")

,model,numeric_n,support_n,binary_n,estimator
0,Age,1,0,0,ridge
1,Support-only,0,3,0,ridge
2,QADD,3,3,0,ridge
3,QGAIN,4,0,0,ridge
4,QREV,1,0,0,ridge
5,QCHAN,4,0,0,ridge
6,Core-Q,12,3,0,ridge
7,Age + Core-Q,13,3,0,ridge
8,Core-Q HGB,12,3,0,hgb


Q REPRESENTATION / MODEL CONTRACT: PASS


## 4. Reference-independent QCHAN spectra and fold-safe reconstruction

The Phase 0.5 cache was already shown to reproduce the frozen Paper 1 QCHAN release. Here the cache is used only as the reference-independent spectral input. Every predictive QCHAN value is reconstructed using participants in the relevant training partition.

In [5]:
spectra = {}
for row in qchan_cache_index.itertuples(index=False):
    cache_path = Path(str(row.cache_path))
    if not cache_path.is_absolute():
        cache_path = ROOT / cache_path
    if not cache_path.exists():
        raise FileNotFoundError(f"QCHAN spectrum cache missing: {cache_path}")
    spectrum = load_recording_spectrum(cache_path)
    if spectrum.status != "measured":
        raise RuntimeError(
            f"QCHAN spectrum is not measured for {row.logical_recording_id}: "
            f"{spectrum.status}"
        )
    spectra[str(row.logical_recording_id)] = spectrum

if len(spectra) != EXPECTED["recordings"]:
    raise RuntimeError(
        f"Expected {EXPECTED['recordings']} cached QCHAN spectra, got {len(spectra)}."
    )

QCHAN_CACHE = OrderedDict()
QCHAN_CACHE_LIMIT = 1024

def qchan_cache_key(reference_participants, target_rows):
    targets = (
        target_rows[["participant_id", "logical_recording_id"]]
        .drop_duplicates()
        .sort_values(["participant_id", "logical_recording_id"])
    )
    payload = {
        "reference_participants": sorted(map(str, reference_participants)),
        "targets": targets.astype(str).to_dict("records"),
        "paper1_commit": observed_paper1_commit,
    }
    return stable_hash(payload)

def reference_metadata(reference_rows):
    meta = (
        reference_rows[["logical_recording_id", "participant_id"]]
        .drop_duplicates()
        .rename(columns={"participant_id": "subject_id"})
        .copy()
    )
    meta["logical_recording_id"] = meta["logical_recording_id"].astype(str)
    meta["subject_id"] = meta["subject_id"].astype(str)
    meta["task_stratum"] = "BAMBOO_PASSAGE"
    return meta

def fold_safe_qchan(reference_participant_ids, target_rows):
    """
    Recompute QCHAN using only reference_participant_ids.

    Training targets receive LOSO references.
    External validation/test targets receive one common training-only reference.
    """
    reference_participant_ids = set(map(str, reference_participant_ids))
    if not reference_participant_ids:
        raise ValueError("QCHAN reference participant set is empty.")

    target_unique = (
        target_rows[["participant_id", "logical_recording_id"]]
        .drop_duplicates()
        .copy()
    )
    target_unique["participant_id"] = target_unique["participant_id"].astype(str)
    target_unique["logical_recording_id"] = target_unique["logical_recording_id"].astype(str)

    # A recording identity may not belong to more than one participant.
    if target_unique["logical_recording_id"].duplicated().any():
        dup = target_unique.loc[
            target_unique["logical_recording_id"].duplicated(False)
        ]
        raise ValueError(
            "QCHAN target recording IDs map to multiple rows/participants:\n"
            + dup.to_string(index=False)
        )

    key = qchan_cache_key(reference_participant_ids, target_unique)
    if key in QCHAN_CACHE:
        cached = QCHAN_CACHE.pop(key)
        QCHAN_CACHE[key] = cached
        return cached.copy()

    reference_rows = recording_table.loc[
        recording_table["participant_id"].isin(reference_participant_ids),
        ["participant_id", "logical_recording_id"],
    ].drop_duplicates()

    observed_reference_participants = set(
        reference_rows["participant_id"].astype(str)
    )
    if observed_reference_participants != reference_participant_ids:
        missing = reference_participant_ids - observed_reference_participants
        raise ValueError(
            f"QCHAN reference participants missing canonical recordings: {sorted(missing)[:10]}"
        )

    ref_spectra = {
        rid: spectra[rid]
        for rid in reference_rows["logical_recording_id"].astype(str)
    }
    ref_meta = reference_metadata(reference_rows)

    training_references = build_subject_balanced_loso_references(
        ref_spectra,
        ref_meta,
        parameters=QCHAN_PARAMETERS,
    )

    external_targets = target_unique.loc[
        ~target_unique["participant_id"].isin(reference_participant_ids)
    ]
    external_reference = None

    if len(external_targets):
        dummy = external_targets.iloc[0]
        dummy_rid = str(dummy["logical_recording_id"])
        dummy_pid = str(dummy["participant_id"])

        combo_spectra = dict(ref_spectra)
        combo_spectra[dummy_rid] = spectra[dummy_rid]
        combo_meta = pd.concat(
            [
                ref_meta,
                pd.DataFrame([{
                    "logical_recording_id": dummy_rid,
                    "subject_id": dummy_pid,
                    "task_stratum": "BAMBOO_PASSAGE",
                }]),
            ],
            ignore_index=True,
        )
        combo_refs = build_subject_balanced_loso_references(
            combo_spectra,
            combo_meta,
            parameters=QCHAN_PARAMETERS,
        )
        external_reference = combo_refs[dummy_rid]

        members = set(map(str, external_reference.member_subject_ids))
        if not members.issubset(reference_participant_ids):
            raise RuntimeError("HELD-OUT QCHAN REFERENCE LEAKAGE DETECTED.")
        if dummy_pid in members:
            raise RuntimeError("External dummy participant entered its QCHAN reference.")

    output_rows = []
    for row in target_unique.itertuples(index=False):
        pid = str(row.participant_id)
        rid = str(row.logical_recording_id)

        if pid in reference_participant_ids:
            reference = training_references[rid]
            members = set(map(str, reference.member_subject_ids))
            if pid in members:
                raise RuntimeError(f"QCHAN LOSO failure for participant {pid}.")
            if not members.issubset(reference_participant_ids):
                raise RuntimeError("QCHAN training reference contains non-training participants.")
        else:
            if external_reference is None:
                raise RuntimeError("Missing external QCHAN reference.")
            reference = external_reference

        values = compute_reference_relative_features(
            spectra[rid],
            reference,
            parameters=QCHAN_PARAMETERS,
        )
        output_rows.append({
            "participant_id": pid,
            "logical_recording_id": rid,
            **{feature: values[feature] for feature in QCHAN},
        })

    result = pd.DataFrame(output_rows)
    if len(result) != len(target_unique):
        raise RuntimeError("QCHAN output row-count mismatch.")
    if result[QCHAN].isna().any().any():
        raise RuntimeError("Unexpected missing fold-safe QCHAN values.")

    QCHAN_CACHE[key] = result.copy()
    while len(QCHAN_CACHE) > QCHAN_CACHE_LIMIT:
        QCHAN_CACHE.popitem(last=False)

    return result

# Leakage smoke test on one outer fold.
smoke_test_ids = set(
    split_manifest.loc[
        (split_manifest["repeat"] == 1)
        & (split_manifest["outer_fold"] == 1),
        "participant_id",
    ].astype(str)
)
smoke_train = diagnosis_index.loc[
    ~diagnosis_index["participant_id"].isin(smoke_test_ids)
].copy()
smoke_test = diagnosis_index.loc[
    diagnosis_index["participant_id"].isin(smoke_test_ids)
].copy()

smoke_q_train = fold_safe_qchan(
    set(smoke_train["participant_id"]),
    smoke_train,
)
smoke_q_test = fold_safe_qchan(
    set(smoke_train["participant_id"]),
    smoke_test,
)

assert len(smoke_q_train) == len(smoke_train)
assert len(smoke_q_test) == len(smoke_test)
print("FOLD-SAFE QCHAN LEAKAGE SMOKE TEST: PASS")

FOLD-SAFE QCHAN LEAKAGE SMOKE TEST: PASS


## 5. Predictor construction and preprocessing

Support indicators are generated from the original un-imputed values before any numerical completion. Unavailable values are never converted to zero. Frozen transforms are applied exactly; only medians and scaling parameters are learned from training data.

In [6]:
def apply_frozen_transform(series, feature):
    x = pd.to_numeric(series, errors="coerce").astype(float)

    if feature == AGE:
        return x

    rule = TRANSFORM.get(feature)
    if rule is None:
        raise KeyError(f"No frozen transform for {feature}")

    if rule == "none":
        return x

    if rule == "log1p":
        finite = x[np.isfinite(x)]
        if len(finite) and finite.min() < -1e-8:
            raise ValueError(
                f"{feature}: negative value encountered for frozen log1p transform."
            )
        return np.log1p(x.clip(lower=0))

    if rule == "asinh":
        return np.arcsinh(x)

    raise ValueError(f"Unsupported frozen transform {rule!r} for {feature}")

def analysis_subset(frame, spec):
    out = frame.copy()

    if spec["requires_age"]:
        out = out.loc[pd.to_numeric(out[AGE], errors="coerce").notna()].copy()

    if spec["requires_sex"]:
        out = out.loc[pd.to_numeric(out[SEX_BINARY], errors="coerce").notna()].copy()

    if out["participant_id"].isna().any():
        raise ValueError("Analysis frame contains missing participant IDs.")

    return out.reset_index(drop=True)

def predictor_columns(spec):
    support_columns = [
        f"{feature}__supported"
        for feature in spec["support_sources"]
    ]
    presence_columns = [
        f"{feature}__present"
        for feature in spec["qchan_presence"]
    ]
    return (
        list(spec["numeric"])
        + support_columns
        + list(spec["binary_sources"])
        + presence_columns
    )

def build_predictor_frame(rows, spec, reference_participant_ids):
    """
    Construct raw predictors without using outcomes for feature definition.
    """
    base_cols = ["participant_id", "logical_recording_id", "y"]

    # Source columns required to construct support indicators must be present
    # even when they are not numeric predictors (e.g. Support-only model).
    source_features = set(spec["numeric"]) | set(spec["support_sources"])
    source_features -= set(QCHAN)
    source_features -= {AGE}

    for column in sorted(source_features):
        if column not in rows.columns:
            raise KeyError(
                f"Predictor source column {column!r} is missing from analysis rows."
            )

    extra_cols = sorted(source_features)
    if AGE in spec["numeric"]:
        extra_cols.append(AGE)
    for column in spec["binary_sources"]:
        if column not in rows.columns:
            raise KeyError(f"Binary predictor source {column!r} is missing.")
        extra_cols.append(column)

    local = rows[list(dict.fromkeys(base_cols + extra_cols))].copy()

    needs_qchan = any(feature in QCHAN for feature in spec["numeric"]) or bool(
        spec["qchan_presence"]
    )
    if needs_qchan:
        dynamic = fold_safe_qchan(reference_participant_ids, rows)
        local = local.merge(
            dynamic,
            on=["participant_id", "logical_recording_id"],
            how="left",
            validate="many_to_one",
        )

    # Support indicators are created from source availability before imputation.
    for feature in spec["support_sources"]:
        if feature not in local.columns:
            raise KeyError(
                f"Cannot construct support indicator: source feature {feature!r} absent."
            )
        local[f"{feature}__supported"] = local[feature].notna().astype(float)

    # QCHAN two-part sensitivity: presence + continuous magnitude.
    for feature in spec["qchan_presence"]:
        if feature not in local.columns:
            raise KeyError(f"QCHAN two-part source {feature!r} absent.")
        numeric = pd.to_numeric(local[feature], errors="coerce")
        if numeric.isna().any():
            raise ValueError(f"QCHAN two-part source {feature!r} has missing values.")
        local[f"{feature}__present"] = (numeric > 0).astype(float)

    required = predictor_columns(spec)
    missing = [column for column in required if column not in local.columns]
    if missing:
        raise KeyError(f"Constructed predictor frame is missing {missing}")

    return local[base_cols + required].copy()

def preprocess_train_eval(train_raw, eval_raw, spec):
    tr = train_raw.copy()
    ev = eval_raw.copy()

    numeric = list(spec["numeric"])
    imputed_numeric = set(spec["support_sources"])

    for feature in numeric:
        tr[feature] = apply_frozen_transform(tr[feature], feature)
        ev[feature] = apply_frozen_transform(ev[feature], feature)

    for feature in numeric:
        if feature in imputed_numeric:
            median = float(tr[feature].median(skipna=True))
            if not np.isfinite(median):
                raise ValueError(
                    f"{feature}: no finite training values available for median imputation."
                )
            tr[feature] = tr[feature].fillna(median)
            ev[feature] = ev[feature].fillna(median)
        else:
            if tr[feature].isna().any():
                raise ValueError(
                    f"Unexpected missing training values in primary numeric feature {feature}."
                )
            if ev[feature].isna().any():
                raise ValueError(
                    f"Unexpected missing held-out values in primary numeric feature {feature}."
                )

    columns = predictor_columns(spec)
    if not columns:
        raise ValueError("Model has no predictors.")

    tr_values = tr[columns].apply(pd.to_numeric, errors="coerce")
    ev_values = ev[columns].apply(pd.to_numeric, errors="coerce")

    if tr_values.isna().any().any() or ev_values.isna().any().any():
        raise ValueError("Non-numeric or missing predictor remained after preprocessing.")

    scaler = StandardScaler()
    X_train = scaler.fit_transform(tr_values.to_numpy(float))
    X_eval = scaler.transform(ev_values.to_numpy(float))

    if not np.isfinite(X_train).all() or not np.isfinite(X_eval).all():
        raise FloatingPointError("Non-finite standardized predictor encountered.")

    return X_train, X_eval

# Explicit unit tests for the exact bug that broke the previous notebook.
test_rows = diagnosis_index.head(12).copy()
test_reference_ids = set(
    diagnosis_index.loc[
        ~diagnosis_index["participant_id"].isin(test_rows["participant_id"]),
        "participant_id",
    ]
)
if not test_reference_ids:
    test_reference_ids = set(diagnosis_index["participant_id"])

age_test = build_predictor_frame(
    test_rows,
    PRIMARY_MODEL_SPECS["Age"],
    test_reference_ids,
)
support_test = build_predictor_frame(
    test_rows,
    PRIMARY_MODEL_SPECS["Support-only"],
    test_reference_ids,
)
qadd_test = build_predictor_frame(
    test_rows,
    PRIMARY_MODEL_SPECS["QADD"],
    test_reference_ids,
)

assert list(age_test.columns[-1:]) == [AGE]
assert all(
    f"{feature}__supported" in support_test.columns
    for feature in QADD
)
assert all(feature in qadd_test.columns for feature in QADD)

print("PREDICTOR CONSTRUCTION UNIT TESTS: PASS")

PREDICTOR CONSTRUCTION UNIT TESTS: PASS


## 6. Participant-grouped nested cross-validation engine

This section defines the exact grouped inner/outer fitting engine and then runs four real-data integration checks before any Goal 1 result is produced: Age diagnosis, Support-only diagnosis, Core-Q diagnosis, and Core-Q severity on one held-out outer fold. The tests exercise the same fold-safe preprocessing and QCHAN machinery used by the full run. If any fails, the notebook stops here and no result tables are produced.

The estimator calls avoid deprecated scikit-learn arguments so the notebook remains compatible with the current installed environment.

In [7]:
def participant_weights(frame):
    counts = frame.groupby("participant_id")["participant_id"].transform("size")
    weights = 1.0 / counts.astype(float)
    # Sum of weights per participant must be exactly 1 within numerical tolerance.
    check = pd.DataFrame({
        "participant_id": frame["participant_id"],
        "weight": weights,
    }).groupby("participant_id")["weight"].sum()
    if not np.allclose(check.to_numpy(), 1.0):
        raise RuntimeError("Participant-normalized weight construction failed.")
    return weights.to_numpy(float)

def participant_table_for_split(frame, task):
    grouped = frame.groupby("participant_id", sort=True)

    rows = []
    for participant_id, local in grouped:
        y_values = pd.to_numeric(local["y"], errors="raise").to_numpy(float)
        if task == "diagnosis":
            if len(np.unique(y_values)) != 1:
                raise ValueError(
                    f"Diagnosis is inconsistent within participant {participant_id}."
                )
            y_split = int(y_values[0])
        else:
            # Severity score is not used for stratification.
            y_split = float(np.mean(y_values))
        rows.append({"participant_id": participant_id, "y_split": y_split})
    return pd.DataFrame(rows)

def make_inner_partitions(frame, task, seed):
    participants = participant_table_for_split(frame, task)

    if len(participants) < INNER_FOLDS:
        raise ValueError("Too few participants for 5-fold inner CV.")

    if task == "diagnosis":
        class_counts = participants["y_split"].value_counts()
        if class_counts.min() < INNER_FOLDS:
            raise ValueError(
                f"Insufficient diagnosis class count for {INNER_FOLDS}-fold inner CV: "
                f"{class_counts.to_dict()}"
            )
        splitter = StratifiedKFold(
            n_splits=INNER_FOLDS,
            shuffle=True,
            random_state=seed,
        )
        iterator = splitter.split(
            participants["participant_id"],
            participants["y_split"].astype(int),
        )
    else:
        splitter = KFold(
            n_splits=INNER_FOLDS,
            shuffle=True,
            random_state=seed,
        )
        iterator = splitter.split(participants["participant_id"])

    partitions = []
    for inner_fold, (train_idx, val_idx) in enumerate(iterator, start=1):
        train_ids = set(
            participants.iloc[train_idx]["participant_id"].astype(str)
        )
        val_ids = set(
            participants.iloc[val_idx]["participant_id"].astype(str)
        )
        if train_ids & val_ids:
            raise RuntimeError("Inner participant leakage detected.")
        partitions.append((inner_fold, train_ids, val_ids))

    return partitions

def fit_estimator(task, kind, param, seed, X, y, sample_weight):
    if kind == "ridge":
        if task == "diagnosis":
            model = LogisticRegression(
                C=float(param),
                solver="lbfgs",
                max_iter=5000,
                class_weight=None,
                random_state=seed,
            )
        else:
            model = Ridge(alpha=float(param))
    elif kind == "hgb":
        kwargs = dict(param)
        if task == "diagnosis":
            model = HistGradientBoostingClassifier(
                loss="log_loss",
                max_iter=200,
                early_stopping=True,
                random_state=seed,
                **kwargs,
            )
        else:
            model = HistGradientBoostingRegressor(
                loss="squared_error",
                max_iter=200,
                early_stopping=True,
                random_state=seed,
                **kwargs,
            )
    else:
        raise ValueError(f"Unknown model kind: {kind}")

    model.fit(X, y, sample_weight=sample_weight)
    return model

def model_predict(model, task, X):
    if task == "diagnosis":
        p = model.predict_proba(X)[:, 1]
        return np.clip(p.astype(float), 1e-8, 1 - 1e-8)
    return model.predict(X).astype(float)

def prediction_frame(rows, predictions, model_name, task, repeat, outer_fold):
    if len(rows) != len(predictions):
        raise RuntimeError("Prediction length mismatch.")

    output = rows[
        ["participant_id", "logical_recording_id", "y"]
    ].copy()
    output["model"] = model_name
    output["task"] = task
    output["repeat"] = int(repeat)
    output["outer_fold"] = int(outer_fold)
    output["prediction"] = np.asarray(predictions, dtype=float)
    return output

def score_predictions(predictions, task):
    """
    Compute a repeat-level metric object.

    Diagnosis: aggregate recording predictions within participant first.
    Severity: participant-normalized pair weighting.
    """
    if task == "diagnosis":
        participant = (
            predictions.groupby("participant_id", as_index=False)
            .agg(
                y=("y", "first"),
                prediction=("prediction", "mean"),
            )
        )
        y = participant["y"].to_numpy(float)
        p = participant["prediction"].to_numpy(float)

        if len(np.unique(y)) < 2:
            raise ValueError("Diagnosis scoring requires both outcome classes.")

        return {
            "AUROC": float(roc_auc_score(y, p)),
            "Brier": float(brier_score_loss(y, p)),
            "AUPRC": float(average_precision_score(y, p)),
            "observed_prevalence": float(np.mean(y)),
        }

    local = predictions.copy()
    local["abs_error"] = np.abs(local["y"] - local["prediction"])
    local["sq_error"] = (local["y"] - local["prediction"]) ** 2

    by_participant = local.groupby("participant_id", as_index=False).agg(
        mae=("abs_error", "mean"),
        mse=("sq_error", "mean"),
        y_mean=("y", "mean"),
        prediction_mean=("prediction", "mean"),
    )

    mae = float(by_participant["mae"].mean())
    rmse = float(np.sqrt(by_participant["mse"].mean()))
    rho = stats.spearmanr(
        by_participant["y_mean"],
        by_participant["prediction_mean"],
        nan_policy="omit",
    ).statistic

    # Participant-weighted pair-level cross-validated R2.
    weights = participant_weights(local)
    y = local["y"].to_numpy(float)
    p = local["prediction"].to_numpy(float)
    ybar = float(np.average(y, weights=weights))
    numerator = float(np.sum(weights * (y - p) ** 2))
    denominator = float(np.sum(weights * (y - ybar) ** 2))
    r2 = np.nan if denominator <= 0 else 1.0 - numerator / denominator

    return {
        "MAE": mae,
        "RMSE": rmse,
        "Spearman_rho": float(rho) if np.isfinite(rho) else np.nan,
        "R2": float(r2) if np.isfinite(r2) else np.nan,
    }

def exact_tuning_loss(predictions, task):
    if task == "diagnosis":
        participant = (
            predictions.groupby("participant_id", as_index=False)
            .agg(y=("y", "first"), prediction=("prediction", "mean"))
        )
        y = participant["y"].to_numpy(int)
        p = np.clip(participant["prediction"].to_numpy(float), 1e-8, 1 - 1e-8)
        return float(log_loss(y, p, labels=[0, 1]))

    local = predictions.copy()
    local["abs_error"] = np.abs(local["y"] - local["prediction"])
    return float(
        local.groupby("participant_id")["abs_error"].mean().mean()
    )

def tuning_grid(spec, task):
    if spec["kind"] == "hgb":
        return HGB_GRID
    return C_GRID if task == "diagnosis" else ALPHA_GRID

def tune_model(train_frame, spec, task, seed):
    partitions = make_inner_partitions(train_frame, task, seed)

    fold_matrices = []
    split_log_rows = []

    for inner_fold, inner_train_ids, inner_val_ids in partitions:
        inner_train = train_frame.loc[
            train_frame["participant_id"].isin(inner_train_ids)
        ].copy()
        inner_val = train_frame.loc[
            train_frame["participant_id"].isin(inner_val_ids)
        ].copy()

        if set(inner_train["participant_id"]) & set(inner_val["participant_id"]):
            raise RuntimeError("Inner participant leakage.")

        raw_train = build_predictor_frame(
            inner_train, spec, inner_train_ids
        )
        raw_val = build_predictor_frame(
            inner_val, spec, inner_train_ids
        )
        X_train, X_val = preprocess_train_eval(
            raw_train, raw_val, spec
        )

        y_train = inner_train["y"].to_numpy(float)
        w_train = participant_weights(inner_train)

        fold_matrices.append({
            "inner_fold": inner_fold,
            "X_train": X_train,
            "y_train": y_train,
            "w_train": w_train,
            "X_val": X_val,
            "val_rows": inner_val,
        })

        for pid in sorted(inner_val_ids):
            split_log_rows.append({
                "inner_fold": inner_fold,
                "participant_id": pid,
            })

    scores = []
    grid = tuning_grid(spec, task)

    for param_index, param in enumerate(grid):
        losses = []

        for bundle in fold_matrices:
            model = fit_estimator(
                task,
                spec["kind"],
                param,
                seed + 1000 * (param_index + 1) + bundle["inner_fold"],
                bundle["X_train"],
                bundle["y_train"],
                bundle["w_train"],
            )
            pred = model_predict(
                model,
                task,
                bundle["X_val"],
            )
            pred_frame = prediction_frame(
                bundle["val_rows"],
                pred,
                "inner",
                task,
                repeat=0,
                outer_fold=bundle["inner_fold"],
            )
            losses.append(exact_tuning_loss(pred_frame, task))

        scores.append({
            "parameter": param,
            "mean_inner_loss": float(np.mean(losses)),
        })

    best_index = int(
        np.argmin([row["mean_inner_loss"] for row in scores])
    )
    return (
        scores[best_index]["parameter"],
        scores[best_index]["mean_inner_loss"],
        pd.DataFrame(scores),
        pd.DataFrame(split_log_rows),
    )

def validate_oof(oof, frame, task, repeats):
    expected_ids = set(frame["participant_id"].astype(str))
    if set(oof["participant_id"].astype(str)) != expected_ids:
        raise RuntimeError("OOF participant coverage mismatch.")
    if oof["repeat"].nunique() != repeats:
        raise RuntimeError("OOF repeat count mismatch.")

    # Every row must be predicted once per repeat.
    expected_rows = len(frame) * repeats
    if len(oof) != expected_rows:
        raise RuntimeError(
            f"OOF row count mismatch: expected {expected_rows}, got {len(oof)}."
        )

    key_counts = oof.groupby(
        ["repeat", "participant_id", "logical_recording_id"]
    ).size()
    if not key_counts.eq(1).all():
        raise RuntimeError("Duplicate/missing OOF row predictions detected.")

def run_nested_cv(
    frame,
    model_name,
    spec,
    task,
    *,
    repeats=OUTER_REPEATS,
    record_splits=False,
):
    data = analysis_subset(frame, spec)

    if len(data) == 0:
        raise ValueError(f"{model_name}: analysis subset is empty.")

    oof_rows = []
    fit_rows = []
    coef_rows = []
    split_rows = []

    for repeat in range(1, repeats + 1):
        for outer_fold in range(1, OUTER_FOLDS + 1):
            test_ids = set(
                split_manifest.loc[
                    (split_manifest["repeat"] == repeat)
                    & (split_manifest["outer_fold"] == outer_fold),
                    "participant_id",
                ].astype(str)
            )

            test = data.loc[data["participant_id"].isin(test_ids)].copy()
            train = data.loc[~data["participant_id"].isin(test_ids)].copy()

            if len(train) == 0 or len(test) == 0:
                raise ValueError(
                    f"{task}/{model_name}/repeat {repeat}/fold {outer_fold}: "
                    "empty train or test set."
                )
            if set(train["participant_id"]) & set(test["participant_id"]):
                raise RuntimeError("Outer participant leakage detected.")

            seed = BASE_SEED + repeat * 10000 + outer_fold * 100

            selected, inner_loss, _, inner_split_log = tune_model(
                train,
                spec,
                task,
                seed,
            )

            reference_ids = set(train["participant_id"].astype(str))
            raw_train = build_predictor_frame(
                train, spec, reference_ids
            )
            raw_test = build_predictor_frame(
                test, spec, reference_ids
            )
            X_train, X_test = preprocess_train_eval(
                raw_train, raw_test, spec
            )

            final_model = fit_estimator(
                task,
                spec["kind"],
                selected,
                seed + 999999,
                X_train,
                train["y"].to_numpy(float),
                participant_weights(train),
            )
            predictions = model_predict(final_model, task, X_test)

            fold_oof = prediction_frame(
                test,
                predictions,
                model_name,
                task,
                repeat,
                outer_fold,
            )
            oof_rows.append(fold_oof)

            fit_rows.append({
                "model": model_name,
                "task": task,
                "repeat": repeat,
                "outer_fold": outer_fold,
                "n_train_rows": len(train),
                "n_test_rows": len(test),
                "n_train_participants": train["participant_id"].nunique(),
                "n_test_participants": test["participant_id"].nunique(),
                "selected_parameter": (
                    json.dumps(selected, sort_keys=True)
                    if isinstance(selected, dict)
                    else float(selected)
                ),
                "best_inner_loss": float(inner_loss),
            })

            if spec["kind"] == "ridge":
                coefficients = np.asarray(final_model.coef_).reshape(-1)
                for feature, value in zip(
                    predictor_columns(spec), coefficients
                ):
                    coef_rows.append({
                        "model": model_name,
                        "task": task,
                        "repeat": repeat,
                        "outer_fold": outer_fold,
                        "feature": feature,
                        "standardized_coefficient": float(value),
                    })

            if record_splits:
                for pid in sorted(test["participant_id"].unique()):
                    split_rows.append({
                        "task": task,
                        "model": model_name,
                        "repeat": repeat,
                        "outer_fold": outer_fold,
                        "inner_fold": 0,
                        "participant_id": pid,
                        "role": "outer_test",
                        "seed": seed,
                    })
                for row in inner_split_log.itertuples(index=False):
                    split_rows.append({
                        "task": task,
                        "model": model_name,
                        "repeat": repeat,
                        "outer_fold": outer_fold,
                        "inner_fold": int(row.inner_fold),
                        "participant_id": str(row.participant_id),
                        "role": "inner_validation",
                        "seed": seed,
                    })

        print(
            f"{task:9s} | {model_name:18s} | "
            f"repeat {repeat}/{repeats} complete"
        )

    oof = pd.concat(oof_rows, ignore_index=True)
    fits = pd.DataFrame(fit_rows)
    coefs = pd.DataFrame(coef_rows)
    splits = pd.DataFrame(split_rows)

    validate_oof(oof, data, task, repeats)

    return oof, fits, coefs, splits


def preflight_one_outer_fold(frame, spec, task, label):
    """
    Exercise the exact nested-tuning -> preprocessing -> final-fit path on the
    first outer fold. This is a real-data integration test, not a result.
    """
    data = analysis_subset(frame, spec)
    test_ids = set(
        split_manifest.loc[
            (split_manifest["repeat"].eq(1))
            & (split_manifest["outer_fold"].eq(1)),
            "participant_id",
        ].astype(str)
    )
    train = data.loc[~data["participant_id"].isin(test_ids)].copy()
    test = data.loc[data["participant_id"].isin(test_ids)].copy()

    if len(train) == 0 or len(test) == 0:
        raise RuntimeError(f"{label}: preflight outer fold is empty.")
    if set(train["participant_id"]) & set(test["participant_id"]):
        raise RuntimeError(f"{label}: participant leakage in preflight.")

    seed = BASE_SEED + 991_001
    selected, _, _, _ = tune_model(train, spec, task, seed)

    reference_ids = set(train["participant_id"].astype(str))
    raw_train = build_predictor_frame(train, spec, reference_ids)
    raw_test = build_predictor_frame(test, spec, reference_ids)
    X_train, X_test = preprocess_train_eval(raw_train, raw_test, spec)

    fitted = fit_estimator(
        task,
        spec["kind"],
        selected,
        seed + 1,
        X_train,
        train["y"].to_numpy(float),
        participant_weights(train),
    )
    pred = model_predict(fitted, task, X_test)
    scored = score_predictions(
        prediction_frame(test, pred, label, task, 1, 1),
        task,
    )

    if not scored:
        raise RuntimeError(f"{label}: preflight produced no metrics.")
    return {
        "test": label,
        "task": task,
        "n_train_participants": train["participant_id"].nunique(),
        "n_test_participants": test["participant_id"].nunique(),
        "selected_parameter": selected,
        "status": "PASS",
    }

# Interface-level HGB check for the installed scikit-learn version.
_hgb_X = np.column_stack([
    np.linspace(-1, 1, 50),
    np.sin(np.linspace(-2, 2, 50)),
])
_hgb_y = np.array([0, 1] * 25)
_hgb_w = np.ones(50)
_hgb_check = HistGradientBoostingClassifier(
    loss="log_loss",
    max_iter=5,
    max_depth=2,
    min_samples_leaf=5,
    early_stopping=False,
    random_state=BASE_SEED,
)
_hgb_check.fit(_hgb_X, _hgb_y, sample_weight=_hgb_w)
assert _hgb_check.predict_proba(_hgb_X[:3]).shape == (3, 2)

# Real-data integration checks. Age and Support-only explicitly guard the
# failure mode of the previous notebook; Core-Q checks nested fold-safe QCHAN.
preflight_rows = [
    preflight_one_outer_fold(
        diagnosis_index,
        PRIMARY_MODEL_SPECS["Age"],
        "diagnosis",
        "Age diagnosis preflight",
    ),
    preflight_one_outer_fold(
        diagnosis_index,
        PRIMARY_MODEL_SPECS["Support-only"],
        "diagnosis",
        "Support-only diagnosis preflight",
    ),
    preflight_one_outer_fold(
        diagnosis_index,
        PRIMARY_MODEL_SPECS["Core-Q"],
        "diagnosis",
        "Core-Q diagnosis preflight",
    ),
    preflight_one_outer_fold(
        severity_index,
        PRIMARY_MODEL_SPECS["Core-Q"],
        "severity",
        "Core-Q severity preflight",
    ),
]
preflight_table = pd.DataFrame(preflight_rows)
atomic_write_csv(
    preflight_table,
    TABLES / "goal1_preflight_integration_tests.csv",
)
display(preflight_table)
print("REAL-DATA MODEL INTEGRATION PREFLIGHT: PASS")

print("NESTED-CV ENGINE: READY")

,test,task,n_train_participants,n_test_participants,selected_parameter,status
0,Age diagnosis preflight,diagnosis,161,38,1.0000,PASS
1,Support-only diagnosis preflight,diagnosis,179,45,0.0001,PASS
2,Core-Q diagnosis preflight,diagnosis,179,45,0.1000,PASS
3,Core-Q severity preflight,severity,116,29,10.0000,PASS


REAL-DATA MODEL INTEGRATION PREFLIGHT: PASS
NESTED-CV ENGINE: READY


## 7. Metrics, bootstrap uncertainty, calibration and checkpoint validation

In [8]:
def repeat_metric_table(oof, task):
    rows = []
    for repeat, local in oof.groupby("repeat"):
        rows.append({
            "repeat": int(repeat),
            **score_predictions(local, task),
        })
    return pd.DataFrame(rows)

def mean_metric_dict(oof, task):
    table = repeat_metric_table(oof, task)
    output = {}
    for column in table.columns:
        if column == "repeat":
            continue
        values = pd.to_numeric(table[column], errors="coerce")
        output[column] = float(values.mean(skipna=True))
    return output

def participant_repeat_predictions(oof, task):
    if task == "diagnosis":
        return (
            oof.groupby(["participant_id", "repeat"], as_index=False)
            .agg(
                y=("y", "first"),
                prediction=("prediction", "mean"),
            )
        )

    # For primary severity this remains one row; for repeated sensitivities
    # it creates a participant-mean descriptive summary.
    return (
        oof.groupby(["participant_id", "repeat"], as_index=False)
        .agg(
            y=("y", "mean"),
            prediction=("prediction", "mean"),
        )
    )

def final_participant_predictions(oof, task):
    repeated = participant_repeat_predictions(oof, task)
    return (
        repeated.groupby("participant_id", as_index=False)
        .agg(
            y=("y", "mean"),
            prediction=("prediction", "mean"),
            prediction_sd=("prediction", "std"),
        )
    )

def _bootstrap_repeat_summaries(oof, task):
    """
    Pre-aggregate each CV repeat to participant-level sufficient statistics.

    This preserves the participant-cluster bootstrap estimand while avoiding
    repeated DataFrame expansion/groupby work inside thousands of replicates.
    """
    summaries = {}
    participant_order = None

    for repeat, local in oof.groupby("repeat", sort=True):
        if task == "diagnosis":
            summary = (
                local.groupby("participant_id", as_index=False)
                .agg(
                    y=("y", "first"),
                    prediction=("prediction", "mean"),
                )
                .sort_values("participant_id")
                .reset_index(drop=True)
            )
        else:
            work = local.copy()
            work["abs_error"] = np.abs(
                work["y"] - work["prediction"]
            )
            work["sq_error"] = (
                work["y"] - work["prediction"]
            ) ** 2
            work["y_squared"] = work["y"] ** 2
            summary = (
                work.groupby("participant_id", as_index=False)
                .agg(
                    mae=("abs_error", "mean"),
                    mse=("sq_error", "mean"),
                    y_mean=("y", "mean"),
                    y2_mean=("y_squared", "mean"),
                    prediction_mean=("prediction", "mean"),
                )
                .sort_values("participant_id")
                .reset_index(drop=True)
            )

        participants = summary["participant_id"].astype(str).to_numpy()
        if participant_order is None:
            participant_order = participants
        elif not np.array_equal(participant_order, participants):
            raise RuntimeError(
                "Participant order/coverage differs across CV repeats."
            )

        summaries[int(repeat)] = summary

    if participant_order is None or len(participant_order) == 0:
        raise ValueError("Cannot bootstrap an empty OOF object.")

    return participant_order, summaries

def _metric_from_bootstrap_summary(summary, sampled_index, task, metric):
    if task == "diagnosis":
        y = summary["y"].to_numpy(float)[sampled_index]
        prediction = summary["prediction"].to_numpy(float)[sampled_index]

        if metric == "AUROC":
            if len(np.unique(y)) < 2:
                return np.nan
            return float(roc_auc_score(y, prediction))
        if metric == "Brier":
            return float(brier_score_loss(y, prediction))
        if metric == "AUPRC":
            if len(np.unique(y)) < 2:
                return np.nan
            return float(average_precision_score(y, prediction))
        raise ValueError((task, metric))

    mae = summary["mae"].to_numpy(float)[sampled_index]
    mse = summary["mse"].to_numpy(float)[sampled_index]
    y_mean = summary["y_mean"].to_numpy(float)[sampled_index]
    y2_mean = summary["y2_mean"].to_numpy(float)[sampled_index]
    pred_mean = summary["prediction_mean"].to_numpy(float)[sampled_index]

    if metric == "MAE":
        return float(np.mean(mae))
    if metric == "RMSE":
        return float(np.sqrt(np.mean(mse)))
    if metric == "Spearman_rho":
        rho = stats.spearmanr(
            y_mean, pred_mean, nan_policy="omit"
        ).statistic
        return float(rho) if np.isfinite(rho) else np.nan
    if metric == "R2":
        ybar = float(np.mean(y_mean))
        numerator = float(np.sum(mse))
        denominator = float(
            np.sum(y2_mean - 2.0 * ybar * y_mean + ybar ** 2)
        )
        if denominator <= 0:
            return np.nan
        return float(1.0 - numerator / denominator)

    raise ValueError((task, metric))

def bootstrap_metric_distribution(oof, task, metric, B, seed):
    participants, summaries = _bootstrap_repeat_summaries(oof, task)
    n = len(participants)
    rng = np.random.default_rng(seed)
    values = np.empty(B, dtype=float)

    completed = 0
    attempts = 0
    max_attempts = max(B * 20, B + 100)

    while completed < B:
        attempts += 1
        if attempts > max_attempts:
            raise RuntimeError(
                f"Could not obtain {B} finite bootstrap replicates for "
                f"{task}/{metric} after {max_attempts} attempts."
            )

        sampled_index = rng.integers(0, n, size=n)
        per_repeat = [
            _metric_from_bootstrap_summary(
                summary,
                sampled_index,
                task,
                metric,
            )
            for _, summary in sorted(summaries.items())
        ]

        if np.all(np.isfinite(per_repeat)):
            values[completed] = float(np.mean(per_repeat))
            completed += 1

    return values

def bootstrap_ci(values):
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]
    if len(finite) == 0:
        return np.nan, np.nan
    return (
        float(np.quantile(finite, 0.025)),
        float(np.quantile(finite, 0.975)),
    )

def paired_bootstrap_difference(
    oof_a,
    oof_b,
    task,
    metric,
    B,
    seed,
):
    participants_a, summaries_a = _bootstrap_repeat_summaries(oof_a, task)
    participants_b, summaries_b = _bootstrap_repeat_summaries(oof_b, task)

    common = sorted(set(participants_a) & set(participants_b))
    if not common:
        raise ValueError("No participants overlap for paired bootstrap.")

    # Restrict/reorder each repeat to the same paired participant order.
    def aligned(summary, task):
        indexed = summary.copy()
        indexed["participant_id"] = indexed["participant_id"].astype(str)
        indexed = indexed.set_index("participant_id")
        missing = set(common) - set(indexed.index)
        if missing:
            raise RuntimeError(
                "Paired bootstrap participant missing from a CV repeat."
            )
        return indexed.loc[common].reset_index()

    aligned_a = {
        repeat: aligned(summary, task)
        for repeat, summary in summaries_a.items()
    }
    aligned_b = {
        repeat: aligned(summary, task)
        for repeat, summary in summaries_b.items()
    }

    repeats = sorted(set(aligned_a) & set(aligned_b))
    if not repeats:
        raise ValueError("No CV repeats overlap for paired bootstrap.")

    n = len(common)
    rng = np.random.default_rng(seed)
    differences = np.empty(B, dtype=float)

    completed = 0
    attempts = 0
    max_attempts = max(B * 20, B + 100)

    while completed < B:
        attempts += 1
        if attempts > max_attempts:
            raise RuntimeError(
                f"Could not obtain {B} finite paired bootstrap replicates."
            )

        sampled_index = rng.integers(0, n, size=n)
        repeat_diffs = []

        for repeat in repeats:
            ma = _metric_from_bootstrap_summary(
                aligned_a[repeat],
                sampled_index,
                task,
                metric,
            )
            mb = _metric_from_bootstrap_summary(
                aligned_b[repeat],
                sampled_index,
                task,
                metric,
            )
            if not np.isfinite(ma) or not np.isfinite(mb):
                repeat_diffs = []
                break
            repeat_diffs.append(mb - ma)

        if repeat_diffs:
            differences[completed] = float(np.mean(repeat_diffs))
            completed += 1

    return differences

def safe_calibration_statistics(oof):
    participant = final_participant_predictions(oof, "diagnosis")
    y = participant["y"].to_numpy(float)
    p = np.clip(
        participant["prediction"].to_numpy(float),
        1e-6,
        1 - 1e-6,
    )
    logit_p = np.log(p / (1 - p))

    result = {
        "calibration_intercept": np.nan,
        "calibration_slope": np.nan,
        "joint_calibration_intercept": np.nan,
        "calibration_status": "not_estimated",
    }

    try:
        joint = sm.GLM(
            y,
            sm.add_constant(logit_p),
            family=Binomial(),
        ).fit()
        citl = sm.GLM(
            y,
            np.ones((len(y), 1)),
            family=Binomial(),
            offset=logit_p,
        ).fit()

        result.update({
            "calibration_intercept": float(citl.params[0]),
            "calibration_slope": float(joint.params[1]),
            "joint_calibration_intercept": float(joint.params[0]),
            "calibration_status": "ok",
        })
    except Exception as exc:
        result["calibration_status"] = (
            f"{type(exc).__name__}: {exc}"
        )

    return result

def validate_model_checkpoint(path, frame, task, repeats):
    try:
        oof = safe_read_csv(
            path,
            required_columns=[
                "participant_id", "logical_recording_id", "y",
                "prediction", "repeat", "outer_fold", "model", "task",
            ],
        )
        validate_oof(
            oof,
            analysis_subset(frame, {"requires_age": False, "requires_sex": False}),
            task,
            repeats,
        )
        return oof
    except Exception:
        return None

print("METRICS / BOOTSTRAP / CALIBRATION UTILITIES: READY")

METRICS / BOOTSTRAP / CALIBRATION UTILITIES: READY


## 8. Prepare primary and sensitivity analysis frames

All sensitivity rows are derived deterministically from the frozen Phase 0 ledgers; no clinical result determines inclusion.

In [9]:
# Primary one-row frames are already frozen by Phase 0.
dx_primary = diagnosis_index.copy()
sev_primary = severity_index.copy()

# All-recording diagnosis sensitivity.
dx_all_recordings = recording_table.copy()

def enrich_severity_pair_rows(pair_rows, *, validation):
    """Idempotently enrich severity pair rows from the canonical recording table."""
    # Reuse the single authoritative enrichment function defined at data load.
    # Strip any previously attached canonical recording columns first so a
    # sensitivity table can be re-enriched without suffixes or stale predictors.
    pair_columns = [
        col for col in [
            "participant_id", "logical_recording_id", "bulbar_score",
            "abs_delta_days", "recording_date", "assessment_date",
            "alsfrs_total", "delta_days", "within_60_days", "within_90_days",
        ]
        if col in pair_rows.columns
    ]
    compact = pair_rows[pair_columns].copy()
    return enrich_severity_rows(compact, validation=validation)

# All matched <=60-day severity-pair sensitivity.
sev_all_60_source = severity_pairs.loc[
    severity_pairs["within_60_days"].astype(bool)
].copy()
sev_all_60 = enrich_severity_pair_rows(
    sev_all_60_source,
    validation="many_to_one",
)

# <=90-day one-row severity sensitivity.
# The frozen pair table already contains the nearest assessment for each
# recording. We now choose the earliest eligible recording per participant
# using the same deterministic date -> recording-ID rule as the primary run.
sev_90_candidates = severity_pairs.loc[
    severity_pairs["within_90_days"].astype(bool)
].copy()

if "recording_date" not in sev_90_candidates.columns:
    date_lookup = recording_table[
        ["logical_recording_id", "recording_date"]
    ].drop_duplicates()
    sev_90_candidates = sev_90_candidates.merge(
        date_lookup,
        on="logical_recording_id",
        how="left",
        validate="many_to_one",
    )

sev_90_candidates["recording_date"] = pd.to_datetime(
    sev_90_candidates["recording_date"],
    errors="raise",
)

if sev_90_candidates["recording_date"].isna().any():
    raise ValueError("The <=90-day sensitivity contains a missing recording date.")

sev_90_index_source = (
    sev_90_candidates
    .sort_values(
        ["participant_id", "recording_date", "logical_recording_id"]
    )
    .drop_duplicates("participant_id", keep="first")
    .copy()
)

sev_90 = enrich_severity_pair_rows(
    sev_90_index_source,
    validation="one_to_one",
)

# All required demographic/censor fields must already be available through
# the frozen one-row tables or the canonical recording table.
for frame_name, frame in [
    ("dx_primary", dx_primary),
    ("sev_primary", sev_primary),
    ("dx_all_recordings", dx_all_recordings),
    ("sev_all_60", sev_all_60),
    ("sev_90", sev_90),
]:
    if SEX_BINARY not in frame.columns:
        raise KeyError(f"{frame_name}: missing attached sex field {SEX_BINARY!r}.")
    if PERSISTENCE_CENSOR not in frame.columns:
        raise KeyError(
            f"{frame_name}: missing QREV censor field {PERSISTENCE_CENSOR!r}."
        )

# Sanity checks.
assert dx_primary["participant_id"].nunique() == 224
assert len(dx_all_recordings) == 519
assert dx_all_recordings["participant_id"].nunique() == 224
assert sev_primary["participant_id"].nunique() == 145
assert sev_all_60["participant_id"].nunique() >= sev_primary["participant_id"].nunique()
assert sev_90["participant_id"].nunique() >= sev_primary["participant_id"].nunique()
assert sev_90["participant_id"].is_unique
assert sev_90["y"].between(0, 12).all()

population_summary = pd.DataFrame([
    {
        "analysis": "Primary diagnosis index",
        "rows": len(dx_primary),
        "participants": dx_primary["participant_id"].nunique(),
    },
    {
        "analysis": "Primary severity <=60 d",
        "rows": len(sev_primary),
        "participants": sev_primary["participant_id"].nunique(),
    },
    {
        "analysis": "Diagnosis all recordings sensitivity",
        "rows": len(dx_all_recordings),
        "participants": dx_all_recordings["participant_id"].nunique(),
    },
    {
        "analysis": "Severity all <=60 d pairs sensitivity",
        "rows": len(sev_all_60),
        "participants": sev_all_60["participant_id"].nunique(),
    },
    {
        "analysis": "Severity <=90 d index sensitivity",
        "rows": len(sev_90),
        "participants": sev_90["participant_id"].nunique(),
    },
])

atomic_write_csv(
    population_summary,
    TABLES / "goal1_population_summary.csv",
)
display(population_summary)

print("PRIMARY / SENSITIVITY POPULATIONS: PASS")


,analysis,rows,participants
0,Primary diagnosis index,224,224
1,Primary severity <=60 d,145,145
2,Diagnosis all recordings sensitivity,519,224
3,Severity all <=60 d pairs sensitivity,398,145
4,Severity <=90 d index sensitivity,145,145


PRIMARY / SENSITIVITY POPULATIONS: PASS


## 9. Safe observed-model runner

Each completed model is checkpointed independently. A checkpoint is reused only when its schema and participant/row coverage validate under the current run signature.

In [10]:
def model_checkpoint_paths(task, model_name):
    stem = f"{task}__{safe_name(model_name)}"
    return {
        "oof": OOF_DIR / f"{stem}.csv",
        "fits": CHECKPOINTS / f"{stem}__fits.csv",
        "coefs": CHECKPOINTS / f"{stem}__coefs.csv",
        "splits": CHECKPOINTS / f"{stem}__splits.csv",
    }

def load_valid_model_checkpoint(frame, model_name, task, repeats):
    paths = model_checkpoint_paths(task, model_name)
    if not RESUME_IF_VALID or not paths["oof"].exists():
        return None
    try:
        oof = safe_read_csv(
            paths["oof"],
            required_columns=[
                "participant_id", "logical_recording_id", "y",
                "prediction", "repeat", "outer_fold", "model", "task",
            ],
        )
        validate_oof(
            oof,
            analysis_subset(
                frame,
                PRIMARY_MODEL_SPECS.get(
                    model_name,
                    {"requires_age": False, "requires_sex": False},
                ),
            ),
            task,
            repeats,
        )
        return oof
    except Exception as exc:
        print(
            f"Ignoring invalid checkpoint for {task}/{model_name}: "
            f"{type(exc).__name__}: {exc}"
        )
        return None

def run_or_load_model(
    frame,
    model_name,
    spec,
    task,
    *,
    repeats=OUTER_REPEATS,
    record_splits=False,
):
    paths = model_checkpoint_paths(task, model_name)

    # Only primary model names use automatic checkpoint validation by spec.
    if paths["oof"].exists() and RESUME_IF_VALID:
        try:
            oof = safe_read_csv(
                paths["oof"],
                required_columns=[
                    "participant_id", "logical_recording_id", "y",
                    "prediction", "repeat", "outer_fold", "model", "task",
                ],
            )
            validate_oof(
                oof,
                analysis_subset(frame, spec),
                task,
                repeats,
            )
            print(f"Loaded valid checkpoint: {task} | {model_name}")
            return oof
        except Exception as exc:
            print(
                f"Checkpoint rejected; recomputing {task}/{model_name}: "
                f"{type(exc).__name__}: {exc}"
            )

    oof, fits, coefs, splits = run_nested_cv(
        frame,
        model_name,
        spec,
        task,
        repeats=repeats,
        record_splits=record_splits,
    )

    # Validate before any write.
    validate_oof(oof, analysis_subset(frame, spec), task, repeats)

    atomic_write_csv(
        oof,
        paths["oof"],
        required_columns=[
            "participant_id", "logical_recording_id", "y",
            "prediction", "repeat", "outer_fold", "model", "task",
        ],
    )
    atomic_write_csv(fits, paths["fits"])
    if len(coefs):
        atomic_write_csv(coefs, paths["coefs"])
    if len(splits):
        atomic_write_csv(splits, paths["splits"])

    return oof

print("CHECKPOINTED MODEL RUNNER: READY")

CHECKPOINTED MODEL RUNNER: READY


## 10. Full-pipeline permutation engine with restart-safe checkpoints

The fixed outer split manifest is retained. Outcomes are shuffled at participant level. Preprocessing, inner tuning, fold-specific QCHAN construction and ridge fitting are rerun for every permutation.

The permutation null is scoped to Core-Q ridge only. In FINAL mode, all 10 repeated outer CV partitions and 1,000 permutations are used. Development mode follows the same code path with reduced counts and is not inferential.

In [11]:
def permute_outcomes_one_row(frame, seed):
    out = frame.copy()
    if not out["participant_id"].is_unique:
        raise ValueError(
            "Formal Goal 1 permutation null is defined on the one-row "
            "primary participant population."
        )
    rng = np.random.default_rng(seed)
    out["y"] = rng.permutation(out["y"].to_numpy(float))
    return out

def permutation_checkpoint_path(task):
    return CHECKPOINTS / f"{task}__coreq_permutations.csv"

def run_permutation_null(frame, task, observed_oof):
    if not RUN_PERMUTATIONS:
        return pd.DataFrame()

    checkpoint = permutation_checkpoint_path(task)
    required_cols = ["permutation", "null_metric"]

    if checkpoint.exists() and RESUME_IF_VALID:
        try:
            existing = safe_read_csv(
                checkpoint,
                required_columns=required_cols,
                allow_empty=True,
            )
        except Exception:
            existing = pd.DataFrame(columns=required_cols)
    else:
        existing = pd.DataFrame(columns=required_cols)

    done = set(
        pd.to_numeric(
            existing.get("permutation", pd.Series(dtype=float)),
            errors="coerce",
        ).dropna().astype(int)
    )

    primary_metric_name = "AUROC" if task == "diagnosis" else "MAE"

    # Observed statistic must use the same repeat subset as the null.
    observed_subset = observed_oof.loc[
        observed_oof["repeat"].between(1, PERMUTATION_REPEATS)
    ].copy()
    observed_metric = float(
        repeat_metric_table(observed_subset, task)[primary_metric_name].mean()
    )

    rows = existing[required_cols].to_dict("records")

    for permutation in range(1, N_PERMUTATIONS + 1):
        if permutation in done:
            continue

        seed_offset = 10_000_000 if task == "diagnosis" else 20_000_000
        permuted = permute_outcomes_one_row(
            frame,
            BASE_SEED + seed_offset + permutation,
        )

        null_oof, _, _, _ = run_nested_cv(
            permuted,
            f"Core-Q permutation {permutation}",
            PRIMARY_MODEL_SPECS["Core-Q"],
            task,
            repeats=PERMUTATION_REPEATS,
            record_splits=False,
        )
        null_metric = float(
            repeat_metric_table(null_oof, task)[primary_metric_name].mean()
        )

        rows.append({
            "permutation": permutation,
            "null_metric": null_metric,
        })

        checkpoint_frame = (
            pd.DataFrame(rows)
            .sort_values("permutation")
            .drop_duplicates("permutation", keep="last")
        )
        atomic_write_csv(
            checkpoint_frame,
            checkpoint,
            required_columns=required_cols,
            allow_empty=False,
        )

        print(
            f"{task} permutation {permutation}/{N_PERMUTATIONS} "
            f"complete | {primary_metric_name}={null_metric:.4f}"
        )

    null = safe_read_csv(
        checkpoint,
        required_columns=required_cols,
    ).sort_values("permutation")

    if len(null) != N_PERMUTATIONS:
        raise RuntimeError(
            f"{task}: expected {N_PERMUTATIONS} completed permutations, "
            f"found {len(null)}."
        )

    if task == "diagnosis":
        extreme = int(
            (null["null_metric"] >= observed_metric).sum()
        )
    else:
        extreme = int(
            (null["null_metric"] <= observed_metric).sum()
        )

    empirical_p = (1 + extreme) / (1 + N_PERMUTATIONS)
    null["task"] = task
    null["metric"] = primary_metric_name
    null["observed_metric"] = observed_metric
    null["empirical_p"] = empirical_p
    null["permutation_repeats"] = PERMUTATION_REPEATS
    null["run_mode"] = RUN_MODE

    return null

print("PERMUTATION ENGINE: READY")

PERMUTATION ENGINE: READY


## 11. Figure style

The figure layer follows current Nature/npj conventions: final-size dimensions, editable sans-serif text, 5–7 pt labels, 8 pt bold lowercase panel letters, visible axes/ticks, no background gridlines, and vector PDF/SVG export. Figures are generated only from validated machine-readable result tables.

In [12]:
MM_TO_IN = 1 / 25.4
DOUBLE_COLUMN_IN = 183 * MM_TO_IN
SINGLE_COLUMN_IN = 89 * MM_TO_IN
MAX_HEIGHT_IN = 170 * MM_TO_IN

available_fonts = {f.name for f in font_manager.fontManager.ttflist}
FIGURE_FONT = (
    "Arial"
    if "Arial" in available_fonts
    else "Helvetica"
    if "Helvetica" in available_fonts
    else "DejaVu Sans"
)

mpl.rcParams.update({
    "font.family": FIGURE_FONT,
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "legend.fontsize": 6,
    "axes.linewidth": 0.7,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
})

def clean_axis(ax):
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="out")

def panel_label(ax, label):
    ax.text(
        -0.12,
        1.04,
        label,
        transform=ax.transAxes,
        fontsize=8,
        fontweight="bold",
        va="bottom",
        ha="left",
    )

def save_figure(fig, stem, source_tables):
    if fig.get_figheight() > MAX_HEIGHT_IN * 1.05:
        raise ValueError(
            f"{stem}: figure exceeds target Nature maximum height."
        )

    outputs = {}
    for suffix, kwargs in [
        (".pdf", {}),
        (".svg", {}),
        (".png", {"dpi": 600}),
    ]:
        path = FIGURES / f"{stem}{suffix}"
        fig.savefig(path, bbox_inches="tight", **kwargs)
        outputs[suffix] = {
            "file": path.name,
            "sha256": sha256_file(path),
        }
    plt.close(fig)

    atomic_write_json(
        {
            "stem": stem,
            "run_mode": RUN_MODE,
            "engine_version": ENGINE_VERSION,
            "created_utc": datetime.now(timezone.utc).isoformat(),
            "font": FIGURE_FONT,
            "source_tables": source_tables,
            "outputs": outputs,
        },
        FIGURES / f"{stem}.provenance.json",
    )

print("FIGURE STYLE: READY | font:", FIGURE_FONT)

FIGURE STYLE: READY | font: Arial


## 12. End-to-end Goal 1 runner

This cell is the only execution cell for the statistical analysis. Run the notebook from a fresh kernel using **Restart Kernel and Run All**. If any stage fails, execution stops before downstream result files or figures are produced.

The runner:
1. fits/loads all observed primary models;
2. computes bootstrap CIs and paired age contrasts;
3. calculates native calibration;
4. runs/restarts the full Core-Q permutation nulls;
5. runs the prespecified sensitivities;
6. writes required OOF/metric/permutation/split artifacts;
7. builds Figure 2 and Supplementary Figures S1–S5 plus the descriptive Q-age figure;
8. seals a run manifest only after all required stages succeed.

In [13]:
def summarize_model_metrics(results, task):
    rows = []
    for model_name, oof in results.items():
        row = {
            "model": model_name,
            "task": task,
            "n_rows_per_repeat": int(
                len(oof.loc[oof["repeat"].eq(1)])
            ),
            "n_participants": int(
                oof["participant_id"].nunique()
            ),
            **mean_metric_dict(oof, task),
        }
        if task == "diagnosis":
            row.update(safe_calibration_statistics(oof))
        rows.append(row)
    return pd.DataFrame(rows)

def bootstrap_model_table(results, task):
    metrics = (
        ["AUROC", "Brier", "AUPRC"]
        if task == "diagnosis"
        else ["MAE", "RMSE", "Spearman_rho", "R2"]
    )
    rows = []

    for model_index, (model_name, oof) in enumerate(results.items()):
        observed = mean_metric_dict(oof, task)
        for metric_index, metric in enumerate(metrics):
            dist = bootstrap_metric_distribution(
                oof,
                task,
                metric,
                N_BOOTSTRAPS,
                BASE_SEED
                + 100_000
                + model_index * 100
                + metric_index,
            )
            lo, hi = bootstrap_ci(dist)
            rows.append({
                "model": model_name,
                "task": task,
                "metric": metric,
                "estimate": observed.get(metric, np.nan),
                "ci_low": lo,
                "ci_high": hi,
                "bootstrap_replicates": N_BOOTSTRAPS,
                "n_participants": oof["participant_id"].nunique(),
            })

    return pd.DataFrame(rows)

def make_wide_primary_oof(results, task):
    rows = []
    for model_name, oof in results.items():
        repeated = participant_repeat_predictions(oof, task)
        pivot = repeated.pivot(
            index="participant_id",
            columns="repeat",
            values="prediction",
        )
        true_y = repeated.groupby("participant_id")["y"].mean()
        local = pd.DataFrame({
            "participant_id": pivot.index.astype(str),
            "model": model_name,
            "true_outcome": true_y.reindex(pivot.index).to_numpy(float),
        })
        for repeat in range(1, OUTER_REPEATS + 1):
            local[f"repeat_{repeat}_oof"] = pivot[repeat].to_numpy(float)
        local["mean_oof_prediction"] = pivot.mean(axis=1).to_numpy(float)
        rows.append(local)
    return pd.concat(rows, ignore_index=True)

def fit_severity_mean_baseline(frame):
    rows = []
    data = frame.copy()

    for repeat in range(1, OUTER_REPEATS + 1):
        for fold in range(1, OUTER_FOLDS + 1):
            test_ids = set(
                split_manifest.loc[
                    (split_manifest["repeat"].eq(repeat))
                    & (split_manifest["outer_fold"].eq(fold)),
                    "participant_id",
                ].astype(str)
            )
            train = data.loc[~data["participant_id"].isin(test_ids)].copy()
            test = data.loc[data["participant_id"].isin(test_ids)].copy()
            weights = participant_weights(train)
            mean_y = float(np.average(train["y"], weights=weights))
            pred = np.full(len(test), mean_y, dtype=float)
            rows.append(
                prediction_frame(
                    test, pred, "Mean baseline", "severity", repeat, fold
                )
            )
    oof = pd.concat(rows, ignore_index=True)
    validate_oof(oof, data, "severity", OUTER_REPEATS)
    return oof

def run_sensitivity(
    label,
    frame,
    spec,
    task,
):
    return run_or_load_model(
        frame,
        label,
        spec,
        task,
        repeats=OUTER_REPEATS,
        record_splits=False,
    )

def descriptive_q_age_table():
    rows = []
    scopes = [
        (
            "All age-complete",
            dx_primary.loc[dx_primary[AGE].notna()],
        ),
        (
            "Controls age-complete",
            dx_primary.loc[
                dx_primary[AGE].notna()
                & dx_primary["diagnosis"].eq("CONTROLS")
            ],
        ),
    ]

    rng_master = np.random.default_rng(BASE_SEED + 30_000_000)

    for feature in CORE_Q:
        for scope_name, subset in scopes:
            local = subset[[feature, AGE]].dropna()
            if len(local) < 10:
                rows.append({
                    "feature": feature,
                    "scope": scope_name,
                    "n": len(local),
                    "spearman_rho": np.nan,
                    "ci_low": np.nan,
                    "ci_high": np.nan,
                })
                continue

            rho = stats.spearmanr(
                local[feature], local[AGE], nan_policy="omit"
            ).statistic

            boot = []
            values = local.to_numpy(float)
            for _ in range(1000):
                idx = rng_master.integers(
                    0, len(values), size=len(values)
                )
                sampled = values[idx]
                r = stats.spearmanr(
                    sampled[:, 0],
                    sampled[:, 1],
                    nan_policy="omit",
                ).statistic
                if np.isfinite(r):
                    boot.append(float(r))

            lo, hi = bootstrap_ci(np.asarray(boot))
            rows.append({
                "feature": feature,
                "scope": scope_name,
                "n": len(local),
                "spearman_rho": (
                    float(rho) if np.isfinite(rho) else np.nan
                ),
                "ci_low": lo,
                "ci_high": hi,
            })

    return pd.DataFrame(rows)

def generate_figures(
    dx_results,
    sev_results,
    dx_ci,
    sev_ci,
    dx_perm,
    sev_perm,
    sensitivity_table,
    calibration_curves,
    q_age,
):
    # ------------------------------------------------------------------
    # Supplementary Figure S1: cohort flow
    # ------------------------------------------------------------------
    fig, ax = plt.subplots(
        figsize=(DOUBLE_COLUMN_IN, 75 * MM_TO_IN),
        constrained_layout=True,
    )
    ax.axis("off")

    boxes = [
        (0.03, 0.62, "Source Bamboo recordings\nn = 573"),
        (0.28, 0.62, "Retained recordings\nn = 519"),
        (0.53, 0.62, "Retained participants\nn = 224"),
        (0.78, 0.75, "ALS\nn = 158"),
        (0.78, 0.49, "Controls\nn = 66"),
        (0.53, 0.18, f"Primary bulbar analysis\nALS n = {len(sev_primary)}"),
    ]
    for x, y, text in boxes:
        ax.text(
            x, y, text,
            transform=ax.transAxes,
            ha="left", va="center",
            bbox={"boxstyle": "round,pad=0.35", "fill": False, "linewidth": 0.7},
        )

    arrows = [
        ((0.20, 0.62), (0.27, 0.62)),
        ((0.45, 0.62), (0.52, 0.62)),
        ((0.70, 0.62), (0.77, 0.75)),
        ((0.70, 0.62), (0.77, 0.49)),
        ((0.62, 0.52), (0.62, 0.28)),
    ]
    for start, end in arrows:
        ax.annotate(
            "",
            xy=end, xytext=start,
            xycoords=ax.transAxes,
            arrowprops={"arrowstyle": "->", "linewidth": 0.7},
        )

    save_figure(
        fig,
        f"FigureS1_cohort_flow_{RUN_TAG}",
        ["goal1_population_summary.csv"],
    )

    # ------------------------------------------------------------------
    # Main Figure 2
    # ------------------------------------------------------------------
    order = [
        "Age", "Support-only", "QADD", "QGAIN", "QREV", "QCHAN",
        "Core-Q", "Age + Core-Q",
    ] + (["Core-Q HGB"] if "Core-Q HGB" in dx_results else [])

    diag_plot = (
        dx_ci.loc[dx_ci["metric"].eq("AUROC")]
        .set_index("model")
        .loc[order]
        .reset_index()
    )
    sev_order = ["Mean baseline"] + order
    sev_plot = (
        sev_ci.loc[sev_ci["metric"].eq("MAE")]
        .set_index("model")
        .loc[sev_order]
        .reset_index()
    )

    atomic_write_csv(
        diag_plot,
        TABLES / "figure2a_diagnosis_source.csv",
    )
    atomic_write_csv(
        sev_plot,
        TABLES / "figure2c_severity_source.csv",
    )

    fig, axes = plt.subplots(
        2, 2,
        figsize=(DOUBLE_COLUMN_IN, 158 * MM_TO_IN),
        constrained_layout=True,
    )

    ax = axes[0, 0]
    y = np.arange(len(diag_plot))
    x = diag_plot["estimate"].to_numpy(float)
    err = np.vstack([
        np.maximum(0.0, x - diag_plot["ci_low"].to_numpy(float)),
        np.maximum(0.0, diag_plot["ci_high"].to_numpy(float) - x),
    ])
    ax.errorbar(
        x, y, xerr=err,
        fmt="o", markersize=3.5, linewidth=0.8, capsize=2,
    )
    ax.axvline(0.5, linestyle="--", linewidth=0.7)
    ax.set_yticks(y)
    ax.set_yticklabels([
        f"{m} (n={int(n)})"
        for m, n in zip(
            diag_plot["model"], diag_plot["n_participants"]
        )
    ])
    ax.invert_yaxis()
    ax.set_xlabel("AUROC")
    ax.set_xlim(0.35, 1.0)
    clean_axis(ax)
    panel_label(ax, "a")

    ax = axes[0, 1]
    ax.hist(
        dx_perm["null_metric"],
        bins=min(30, max(8, len(dx_perm) // 20)),
        density=True,
    )
    observed = float(dx_perm["observed_metric"].iloc[0])
    ax.axvline(observed, linewidth=1.0)
    ax.axvline(0.5, linestyle="--", linewidth=0.7)
    ax.set_xlabel("Core-Q AUROC under participant-level null")
    ax.set_ylabel("Density")
    ax.text(
        0.98, 0.96,
        f"Observed = {observed:.3f}\n"
        f"Empirical P = {float(dx_perm['empirical_p'].iloc[0]):.4f}\n"
        f"B = {len(dx_perm)}",
        transform=ax.transAxes,
        ha="right", va="top",
    )
    clean_axis(ax)
    panel_label(ax, "b")

    ax = axes[1, 0]
    y = np.arange(len(sev_plot))
    x = sev_plot["estimate"].to_numpy(float)
    err = np.vstack([
        np.maximum(0.0, x - sev_plot["ci_low"].to_numpy(float)),
        np.maximum(0.0, sev_plot["ci_high"].to_numpy(float) - x),
    ])
    ax.errorbar(
        x, y, xerr=err,
        fmt="o", markersize=3.5, linewidth=0.8, capsize=2,
    )
    ax.set_yticks(y)
    ax.set_yticklabels([
        f"{m} (n={int(n)})"
        for m, n in zip(
            sev_plot["model"], sev_plot["n_participants"]
        )
    ])
    ax.invert_yaxis()
    ax.set_xlabel("MAE (ALSFRS-R bulbar points)")
    clean_axis(ax)
    panel_label(ax, "c")

    ax = axes[1, 1]
    ax.hist(
        sev_perm["null_metric"],
        bins=min(30, max(8, len(sev_perm) // 20)),
        density=True,
    )
    observed = float(sev_perm["observed_metric"].iloc[0])
    ax.axvline(observed, linewidth=1.0)
    ax.set_xlabel("Core-Q MAE under participant-level null (points)")
    ax.set_ylabel("Density")
    ax.text(
        0.98, 0.96,
        f"Observed = {observed:.3f}\n"
        f"Empirical P = {float(sev_perm['empirical_p'].iloc[0]):.4f}\n"
        f"B = {len(sev_perm)}",
        transform=ax.transAxes,
        ha="right", va="top",
    )
    clean_axis(ax)
    panel_label(ax, "d")

    save_figure(
        fig,
        f"Figure2_Goal1_information_availability_{RUN_TAG}",
        [
            "figure2a_diagnosis_source.csv",
            "goal1_permutation.csv",
            "figure2c_severity_source.csv",
        ],
    )

    # ------------------------------------------------------------------
    # Supplementary Figure S2: support/availability map
    # ------------------------------------------------------------------
    availability = dx_primary[
        ["participant_id", "diagnosis"] + CORE_Q
    ].copy()
    support = availability[CORE_Q].notna().astype(int)
    availability["n_available"] = support.sum(axis=1)
    row_order = (
        availability.assign(
            diagnosis_sort=availability["diagnosis"].map(
                {"CONTROLS": 0, "ALS": 1}
            )
        )
        .sort_values(
            ["diagnosis_sort", "n_available", "participant_id"]
        )
        .index
    )
    matrix = support.loc[row_order].to_numpy()

    feature_availability = pd.DataFrame({
        "feature": CORE_Q,
        "n_available": support.sum(axis=0).to_numpy(),
        "fraction_available": support.mean(axis=0).to_numpy(),
    })
    atomic_write_csv(
        feature_availability,
        TABLES / "figureS2_feature_availability_source.csv",
    )

    fig, ax = plt.subplots(
        figsize=(DOUBLE_COLUMN_IN, 95 * MM_TO_IN),
        constrained_layout=True,
    )
    ax.imshow(matrix, aspect="auto", interpolation="nearest")
    ax.set_xticks(np.arange(len(CORE_Q)))
    ax.set_xticklabels(CORE_Q, rotation=45, ha="right")
    ax.set_ylabel("Participants, grouped by diagnosis")
    ax.set_xlabel("Core-Q feature")
    ax.set_yticks([])
    clean_axis(ax)
    save_figure(
        fig,
        f"FigureS2_CoreQ_availability_{RUN_TAG}",
        ["figureS2_feature_availability_source.csv"],
    )

    # ------------------------------------------------------------------
    # Supplementary Figure S3: native OOF calibration
    # ------------------------------------------------------------------
    fig, ax = plt.subplots(
        figsize=(SINGLE_COLUMN_IN, 82 * MM_TO_IN),
        constrained_layout=True,
    )
    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=0.7)

    for model_name in ["Age", "Core-Q", "Age + Core-Q"]:
        local = calibration_curves.loc[
            calibration_curves["model"].eq(model_name)
        ]
        ax.plot(
            local["predicted_probability"],
            local["observed_fraction_smooth"],
            linewidth=1.0,
            label=model_name,
        )

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Mean cross-fitted predicted probability")
    ax.set_ylabel("Observed ALS fraction")
    ax.legend(frameon=False)
    clean_axis(ax)
    save_figure(
        fig,
        f"FigureS3_diagnosis_calibration_{RUN_TAG}",
        ["diagnosis_calibration_curve.csv"],
    )

    # ------------------------------------------------------------------
    # Supplementary Figure S4: bulbar predictions/residuals
    # ------------------------------------------------------------------
    sev_final = final_participant_predictions(
        sev_results["Core-Q"], "severity"
    )
    sev_final["residual"] = (
        sev_final["y"] - sev_final["prediction"]
    )
    atomic_write_csv(
        sev_final,
        TABLES / "figureS4_bulbar_predictions_source.csv",
    )

    fig, axes = plt.subplots(
        1, 2,
        figsize=(DOUBLE_COLUMN_IN, 75 * MM_TO_IN),
        constrained_layout=True,
    )

    ax = axes[0]
    ax.scatter(
        sev_final["y"],
        sev_final["prediction"],
        s=11,
        alpha=0.65,
    )
    ax.plot([0, 12], [0, 12], linestyle="--", linewidth=0.7)
    ax.set_xlim(-0.2, 12.2)
    ax.set_ylim(-0.2, 12.2)
    ax.set_xlabel("Observed ALSFRS-R bulbar score (points)")
    ax.set_ylabel("Mean cross-fitted prediction (points)")
    clean_axis(ax)
    panel_label(ax, "a")

    ax = axes[1]
    ax.scatter(
        sev_final["prediction"],
        sev_final["residual"],
        s=11,
        alpha=0.65,
    )
    ax.axhline(0, linestyle="--", linewidth=0.7)
    ax.set_xlabel("Mean cross-fitted prediction (points)")
    ax.set_ylabel("Observed − predicted (points)")
    clean_axis(ax)
    panel_label(ax, "b")

    save_figure(
        fig,
        f"FigureS4_bulbar_prediction_residuals_{RUN_TAG}",
        ["figureS4_bulbar_predictions_source.csv"],
    )

    # ------------------------------------------------------------------
    # Supplementary Figure S5: sensitivity summary
    # ------------------------------------------------------------------
    fig, axes = plt.subplots(
        1, 2,
        figsize=(DOUBLE_COLUMN_IN, 105 * MM_TO_IN),
        constrained_layout=True,
    )

    dx_s = sensitivity_table.loc[
        sensitivity_table["task"].eq("diagnosis")
    ].copy()
    sev_s = sensitivity_table.loc[
        sensitivity_table["task"].eq("severity")
    ].copy()

    ax = axes[0]
    y = np.arange(len(dx_s))
    x = dx_s["estimate"].to_numpy(float)
    err = np.vstack([
        np.maximum(0.0, x - dx_s["ci_low"].to_numpy(float)),
        np.maximum(0.0, dx_s["ci_high"].to_numpy(float) - x),
    ])
    ax.errorbar(
        x, y, xerr=err,
        fmt="o", linewidth=0.8, markersize=3.5, capsize=2,
    )
    ax.axvline(
        float(
            dx_ci.loc[
                dx_ci["model"].eq("Core-Q")
                & dx_ci["metric"].eq("AUROC"),
                "estimate",
            ].iloc[0]
        ),
        linestyle="--",
        linewidth=0.7,
    )
    ax.set_yticks(y)
    ax.set_yticklabels([
        f"{label} (n={int(n)})"
        for label, n in zip(
            dx_s["sensitivity"], dx_s["n_participants"]
        )
    ])
    ax.invert_yaxis()
    ax.set_xlabel("AUROC")
    clean_axis(ax)
    panel_label(ax, "a")

    ax = axes[1]
    y = np.arange(len(sev_s))
    x = sev_s["estimate"].to_numpy(float)
    err = np.vstack([
        np.maximum(0.0, x - sev_s["ci_low"].to_numpy(float)),
        np.maximum(0.0, sev_s["ci_high"].to_numpy(float) - x),
    ])
    ax.errorbar(
        x, y, xerr=err,
        fmt="o", linewidth=0.8, markersize=3.5, capsize=2,
    )
    ax.axvline(
        float(
            sev_ci.loc[
                sev_ci["model"].eq("Core-Q")
                & sev_ci["metric"].eq("MAE"),
                "estimate",
            ].iloc[0]
        ),
        linestyle="--",
        linewidth=0.7,
    )
    ax.set_yticks(y)
    ax.set_yticklabels([
        f"{label} (n={int(n)})"
        for label, n in zip(
            sev_s["sensitivity"], sev_s["n_participants"]
        )
    ])
    ax.invert_yaxis()
    ax.set_xlabel("MAE (ALSFRS-R bulbar points)")
    clean_axis(ax)
    panel_label(ax, "b")

    save_figure(
        fig,
        f"FigureS5_Goal1_sensitivities_{RUN_TAG}",
        ["goal1_sensitivity_summary.csv"],
    )

    # ------------------------------------------------------------------
    # Descriptive Q-age figure
    # ------------------------------------------------------------------
    features = CORE_Q[::-1]
    fig, ax = plt.subplots(
        figsize=(DOUBLE_COLUMN_IN, 92 * MM_TO_IN),
        constrained_layout=True,
    )

    for scope, offset in [
        ("All age-complete", -0.14),
        ("Controls age-complete", 0.14),
    ]:
        local = (
            q_age.loc[q_age["scope"].eq(scope)]
            .set_index("feature")
            .reindex(features)
            .reset_index()
        )
        y = np.arange(len(features)) + offset
        x = local["spearman_rho"].to_numpy(float)
        lo = local["ci_low"].to_numpy(float)
        hi = local["ci_high"].to_numpy(float)
        ax.errorbar(
            x,
            y,
            xerr=np.vstack([
                np.maximum(0.0, x - lo),
                np.maximum(0.0, hi - x),
            ]),
            fmt="o",
            linewidth=0.7,
            markersize=3.3,
            capsize=1.8,
            label=scope,
        )

    ax.axvline(0, linestyle="--", linewidth=0.7)
    ax.set_yticks(np.arange(len(features)))
    ax.set_yticklabels(features)
    ax.set_xlabel("Spearman ρ with age")
    ax.legend(frameon=False)
    clean_axis(ax)

    save_figure(
        fig,
        f"FigureS6_Q_age_descriptive_{RUN_TAG}",
        ["descriptive_q_age_associations.csv"],
    )

def run_goal1():
    print("\n" + "=" * 80)
    print("GOAL 1 COMPLETE PIPELINE")
    print("=" * 80)

    # ---------------------------------------------------------------
    # A. Observed primary model ladders
    # ---------------------------------------------------------------
    dx_results = OrderedDict()
    sev_results = OrderedDict()
    split_logs = []

    for model_name, spec in PRIMARY_MODEL_SPECS.items():
        if model_name == "Core-Q HGB" and not RUN_HGB:
            continue

        print("\nDIAGNOSIS MODEL:", model_name)
        dx_results[model_name] = run_or_load_model(
            dx_primary,
            model_name,
            spec,
            "diagnosis",
            record_splits=(model_name == "Core-Q"),
        )

        print("\nSEVERITY MODEL:", model_name)
        sev_results[model_name] = run_or_load_model(
            sev_primary,
            model_name,
            spec,
            "severity",
            record_splits=(model_name == "Core-Q"),
        )

    # Mean/intercept baseline for severity figure.
    sev_results_with_baseline = OrderedDict()
    sev_results_with_baseline["Mean baseline"] = fit_severity_mean_baseline(
        sev_primary
    )
    sev_results_with_baseline.update(sev_results)

    # Primary metrics.
    dx_observed = summarize_model_metrics(dx_results, "diagnosis")
    sev_observed = summarize_model_metrics(
        sev_results_with_baseline, "severity"
    )
    atomic_write_csv(
        dx_observed,
        TABLES / "diagnosis_observed_metrics.csv",
    )
    atomic_write_csv(
        sev_observed,
        TABLES / "severity_observed_metrics.csv",
    )

    # ---------------------------------------------------------------
    # B. Bootstrap uncertainty
    # ---------------------------------------------------------------
    print("\nComputing participant-cluster bootstrap intervals...")
    dx_ci = bootstrap_model_table(dx_results, "diagnosis")
    sev_ci = bootstrap_model_table(
        sev_results_with_baseline, "severity"
    )

    atomic_write_csv(
        dx_ci,
        TABLES / "diagnosis_bootstrap_ci.csv",
        required_columns=[
            "model", "metric", "estimate", "ci_low", "ci_high"
        ],
    )
    atomic_write_csv(
        sev_ci,
        TABLES / "severity_bootstrap_ci.csv",
        required_columns=[
            "model", "metric", "estimate", "ci_low", "ci_high"
        ],
    )

    # Paired Age+Core-Q vs Age benchmark.
    dx_delta = paired_bootstrap_difference(
        dx_results["Age"],
        dx_results["Age + Core-Q"],
        "diagnosis",
        "AUROC",
        N_BOOTSTRAPS,
        BASE_SEED + 4_000_001,
    )
    sev_delta = paired_bootstrap_difference(
        sev_results["Age"],
        sev_results["Age + Core-Q"],
        "severity",
        "MAE",
        N_BOOTSTRAPS,
        BASE_SEED + 4_000_002,
    )
    dx_lo, dx_hi = bootstrap_ci(dx_delta)
    sev_lo, sev_hi = bootstrap_ci(sev_delta)

    paired_age = pd.DataFrame([
        {
            "task": "diagnosis",
            "contrast": "Age + Core-Q minus Age",
            "metric": "AUROC",
            "estimate": (
                mean_metric_dict(
                    dx_results["Age + Core-Q"], "diagnosis"
                )["AUROC"]
                - mean_metric_dict(
                    dx_results["Age"], "diagnosis"
                )["AUROC"]
            ),
            "ci_low": dx_lo,
            "ci_high": dx_hi,
            "n_participants": len(
                set(dx_results["Age"]["participant_id"])
                & set(dx_results["Age + Core-Q"]["participant_id"])
            ),
        },
        {
            "task": "severity",
            "contrast": "Age + Core-Q minus Age",
            "metric": "MAE",
            "estimate": (
                mean_metric_dict(
                    sev_results["Age + Core-Q"], "severity"
                )["MAE"]
                - mean_metric_dict(
                    sev_results["Age"], "severity"
                )["MAE"]
            ),
            "ci_low": sev_lo,
            "ci_high": sev_hi,
            "n_participants": len(
                set(sev_results["Age"]["participant_id"])
                & set(sev_results["Age + Core-Q"]["participant_id"])
            ),
        },
    ])
    atomic_write_csv(
        paired_age,
        TABLES / "age_incremental_paired_contrast.csv",
    )

    # ---------------------------------------------------------------
    # C. Native diagnosis calibration
    # ---------------------------------------------------------------
    calibration_rows = []
    calibration_curve_rows = []

    for model_name in ["Age", "Core-Q", "Age + Core-Q"]:
        oof = dx_results[model_name]
        calibration_rows.append({
            "model": model_name,
            "n_participants": oof["participant_id"].nunique(),
            **safe_calibration_statistics(oof),
        })

        participant = final_participant_predictions(
            oof, "diagnosis"
        )
        smooth = lowess(
            participant["y"].to_numpy(float),
            participant["prediction"].to_numpy(float),
            frac=0.40,
            it=0,
            return_sorted=True,
        )
        for x, yhat in smooth:
            calibration_curve_rows.append({
                "model": model_name,
                "predicted_probability": float(x),
                "observed_fraction_smooth": float(
                    np.clip(yhat, 0, 1)
                ),
            })

    calibration_table = pd.DataFrame(calibration_rows)
    calibration_curves = pd.DataFrame(
        calibration_curve_rows
    )
    atomic_write_csv(
        calibration_table,
        TABLES / "diagnosis_calibration_statistics.csv",
    )
    atomic_write_csv(
        calibration_curves,
        TABLES / "diagnosis_calibration_curve.csv",
    )

    # ---------------------------------------------------------------
    # D. Full-pipeline permutation nulls
    # ---------------------------------------------------------------
    print("\nRunning/resuming Core-Q permutation nulls...")
    dx_perm = run_permutation_null(
        dx_primary,
        "diagnosis",
        dx_results["Core-Q"],
    )
    sev_perm = run_permutation_null(
        sev_primary,
        "severity",
        sev_results["Core-Q"],
    )

    if RUN_PERMUTATIONS:
        goal1_permutation = pd.concat(
            [dx_perm, sev_perm],
            ignore_index=True,
        )
        atomic_write_csv(
            goal1_permutation,
            TABLES / "goal1_permutation.csv",
            required_columns=[
                "task", "permutation", "null_metric",
                "observed_metric", "empirical_p",
            ],
        )
    else:
        raise RuntimeError(
            "RUN_PERMUTATIONS=False: complete Goal 1 figure cannot be sealed."
        )

    # ---------------------------------------------------------------
    # E. Prespecified sensitivities
    # ---------------------------------------------------------------
    sensitivity_results = []
    sensitivity_oof = {}

    def add_sensitivity(label, task, oof, metric):
        observed = mean_metric_dict(oof, task)[metric]
        dist = bootstrap_metric_distribution(
            oof,
            task,
            metric,
            N_BOOTSTRAPS,
            BASE_SEED + 8_000_000 + len(sensitivity_results),
        )
        lo, hi = bootstrap_ci(dist)
        sensitivity_results.append({
            "task": task,
            "sensitivity": label,
            "metric": metric,
            "estimate": observed,
            "ci_low": lo,
            "ci_high": hi,
            "n_participants": oof["participant_id"].nunique(),
            "n_rows_per_repeat": len(oof.loc[oof["repeat"].eq(1)]),
        })
        sensitivity_oof[(task, label)] = oof

    if RUN_SENSITIVITIES:
        print("\nRunning prespecified Goal 1 sensitivities...")

        # Reference primary Core-Q result.
        add_sensitivity(
            "Primary index Core-Q",
            "diagnosis",
            dx_results["Core-Q"],
            "AUROC",
        )
        add_sensitivity(
            "Primary ≤60 d Core-Q",
            "severity",
            sev_results["Core-Q"],
            "MAE",
        )

        # All recordings / all matched pairs.
        add_sensitivity(
            "All recordings, participant-weighted",
            "diagnosis",
            run_sensitivity(
                "Sensitivity all recordings Core-Q",
                dx_all_recordings,
                PRIMARY_MODEL_SPECS["Core-Q"],
                "diagnosis",
            ),
            "AUROC",
        )
        add_sensitivity(
            "All ≤60 d pairs, participant-weighted",
            "severity",
            run_sensitivity(
                "Sensitivity all 60d pairs Core-Q",
                sev_all_60,
                PRIMARY_MODEL_SPECS["Core-Q"],
                "severity",
            ),
            "MAE",
        )

        # Extended-Q.
        add_sensitivity(
            "Extended-Q",
            "diagnosis",
            run_sensitivity(
                "Sensitivity Extended-Q",
                dx_primary,
                EXTENDED_SPEC,
                "diagnosis",
            ),
            "AUROC",
        )
        add_sensitivity(
            "Extended-Q",
            "severity",
            run_sensitivity(
                "Sensitivity Extended-Q",
                sev_primary,
                EXTENDED_SPEC,
                "severity",
            ),
            "MAE",
        )

        # QCHAN two-part.
        add_sensitivity(
            "QCHAN two-part Core-Q",
            "diagnosis",
            run_sensitivity(
                "Sensitivity QCHAN two-part",
                dx_primary,
                QCHAN_TWO_PART_SPEC,
                "diagnosis",
            ),
            "AUROC",
        )
        add_sensitivity(
            "QCHAN two-part Core-Q",
            "severity",
            run_sensitivity(
                "Sensitivity QCHAN two-part",
                sev_primary,
                QCHAN_TWO_PART_SPEC,
                "severity",
            ),
            "MAE",
        )

        # 90-day severity window.
        add_sensitivity(
            "≤90 d index Core-Q",
            "severity",
            run_sensitivity(
                "Sensitivity 90d Core-Q",
                sev_90,
                PRIMARY_MODEL_SPECS["Core-Q"],
                "severity",
            ),
            "MAE",
        )

        # Complete-case age+sex demographic benchmark.
        add_sensitivity(
            "Age + sex complete-case",
            "diagnosis",
            run_sensitivity(
                "Sensitivity Age + sex",
                dx_primary,
                AGE_SEX_SPEC,
                "diagnosis",
            ),
            "AUROC",
        )
        add_sensitivity(
            "Age + sex + Core-Q complete-case",
            "diagnosis",
            run_sensitivity(
                "Sensitivity Age + sex + Core-Q",
                dx_primary,
                AGE_SEX_CORE_SPEC,
                "diagnosis",
            ),
            "AUROC",
        )
        add_sensitivity(
            "Age + sex complete-case",
            "severity",
            run_sensitivity(
                "Sensitivity Age + sex",
                sev_primary,
                AGE_SEX_SPEC,
                "severity",
            ),
            "MAE",
        )
        add_sensitivity(
            "Age + sex + Core-Q complete-case",
            "severity",
            run_sensitivity(
                "Sensitivity Age + sex + Core-Q",
                sev_primary,
                AGE_SEX_CORE_SPEC,
                "severity",
            ),
            "MAE",
        )

        # HGB is already in the primary model ladder but is included in
        # the sensitivity summary for clarity.
        if "Core-Q HGB" in dx_results:
            add_sensitivity(
                "Core-Q HGB",
                "diagnosis",
                dx_results["Core-Q HGB"],
                "AUROC",
            )
            add_sensitivity(
                "Core-Q HGB",
                "severity",
                sev_results["Core-Q HGB"],
                "MAE",
            )

    sensitivity_table = pd.DataFrame(sensitivity_results)
    if len(sensitivity_table):
        atomic_write_csv(
            sensitivity_table,
            TABLES / "goal1_sensitivity_summary.csv",
            required_columns=[
                "task", "sensitivity", "metric",
                "estimate", "ci_low", "ci_high",
            ],
        )
    else:
        raise RuntimeError(
            "No Goal 1 sensitivities were run; cannot seal complete Goal 1."
        )

    # ---------------------------------------------------------------
    # F. Optional descriptive Q-age check
    # ---------------------------------------------------------------
    q_age = descriptive_q_age_table()
    atomic_write_csv(
        q_age,
        TABLES / "descriptive_q_age_associations.csv",
    )

    # ---------------------------------------------------------------
    # G. Required final OOF artifacts
    # ---------------------------------------------------------------
    goal1_dx_oof = make_wide_primary_oof(
        dx_results, "diagnosis"
    )
    goal1_bulbar_oof = make_wide_primary_oof(
        sev_results, "severity"
    )
    atomic_write_csv(
        goal1_dx_oof,
        TABLES / "goal1_dx_oof.csv",
    )
    atomic_write_csv(
        goal1_bulbar_oof,
        TABLES / "goal1_bulbar_oof.csv",
    )

    # Combined metrics artifact.
    goal1_metrics = pd.concat(
        [dx_ci, sev_ci],
        ignore_index=True,
    )
    atomic_write_csv(
        goal1_metrics,
        TABLES / "goal1_metrics.csv",
        required_columns=[
            "model", "task", "metric",
            "estimate", "ci_low", "ci_high",
        ],
    )

    # Split artifact: outer manifest + Core-Q inner split logs.
    split_frames = []
    outer_copy = split_manifest.copy()
    outer_copy["task"] = "master"
    outer_copy["model"] = "master"
    outer_copy["inner_fold"] = 0
    outer_copy["role"] = "outer_assignment"
    split_frames.append(
        outer_copy[
            [
                "task", "model", "repeat", "outer_fold",
                "inner_fold", "participant_id", "role"
            ]
        ]
    )

    for task in ["diagnosis", "severity"]:
        split_path = model_checkpoint_paths(
            task, "Core-Q"
        )["splits"]
        if split_path.exists():
            split_frames.append(
                safe_read_csv(split_path)
            )

    goal1_splits = pd.concat(
        split_frames,
        ignore_index=True,
        sort=False,
    )
    atomic_write_csv(
        goal1_splits,
        TABLES / "goal1_splits.csv",
    )

    # ---------------------------------------------------------------
    # H. Figures
    # ---------------------------------------------------------------
    print("\nGenerating publication figures...")
    generate_figures(
        dx_results,
        sev_results,
        dx_ci,
        sev_ci,
        dx_perm,
        sev_perm,
        sensitivity_table,
        calibration_curves,
        q_age,
    )

    # ---------------------------------------------------------------
    # I. Final run manifest and seal
    # ---------------------------------------------------------------
    required_outputs = [
        TABLES / "goal1_dx_oof.csv",
        TABLES / "goal1_bulbar_oof.csv",
        TABLES / "goal1_metrics.csv",
        TABLES / "goal1_permutation.csv",
        TABLES / "goal1_splits.csv",
        TABLES / "goal1_sensitivity_summary.csv",
        FIGURES / f"Figure2_Goal1_information_availability_{RUN_TAG}.pdf",
        FIGURES / f"FigureS1_cohort_flow_{RUN_TAG}.pdf",
        FIGURES / f"FigureS2_CoreQ_availability_{RUN_TAG}.pdf",
        FIGURES / f"FigureS3_diagnosis_calibration_{RUN_TAG}.pdf",
        FIGURES / f"FigureS4_bulbar_prediction_residuals_{RUN_TAG}.pdf",
        FIGURES / f"FigureS5_Goal1_sensitivities_{RUN_TAG}.pdf",
    ]

    for path in required_outputs:
        if not path.exists() or path.stat().st_size == 0:
            raise RuntimeError(
                f"Required final Goal 1 artifact missing/empty: {path}"
            )

    output_hashes = {
        str(path.relative_to(ROOT)): sha256_file(path)
        for path in required_outputs
    }

    manifest = {
        "status": "PASS",
        "engine_version": ENGINE_VERSION,
        "engine_build_sha256": ENGINE_BUILD_SHA256,
        "run_mode": RUN_MODE,
        "run_signature": RUN_SIGNATURE,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "python": sys.version,
        "platform": platform.platform(),
        "packages": {
            "numpy": package_version("numpy"),
            "pandas": package_version("pandas"),
            "scipy": package_version("scipy"),
            "scikit-learn": package_version("scikit-learn"),
            "statsmodels": package_version("statsmodels"),
            "matplotlib": package_version("matplotlib"),
        },
        "paper1_qchan_commit": observed_paper1_commit,
        "base_seed": BASE_SEED,
        "outer_folds": OUTER_FOLDS,
        "outer_repeats": OUTER_REPEATS,
        "inner_folds": INNER_FOLDS,
        "bootstrap_replicates": N_BOOTSTRAPS,
        "permutations": N_PERMUTATIONS,
        "permutation_repeats": PERMUTATION_REPEATS,
        "input_hashes": input_hashes,
        "output_hashes": output_hashes,
        "final_inference": (
            RUN_MODE == "FINAL"
            and N_BOOTSTRAPS == 2000
            and N_PERMUTATIONS == 1000
            and PERMUTATION_REPEATS == 10
        ),
    }

    atomic_write_json(
        manifest,
        OUT / "goal1_run_manifest.json",
    )
    atomic_write_json(
        {
            "status": "PASS",
            "run_mode": RUN_MODE,
            "run_signature": RUN_SIGNATURE,
            "manifest": str(
                (OUT / "goal1_run_manifest.json").relative_to(ROOT)
            ),
        },
        OUT / "SUCCESS.json",
    )

    # Display the key scientific outputs only after the run is sealed.
    display(Markdown("## Goal 1 run sealed successfully"))
    display(
        dx_ci.loc[
            dx_ci["metric"].eq("AUROC"),
            ["model", "n_participants", "estimate", "ci_low", "ci_high"],
        ]
    )
    display(
        sev_ci.loc[
            sev_ci["metric"].eq("MAE"),
            ["model", "n_participants", "estimate", "ci_low", "ci_high"],
        ]
    )
    display(paired_age)
    display(
        pd.DataFrame([
            {
                "task": "diagnosis",
                "observed": dx_perm["observed_metric"].iloc[0],
                "permutation_p": dx_perm["empirical_p"].iloc[0],
                "B": len(dx_perm),
            },
            {
                "task": "severity",
                "observed": sev_perm["observed_metric"].iloc[0],
                "permutation_p": sev_perm["empirical_p"].iloc[0],
                "B": len(sev_perm),
            },
        ])
    )
    display(sensitivity_table)

    if RUN_MODE == "FINAL":
        if not manifest["final_inference"]:
            raise RuntimeError(
                "FINAL run completed without the frozen final resampling counts."
            )
        print("GOAL 1 FINAL INFERENCE: PASS")
    else:
        print(
            "GOAL 1 DEVELOPMENT PIPELINE: PASS\n"
            "All code paths and figures completed. "
            "Do not use development CIs/P values in the manuscript. "
            "Set RUN_MODE='FINAL', RESET_OUTPUTS=True and run from a fresh kernel."
        )

    return {
        "dx_results": dx_results,
        "sev_results": sev_results,
        "dx_ci": dx_ci,
        "sev_ci": sev_ci,
        "dx_perm": dx_perm,
        "sev_perm": sev_perm,
        "sensitivity_table": sensitivity_table,
        "manifest": manifest,
    }

GOAL1 = run_goal1()


GOAL 1 COMPLETE PIPELINE

DIAGNOSIS MODEL: Age
diagnosis | Age                | repeat 1/10 complete
diagnosis | Age                | repeat 2/10 complete
diagnosis | Age                | repeat 3/10 complete
diagnosis | Age                | repeat 4/10 complete
diagnosis | Age                | repeat 5/10 complete
diagnosis | Age                | repeat 6/10 complete
diagnosis | Age                | repeat 7/10 complete
diagnosis | Age                | repeat 8/10 complete
diagnosis | Age                | repeat 9/10 complete
diagnosis | Age                | repeat 10/10 complete

SEVERITY MODEL: Age
severity  | Age                | repeat 1/10 complete
severity  | Age                | repeat 2/10 complete
severity  | Age                | repeat 3/10 complete
severity  | Age                | repeat 4/10 complete
severity  | Age                | repeat 5/10 complete
severity  | Age                | repeat 6/10 complete
severity  | Age                | repeat 7/10 complete
severity  | 

KeyboardInterrupt: 

In [14]:
perm_checkpoint = (
    CHECKPOINTS
    / "diagnosis__coreq_permutations.csv"
)

print("Checkpoint:", perm_checkpoint)
print("Exists:", perm_checkpoint.exists())

if perm_checkpoint.exists():
    _perm = pd.read_csv(perm_checkpoint)

    print("Completed permutations:", len(_perm))
    print(
        "Last completed permutation:",
        int(_perm["permutation"].max())
        if len(_perm)
        else None,
    )

    display(_perm.tail())

Checkpoint: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\outputs\goal1\goal1_complete_v1_1\final\checkpoints\diagnosis__coreq_permutations.csv
Exists: True
Completed permutations: 2
Last completed permutation: 2


,permutation,null_metric
0,1,0.439346
1,2,0.452877


In [15]:
# =====================================================================
# GOAL 1 — ACCELERATED FINAL PERMUTATION ENGINE
# Cell A1: isolated workspace + preservation of original slow checkpoint
# =====================================================================

from pathlib import Path
import os
import shutil
import time
import json

from joblib import dump, load, Parallel, delayed, parallel_backend
from threadpoolctl import threadpool_limits

if RUN_MODE != "FINAL":
    raise RuntimeError(
        f"Accelerated final permutation engine requires RUN_MODE='FINAL'; "
        f"observed {RUN_MODE!r}."
    )

if N_PERMUTATIONS != 1000:
    raise RuntimeError(
        f"Expected N_PERMUTATIONS=1000, observed {N_PERMUTATIONS}."
    )

if PERMUTATION_REPEATS != 10:
    raise RuntimeError(
        f"Expected PERMUTATION_REPEATS=10, observed {PERMUTATION_REPEATS}."
    )

FAST_PERM_ENGINE = "goal1-fast-permutation-fixed-splits-v1.0.0"

FAST_PERM_ROOT = OUT / "accelerated_permutation_v1"
FAST_PERM_CACHE = FAST_PERM_ROOT / "design_cache"
FAST_PERM_CHECKPOINTS = FAST_PERM_ROOT / "checkpoints"
FAST_PERM_AUDIT = FAST_PERM_ROOT / "audit"

for directory in [
    FAST_PERM_ROOT,
    FAST_PERM_CACHE,
    FAST_PERM_CHECKPOINTS,
    FAST_PERM_AUDIT,
]:
    directory.mkdir(parents=True, exist_ok=True)

FAST_PERM_SIGNATURE = stable_hash({
    "engine": FAST_PERM_ENGINE,
    "parent_run_signature": RUN_SIGNATURE,
    "run_mode": RUN_MODE,
    "outer_folds": OUTER_FOLDS,
    "outer_repeats": OUTER_REPEATS,
    "inner_folds": INNER_FOLDS,
    "n_permutations": N_PERMUTATIONS,
    "permutation_repeats": PERMUTATION_REPEATS,
    "split_rule": (
        "reuse exact observed Core-Q outer-test and "
        "inner-validation participant assignments"
    ),
    "preprocessing_rule": (
        "memoize outcome-independent fold-safe predictor matrices; "
        "retune/refit outcome model under every permutation"
    ),
})

# ---------------------------------------------------------------------
# Preserve the original slow checkpoints exactly as they currently exist.
# They will NOT be mixed into the new accelerated null.
# ---------------------------------------------------------------------
for task in ["diagnosis", "severity"]:
    source = permutation_checkpoint_path(task)

    if source.exists() and source.stat().st_size > 0:
        destination = (
            FAST_PERM_AUDIT
            / f"ORIGINAL_SLOW_{task}__coreq_permutations.csv"
        )

        if not destination.exists():
            shutil.copy2(source, destination)

        original = pd.read_csv(source)

        print(
            f"{task}: preserved original slow checkpoint | "
            f"rows={len(original)} | "
            f"path={destination}"
        )
    else:
        print(f"{task}: no original slow permutation checkpoint to preserve.")

atomic_write_json(
    {
        "engine": FAST_PERM_ENGINE,
        "fast_permutation_signature": FAST_PERM_SIGNATURE,
        "parent_run_signature": RUN_SIGNATURE,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "old_slow_permutations_reused": False,
        "reason": (
            "The accelerated null uses the exact persisted observed Core-Q "
            "inner/outer participant partitions. Historical slow null values "
            "were generated by the older permutation implementation and are "
            "retained for audit only."
        ),
    },
    FAST_PERM_ROOT / "accelerated_permutation_contract.json",
)

print()
print("ACCELERATED PERMUTATION WORKSPACE: READY")
print("Engine:", FAST_PERM_ENGINE)
print("Signature:", FAST_PERM_SIGNATURE[:16])

diagnosis: preserved original slow checkpoint | rows=2 | path=C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\outputs\goal1\goal1_complete_v1_1\final\accelerated_permutation_v1\audit\ORIGINAL_SLOW_diagnosis__coreq_permutations.csv
severity: no original slow permutation checkpoint to preserve.

ACCELERATED PERMUTATION WORKSPACE: READY
Engine: goal1-fast-permutation-fixed-splits-v1.0.0
Signature: 8402bfa378c4d904


In [16]:
# =====================================================================
# Cell A2: build outcome-independent Core-Q design cache
#
# CRITICAL:
# - exact observed Core-Q outer/inner participant assignments are reused;
# - QCHAN reference remains training-participant-only;
# - transformations/imputation/scaling remain training-only;
# - NO clinical outcome is stored in the design cache.
# =====================================================================

COREQ_SPEC_FAST = PRIMARY_MODEL_SPECS["Core-Q"]

if COREQ_SPEC_FAST["kind"] != "ridge":
    raise RuntimeError("Primary Core-Q model is unexpectedly not ridge.")


def fast_primary_frame(task):
    if task == "diagnosis":
        source = diagnosis_index
    elif task == "severity":
        source = severity_index
    else:
        raise ValueError(task)

    data = analysis_subset(
        source,
        COREQ_SPEC_FAST,
    ).copy()

    data["participant_id"] = (
        data["participant_id"]
        .astype(str)
        .str.strip()
    )

    data["logical_recording_id"] = (
        data["logical_recording_id"]
        .astype(str)
        .str.strip()
    )

    if not data["participant_id"].is_unique:
        raise RuntimeError(
            f"{task}: accelerated primary permutation population "
            "must be one row per participant."
        )

    return data


def load_observed_coreq_split_log(task):
    path = model_checkpoint_paths(
        task,
        "Core-Q",
    )["splits"]

    if not path.exists():
        raise FileNotFoundError(
            f"Observed Core-Q split log is missing: {path}"
        )

    splits = safe_read_csv(
        path,
        required_columns=[
            "task",
            "model",
            "repeat",
            "outer_fold",
            "inner_fold",
            "participant_id",
            "role",
            "seed",
        ],
    ).copy()

    splits["participant_id"] = (
        splits["participant_id"]
        .astype(str)
        .str.strip()
    )

    splits = splits.loc[
        splits["task"].eq(task)
        & splits["model"].eq("Core-Q")
    ].copy()

    if len(splits) == 0:
        raise RuntimeError(
            f"{task}: no Core-Q split rows found."
        )

    return path, splits


def build_fast_design_cache(task, force_rebuild=False):
    cache_path = (
        FAST_PERM_CACHE
        / f"{task}__coreq_design_cache.joblib"
    )
    metadata_path = (
        FAST_PERM_CACHE
        / f"{task}__coreq_design_cache.json"
    )
    split_copy_path = (
        FAST_PERM_CACHE
        / f"{task}__fixed_coreq_split_manifest.csv"
    )

    if (
        cache_path.exists()
        and metadata_path.exists()
        and not force_rebuild
    ):
        metadata = json.loads(
            metadata_path.read_text(
                encoding="utf-8"
            )
        )

        if (
            metadata.get("fast_permutation_signature")
            == FAST_PERM_SIGNATURE
        ):
            print(
                f"{task}: loading existing validated design cache..."
            )

            return load(
                cache_path,
                mmap_mode="r",
            )

        raise RuntimeError(
            f"{task}: an incompatible accelerated cache already exists. "
            "Do not silently reuse it."
        )

    data = fast_primary_frame(task)
    split_path, observed_splits = (
        load_observed_coreq_split_log(task)
    )

    all_ids = set(
        data["participant_id"].astype(str)
    )

    # Persist the exact split assignments used to build this cache.
    atomic_write_csv(
        observed_splits,
        split_copy_path,
    )

    bundles = []

    print(
        f"\nBuilding {task} design cache | "
        f"{data['participant_id'].nunique()} participants"
    )

    build_start = time.perf_counter()

    for repeat in range(
        1,
        PERMUTATION_REPEATS + 1,
    ):
        repeat_start = time.perf_counter()

        for outer_fold in range(
            1,
            OUTER_FOLDS + 1,
        ):
            local_split = observed_splits.loc[
                observed_splits["repeat"].eq(repeat)
                & observed_splits["outer_fold"].eq(
                    outer_fold
                )
            ].copy()

            outer_test_ids = set(
                local_split.loc[
                    local_split["role"].eq(
                        "outer_test"
                    ),
                    "participant_id",
                ].astype(str)
            )

            if not outer_test_ids:
                raise RuntimeError(
                    f"{task}/r{repeat}/f{outer_fold}: "
                    "missing persisted outer-test IDs."
                )

            if not outer_test_ids.issubset(
                all_ids
            ):
                raise RuntimeError(
                    f"{task}/r{repeat}/f{outer_fold}: "
                    "unknown participant in outer-test split."
                )

            train = data.loc[
                ~data["participant_id"].isin(
                    outer_test_ids
                )
            ].copy()

            test = data.loc[
                data["participant_id"].isin(
                    outer_test_ids
                )
            ].copy()

            if len(train) == 0 or len(test) == 0:
                raise RuntimeError(
                    f"{task}/r{repeat}/f{outer_fold}: "
                    "empty outer train/test population."
                )

            if (
                set(train["participant_id"])
                & set(test["participant_id"])
            ):
                raise RuntimeError(
                    "Outer participant leakage."
                )

            outer_train_ids = set(
                train["participant_id"].astype(str)
            )

            # ---------------------------------------------------------
            # Recover the EXACT five observed inner-validation
            # assignments that were persisted during Core-Q fitting.
            # ---------------------------------------------------------
            inner_log = local_split.loc[
                local_split["role"].eq(
                    "inner_validation"
                )
            ].copy()

            inner_counts = (
                inner_log.groupby(
                    "participant_id"
                )
                .size()
            )

            if (
                set(inner_counts.index.astype(str))
                != outer_train_ids
            ):
                raise RuntimeError(
                    f"{task}/r{repeat}/f{outer_fold}: "
                    "inner split participant coverage mismatch."
                )

            if not inner_counts.eq(1).all():
                raise RuntimeError(
                    f"{task}/r{repeat}/f{outer_fold}: "
                    "each outer-training participant must be "
                    "inner-validation exactly once."
                )

            if (
                inner_log["inner_fold"].nunique()
                != INNER_FOLDS
            ):
                raise RuntimeError(
                    f"{task}/r{repeat}/f{outer_fold}: "
                    "expected five inner folds."
                )

            inner_bundles = []

            for inner_fold in range(
                1,
                INNER_FOLDS + 1,
            ):
                val_ids = set(
                    inner_log.loc[
                        inner_log[
                            "inner_fold"
                        ].eq(inner_fold),
                        "participant_id",
                    ].astype(str)
                )

                train_ids = (
                    outer_train_ids
                    - val_ids
                )

                if (
                    not train_ids
                    or not val_ids
                    or train_ids & val_ids
                ):
                    raise RuntimeError(
                        f"{task}/r{repeat}/f{outer_fold}/"
                        f"inner{inner_fold}: invalid split."
                    )

                inner_train = train.loc[
                    train["participant_id"].isin(
                        train_ids
                    )
                ].copy()

                inner_val = train.loc[
                    train["participant_id"].isin(
                        val_ids
                    )
                ].copy()

                # Exact fold-safe Q/QCHAN preprocessing.
                raw_inner_train = (
                    build_predictor_frame(
                        inner_train,
                        COREQ_SPEC_FAST,
                        train_ids,
                    )
                )

                raw_inner_val = (
                    build_predictor_frame(
                        inner_val,
                        COREQ_SPEC_FAST,
                        train_ids,
                    )
                )

                X_inner_train, X_inner_val = (
                    preprocess_train_eval(
                        raw_inner_train,
                        raw_inner_val,
                        COREQ_SPEC_FAST,
                    )
                )

                inner_bundles.append({
                    "inner_fold": int(
                        inner_fold
                    ),
                    "train_ids": (
                        inner_train[
                            "participant_id"
                        ]
                        .astype(str)
                        .to_numpy()
                    ),
                    "val_ids": (
                        inner_val[
                            "participant_id"
                        ]
                        .astype(str)
                        .to_numpy()
                    ),
                    "X_train": np.asarray(
                        X_inner_train,
                        dtype=float,
                    ),
                    "X_val": np.asarray(
                        X_inner_val,
                        dtype=float,
                    ),
                    "w_train": np.asarray(
                        participant_weights(
                            inner_train
                        ),
                        dtype=float,
                    ),
                })

            # ---------------------------------------------------------
            # Exact outer-fold preprocessing.
            # ---------------------------------------------------------
            reference_ids = set(
                train[
                    "participant_id"
                ].astype(str)
            )

            raw_outer_train = (
                build_predictor_frame(
                    train,
                    COREQ_SPEC_FAST,
                    reference_ids,
                )
            )

            raw_outer_test = (
                build_predictor_frame(
                    test,
                    COREQ_SPEC_FAST,
                    reference_ids,
                )
            )

            X_outer_train, X_outer_test = (
                preprocess_train_eval(
                    raw_outer_train,
                    raw_outer_test,
                    COREQ_SPEC_FAST,
                )
            )

            seed = (
                BASE_SEED
                + repeat * 10000
                + outer_fold * 100
            )

            bundles.append({
                "repeat": int(repeat),
                "outer_fold": int(
                    outer_fold
                ),
                "seed": int(seed),

                "train_ids": (
                    train["participant_id"]
                    .astype(str)
                    .to_numpy()
                ),
                "test_ids": (
                    test["participant_id"]
                    .astype(str)
                    .to_numpy()
                ),
                "test_recording_ids": (
                    test[
                        "logical_recording_id"
                    ]
                    .astype(str)
                    .to_numpy()
                ),

                "X_train": np.asarray(
                    X_outer_train,
                    dtype=float,
                ),
                "X_test": np.asarray(
                    X_outer_test,
                    dtype=float,
                ),
                "w_train": np.asarray(
                    participant_weights(train),
                    dtype=float,
                ),

                "inner": inner_bundles,
            })

        elapsed_repeat = (
            time.perf_counter()
            - repeat_start
        )

        print(
            f"{task} cache | "
            f"repeat {repeat}/"
            f"{PERMUTATION_REPEATS} "
            f"complete | "
            f"{elapsed_repeat / 60:.1f} min"
        )

    if len(bundles) != (
        PERMUTATION_REPEATS
        * OUTER_FOLDS
    ):
        raise RuntimeError(
            f"{task}: expected 50 cached outer folds; "
            f"found {len(bundles)}."
        )

    cache = {
        "task": task,
        "engine": FAST_PERM_ENGINE,
        "signature": FAST_PERM_SIGNATURE,
        "n_participants": int(
            data[
                "participant_id"
            ].nunique()
        ),
        "bundles": bundles,
    }

    # No compression -> joblib can memory-map large NumPy arrays
    # across worker processes instead of making unnecessary copies.
    dump(
        cache,
        cache_path,
        compress=0,
    )

    total_seconds = (
        time.perf_counter()
        - build_start
    )

    atomic_write_json(
        {
            "task": task,
            "engine": FAST_PERM_ENGINE,
            "fast_permutation_signature": (
                FAST_PERM_SIGNATURE
            ),
            "parent_run_signature": (
                RUN_SIGNATURE
            ),
            "observed_split_log": str(
                split_path
            ),
            "observed_split_log_sha256": (
                sha256_file(split_path)
            ),
            "fixed_split_copy": str(
                split_copy_path
            ),
            "n_participants": int(
                data[
                    "participant_id"
                ].nunique()
            ),
            "n_outer_bundles": len(
                bundles
            ),
            "build_seconds": float(
                total_seconds
            ),
            "outcome_stored_in_cache": False,
            "status": "PASS",
        },
        metadata_path,
    )

    print(
        f"\n{task.upper()} DESIGN CACHE: PASS"
    )
    print(
        f"Outer bundles: {len(bundles)}"
    )
    print(
        f"Build time: "
        f"{total_seconds / 60:.1f} min"
    )

    return load(
        cache_path,
        mmap_mode="r",
    )


print("FAST DESIGN-CACHE BUILDER: READY")

FAST DESIGN-CACHE BUILDER: READY


In [17]:
# =====================================================================
# Cell A3: cached nested-CV computation
#
# The design matrices are frozen by participant split.
# Ridge hyperparameter tuning and final fitting are repeated from scratch
# for every permuted outcome.
# =====================================================================

def _labels_for_ids(
    label_lookup,
    participant_ids,
):
    values = np.asarray(
        [
            label_lookup[str(pid)]
            for pid in participant_ids
        ],
        dtype=float,
    )

    if not np.isfinite(values).all():
        raise RuntimeError(
            "Non-finite permuted outcome."
        )

    return values


def _fast_inner_loss(
    task,
    y,
    prediction,
):
    y = np.asarray(y, dtype=float)
    prediction = np.asarray(
        prediction,
        dtype=float,
    )

    if task == "diagnosis":
        return float(
            log_loss(
                y.astype(int),
                np.clip(
                    prediction,
                    1e-8,
                    1 - 1e-8,
                ),
                labels=[0, 1],
            )
        )

    # Primary Goal 1 severity has exactly one row / participant,
    # so this is algebraically identical to exact_tuning_loss().
    return float(
        np.mean(
            np.abs(
                y - prediction
            )
        )
    )


def _tune_cached_coreq(
    outer_bundle,
    task,
    label_lookup,
):
    grid = (
        C_GRID
        if task == "diagnosis"
        else ALPHA_GRID
    )

    seed = int(
        outer_bundle["seed"]
    )

    scores = []

    for param_index, param in enumerate(
        grid
    ):
        losses = []

        for inner in outer_bundle["inner"]:
            y_train = _labels_for_ids(
                label_lookup,
                inner["train_ids"],
            )

            y_val = _labels_for_ids(
                label_lookup,
                inner["val_ids"],
            )

            # The diagnostic ridge cannot be fitted
            # if a pathological permutation left a
            # training fold with one class.
            if (
                task == "diagnosis"
                and np.unique(
                    y_train.astype(int)
                ).size != 2
            ):
                raise RuntimeError(
                    "Permuted diagnosis inner-training "
                    "fold contains only one class."
                )

            model = fit_estimator(
                task,
                "ridge",
                param,
                (
                    seed
                    + 1000
                    * (param_index + 1)
                    + int(
                        inner[
                            "inner_fold"
                        ]
                    )
                ),
                inner["X_train"],
                y_train,
                inner["w_train"],
            )

            pred = model_predict(
                model,
                task,
                inner["X_val"],
            )

            losses.append(
                _fast_inner_loss(
                    task,
                    y_val,
                    pred,
                )
            )

        scores.append(
            float(np.mean(losses))
        )

    best_index = int(
        np.argmin(
            np.asarray(
                scores,
                dtype=float,
            )
        )
    )

    return (
        float(grid[best_index]),
        float(scores[best_index]),
    )


def run_cached_coreq(
    cache,
    label_lookup,
    task,
    *,
    return_oof=False,
):
    if cache["signature"] != FAST_PERM_SIGNATURE:
        raise RuntimeError(
            "Accelerated cache signature mismatch."
        )

    per_repeat_y = {
        repeat: []
        for repeat in range(
            1,
            PERMUTATION_REPEATS + 1,
        )
    }

    per_repeat_pred = {
        repeat: []
        for repeat in range(
            1,
            PERMUTATION_REPEATS + 1,
        )
    }

    oof_rows = []
    fit_rows = []

    for outer in cache["bundles"]:
        repeat = int(
            outer["repeat"]
        )
        outer_fold = int(
            outer["outer_fold"]
        )
        seed = int(
            outer["seed"]
        )

        selected, best_inner_loss = (
            _tune_cached_coreq(
                outer,
                task,
                label_lookup,
            )
        )

        y_train = _labels_for_ids(
            label_lookup,
            outer["train_ids"],
        )

        y_test = _labels_for_ids(
            label_lookup,
            outer["test_ids"],
        )

        if (
            task == "diagnosis"
            and np.unique(
                y_train.astype(int)
            ).size != 2
        ):
            raise RuntimeError(
                "Permuted diagnosis outer-training "
                "fold contains only one class."
            )

        model = fit_estimator(
            task,
            "ridge",
            selected,
            seed + 999999,
            outer["X_train"],
            y_train,
            outer["w_train"],
        )

        prediction = model_predict(
            model,
            task,
            outer["X_test"],
        )

        per_repeat_y[
            repeat
        ].append(y_test)

        per_repeat_pred[
            repeat
        ].append(
            np.asarray(
                prediction,
                dtype=float,
            )
        )

        if return_oof:
            local = pd.DataFrame({
                "participant_id": (
                    np.asarray(
                        outer[
                            "test_ids"
                        ]
                    ).astype(str)
                ),
                "logical_recording_id": (
                    np.asarray(
                        outer[
                            "test_recording_ids"
                        ]
                    ).astype(str)
                ),
                "y": y_test.astype(
                    float
                ),
                "prediction": (
                    np.asarray(
                        prediction,
                        dtype=float,
                    )
                ),
                "repeat": repeat,
                "outer_fold": (
                    outer_fold
                ),
                "model": "Core-Q",
                "task": task,
            })

            oof_rows.append(local)

            fit_rows.append({
                "task": task,
                "repeat": repeat,
                "outer_fold": (
                    outer_fold
                ),
                "selected_parameter": (
                    selected
                ),
                "best_inner_loss": (
                    best_inner_loss
                ),
            })

    repeat_metrics = []

    for repeat in range(
        1,
        PERMUTATION_REPEATS + 1,
    ):
        y = np.concatenate(
            per_repeat_y[repeat]
        )

        prediction = np.concatenate(
            per_repeat_pred[repeat]
        )

        if task == "diagnosis":
            metric = roc_auc_score(
                y.astype(int),
                prediction,
            )
        else:
            # One primary severity row per participant.
            metric = float(
                np.mean(
                    np.abs(
                        y - prediction
                    )
                )
            )

        repeat_metrics.append(
            float(metric)
        )

    mean_metric = float(
        np.mean(
            repeat_metrics
        )
    )

    if not return_oof:
        return mean_metric, None, None

    oof = pd.concat(
        oof_rows,
        ignore_index=True,
    )

    fits = pd.DataFrame(
        fit_rows
    )

    return (
        mean_metric,
        oof,
        fits,
    )


print("CACHED NESTED-CV ENGINE: READY")

CACHED NESTED-CV ENGINE: READY


In [19]:
# =====================================================================
# Cell A4: DIAGNOSIS cache + exact observed-result reproduction gate
# =====================================================================

dx_fast_cache = build_fast_design_cache(
    "diagnosis",
    force_rebuild=False,
)

dx_fast_data = fast_primary_frame(
    "diagnosis"
)

dx_original_labels = dict(
    zip(
        dx_fast_data[
            "participant_id"
        ].astype(str),
        dx_fast_data["y"].astype(
            float
        ),
    )
)

(
    dx_fast_observed_metric,
    dx_fast_observed_oof,
    dx_fast_observed_fits,
) = run_cached_coreq(
    dx_fast_cache,
    dx_original_labels,
    "diagnosis",
    return_oof=True,
)

# ---------------------------------------------------------------------
# Load the ORIGINAL observed Core-Q OOF and fit manifest created earlier
# by this exact FINAL notebook run.
# ---------------------------------------------------------------------
dx_paths = model_checkpoint_paths(
    "diagnosis",
    "Core-Q",
)

dx_original_oof = safe_read_csv(
    dx_paths["oof"],
    required_columns=[
        "participant_id",
        "logical_recording_id",
        "y",
        "prediction",
        "repeat",
        "outer_fold",
        "model",
        "task",
    ],
)

dx_original_fits = safe_read_csv(
    dx_paths["fits"],
    required_columns=[
        "repeat",
        "outer_fold",
        "selected_parameter",
    ],
)

key = [
    "participant_id",
    "logical_recording_id",
    "repeat",
    "outer_fold",
]

comparison = (
    dx_original_oof[
        key + ["prediction"]
    ]
    .rename(
        columns={
            "prediction":
            "prediction_original"
        }
    )
    .merge(
        dx_fast_observed_oof[
            key + ["prediction"]
        ].rename(
            columns={
                "prediction":
                "prediction_cached"
            }
        ),
        on=key,
        how="outer",
        validate="one_to_one",
        indicator=True,
    )
)

if not comparison[
    "_merge"
].eq("both").all():
    raise RuntimeError(
        "Observed OOF identity mismatch between "
        "original and accelerated engines."
    )

comparison[
    "abs_prediction_difference"
] = np.abs(
    comparison[
        "prediction_original"
    ].to_numpy(float)
    - comparison[
        "prediction_cached"
    ].to_numpy(float)
)

max_prediction_difference = float(
    comparison[
        "abs_prediction_difference"
    ].max()
)

# Exact selected-C comparison.
fit_compare = (
    dx_original_fits[
        [
            "repeat",
            "outer_fold",
            "selected_parameter",
        ]
    ]
    .rename(
        columns={
            "selected_parameter":
            "selected_original"
        }
    )
    .merge(
        dx_fast_observed_fits[
            [
                "repeat",
                "outer_fold",
                "selected_parameter",
            ]
        ].rename(
            columns={
                "selected_parameter":
                "selected_cached"
            }
        ),
        on=[
            "repeat",
            "outer_fold",
        ],
        how="outer",
        validate="one_to_one",
        indicator=True,
    )
)

if not fit_compare[
    "_merge"
].eq("both").all():
    raise RuntimeError(
        "Observed fold-fit identity mismatch."
    )

fit_compare[
    "selected_original"
] = pd.to_numeric(
    fit_compare[
        "selected_original"
    ],
    errors="raise",
)

fit_compare[
    "selected_cached"
] = pd.to_numeric(
    fit_compare[
        "selected_cached"
    ],
    errors="raise",
)

parameter_match = np.allclose(
    fit_compare[
        "selected_original"
    ].to_numpy(float),
    fit_compare[
        "selected_cached"
    ].to_numpy(float),
    atol=0,
    rtol=0,
)

dx_original_observed_metric = float(
    repeat_metric_table(
        dx_original_oof,
        "diagnosis",
    )["AUROC"].mean()
)

metric_difference = abs(
    dx_original_observed_metric
    - dx_fast_observed_metric
)

print(
    "Original observed Core-Q AUROC:",
    f"{dx_original_observed_metric:.12f}",
)

print(
    "Cached-engine Core-Q AUROC:",
    f"{dx_fast_observed_metric:.12f}",
)

print(
    "Metric absolute difference:",
    f"{metric_difference:.3e}",
)

print(
    "Maximum OOF probability difference:",
    f"{max_prediction_difference:.3e}",
)

print(
    "All 50 selected ridge C values identical:",
    parameter_match,
)

if not parameter_match:
    display(
        fit_compare.loc[
            ~np.isclose(
                fit_compare[
                    "selected_original"
                ],
                fit_compare[
                    "selected_cached"
                ],
                atol=0,
                rtol=0,
            )
        ]
    )

if metric_difference > 1e-10:
    raise RuntimeError(
        "Accelerated engine failed observed AUROC "
        "reproduction."
    )

if max_prediction_difference > 1e-10:
    raise RuntimeError(
        "Accelerated engine failed observed OOF "
        "prediction reproduction."
    )

if not parameter_match:
    raise RuntimeError(
        "Accelerated engine selected different "
        "observed ridge hyperparameters."
    )

atomic_write_csv(
    comparison,
    FAST_PERM_AUDIT
    / "diagnosis_observed_reproduction_oof.csv",
)

atomic_write_csv(
    fit_compare,
    FAST_PERM_AUDIT
    / "diagnosis_observed_reproduction_fits.csv",
)

atomic_write_json(
    {
        "task": "diagnosis",
        "status": "PASS",
        "original_metric": (
            dx_original_observed_metric
        ),
        "cached_metric": (
            dx_fast_observed_metric
        ),
        "metric_absolute_difference": (
            metric_difference
        ),
        "max_oof_prediction_difference": (
            max_prediction_difference
        ),
        "all_selected_parameters_identical": bool(
            parameter_match
        ),
        "n_outer_fits": int(
            len(
                dx_fast_observed_fits
            )
        ),
    },
    FAST_PERM_AUDIT
    / "diagnosis_observed_reproduction_gate.json",
)

print()
print(
    "DIAGNOSIS NUMERICAL REPRODUCTION GATE: PASS"
)


Building diagnosis design cache | 224 participants
diagnosis cache | repeat 1/10 complete | 3.1 min
diagnosis cache | repeat 2/10 complete | 3.1 min
diagnosis cache | repeat 3/10 complete | 3.2 min
diagnosis cache | repeat 4/10 complete | 3.2 min
diagnosis cache | repeat 5/10 complete | 3.2 min
diagnosis cache | repeat 6/10 complete | 3.3 min
diagnosis cache | repeat 7/10 complete | 3.1 min
diagnosis cache | repeat 8/10 complete | 3.5 min
diagnosis cache | repeat 9/10 complete | 3.4 min
diagnosis cache | repeat 10/10 complete | 4.1 min

DIAGNOSIS DESIGN CACHE: PASS
Outer bundles: 50
Build time: 33.2 min
Original observed Core-Q AUROC: 0.743037974684
Cached-engine Core-Q AUROC: 0.743037974684
Metric absolute difference: 0.000e+00
Maximum OOF probability difference: 1.110e-16
All 50 selected ridge C values identical: True

DIAGNOSIS NUMERICAL REPRODUCTION GATE: PASS


In [21]:
# =====================================================================
# Cell A5: parallel accelerated permutation runner + 8-permutation benchmark
# =====================================================================

def fast_permutation_checkpoint_path(
    task,
):
    return (
        FAST_PERM_CHECKPOINTS
        / f"{task}__coreq_permutations.csv"
    )


def _single_fast_permutation(
    cache,
    task,
    permutation,
    base_ids,
    base_y,
):
    seed_offset = (
        10_000_000
        if task == "diagnosis"
        else 20_000_000
    )

    rng = np.random.default_rng(
        BASE_SEED
        + seed_offset
        + int(permutation)
    )

    permuted_y = rng.permutation(
        np.asarray(
            base_y,
            dtype=float,
        )
    )

    label_lookup = dict(
        zip(
            np.asarray(
                base_ids
            ).astype(str),
            permuted_y,
        )
    )

    # Prevent BLAS/OpenMP oversubscription inside each worker.
    with threadpool_limits(
        limits=1
    ):
        metric, _, _ = (
            run_cached_coreq(
                cache,
                label_lookup,
                task,
                return_oof=False,
            )
        )

    return {
        "permutation": int(
            permutation
        ),
        "null_metric": float(
            metric
        ),
        "engine": (
            FAST_PERM_ENGINE
        ),
        "fast_permutation_signature": (
            FAST_PERM_SIGNATURE
        ),
    }


def run_fast_permutations(
    task,
    cache,
    *,
    target,
    n_jobs,
):
    data = fast_primary_frame(
        task
    )

    base_ids = (
        data["participant_id"]
        .astype(str)
        .to_numpy()
    )

    base_y = (
        data["y"]
        .to_numpy(float)
    )

    checkpoint = (
        fast_permutation_checkpoint_path(
            task
        )
    )

    required = [
        "permutation",
        "null_metric",
        "engine",
        "fast_permutation_signature",
    ]

    if checkpoint.exists():
        existing = safe_read_csv(
            checkpoint,
            required_columns=required,
        )

        signatures = set(
            existing[
                "fast_permutation_signature"
            ].astype(str)
        )

        if signatures != {
            FAST_PERM_SIGNATURE
        }:
            raise RuntimeError(
                "Existing accelerated checkpoint "
                "has incompatible signature."
            )

    else:
        existing = pd.DataFrame(
            columns=required
        )

    done = set(
        pd.to_numeric(
            existing[
                "permutation"
            ],
            errors="coerce",
        )
        .dropna()
        .astype(int)
    )

    todo = [
        p
        for p in range(
            1,
            int(target) + 1,
        )
        if p not in done
    ]

    if not todo:
        print(
            f"{task}: permutations 1-{target} "
            "already complete."
        )

        return (
            existing.sort_values(
                "permutation"
            ),
            0.0,
            0,
        )

    print(
        f"{task}: running "
        f"{len(todo)} new permutations "
        f"with n_jobs={n_jobs}"
    )

    start = time.perf_counter()

    # Small batches -> frequent durable checkpoints.
    batch_width = max(
        4,
        int(n_jobs) * 2,
    )

    working = existing.copy()

    for batch_start in range(
        0,
        len(todo),
        batch_width,
    ):
        batch = todo[
            batch_start:
            batch_start + batch_width
        ]

        batch_start_time = (
            time.perf_counter()
        )

        with parallel_backend(
            "loky",
            inner_max_num_threads=1,
        ):
            results = Parallel(
                n_jobs=int(n_jobs),
                batch_size=1,
                max_nbytes="1M",
                mmap_mode="r",
            )(
                delayed(
                    _single_fast_permutation
                )(
                    cache,
                    task,
                    permutation,
                    base_ids,
                    base_y,
                )
                for permutation in batch
            )

        working = pd.concat(
            [
                working,
                pd.DataFrame(results),
            ],
            ignore_index=True,
        )

        working = (
            working.sort_values(
                "permutation"
            )
            .drop_duplicates(
                "permutation",
                keep="last",
            )
            .reset_index(drop=True)
        )

        atomic_write_csv(
            working,
            checkpoint,
            required_columns=required,
        )

        batch_seconds = (
            time.perf_counter()
            - batch_start_time
        )

        print(
            f"{task}: completed through "
            f"permutation {max(batch)} | "
            f"checkpoint rows={len(working)} | "
            f"batch wall time="
            f"{batch_seconds / 60:.2f} min"
        )

    elapsed = (
        time.perf_counter()
        - start
    )

    return (
        working,
        elapsed,
        len(todo),
    )


cpu_count = os.cpu_count() or 2

# Deliberately leave half the machine available, capped at four workers,
# because Goal 2 / Goal 3 work may be running elsewhere.
N_JOBS_FAST = max(
    1,
    min(
        4,
        cpu_count // 2,
    ),
)

print(
    "Logical CPUs:",
    cpu_count,
)

print(
    "Accelerated permutation workers:",
    N_JOBS_FAST,
)

(
    dx_fast_benchmark,
    benchmark_seconds,
    n_new_benchmark,
) = run_fast_permutations(
    "diagnosis",
    dx_fast_cache,
    target=8,
    n_jobs=N_JOBS_FAST,
)

display(
    dx_fast_benchmark.tail(8)
)

if n_new_benchmark > 0:
    wall_seconds_per_permutation = (
        benchmark_seconds
        / n_new_benchmark
    )

    projected_hours_for_1000 = (
        wall_seconds_per_permutation
        * 1000
        / 3600
    )

    print()
    print(
        "BENCHMARK wall seconds / permutation:",
        f"{wall_seconds_per_permutation:.2f}",
    )

    print(
        "Approximate projected wall time for "
        "1,000 diagnosis permutations:",
        f"{projected_hours_for_1000:.2f} hours",
    )

print()
print(
    "FAST DIAGNOSIS PERMUTATION BENCHMARK: PASS"
)
print(
    "Do NOT resume the original slow permutation cell."
)

Logical CPUs: 24
Accelerated permutation workers: 4
diagnosis: permutations 1-8 already complete.


,permutation,null_metric,engine,fast_permutation_signature
0,1,0.443824,goal1-fast-permutation-fixed-splits-v1.0.0,8402bfa378c4d904b6cbe0620760d7754756899dda9f41...
1,2,0.443843,goal1-fast-permutation-fixed-splits-v1.0.0,8402bfa378c4d904b6cbe0620760d7754756899dda9f41...
2,3,0.457125,goal1-fast-permutation-fixed-splits-v1.0.0,8402bfa378c4d904b6cbe0620760d7754756899dda9f41...
3,4,0.397766,goal1-fast-permutation-fixed-splits-v1.0.0,8402bfa378c4d904b6cbe0620760d7754756899dda9f41...
4,5,0.396270,goal1-fast-permutation-fixed-splits-v1.0.0,8402bfa378c4d904b6cbe0620760d7754756899dda9f41...
5,6,0.393987,goal1-fast-permutation-fixed-splits-v1.0.0,8402bfa378c4d904b6cbe0620760d7754756899dda9f41...
6,7,0.410923,goal1-fast-permutation-fixed-splits-v1.0.0,8402bfa378c4d904b6cbe0620760d7754756899dda9f41...
7,8,0.412543,goal1-fast-permutation-fixed-splits-v1.0.0,8402bfa378c4d904b6cbe0620760d7754756899dda9f41...



FAST DIAGNOSIS PERMUTATION BENCHMARK: PASS
Do NOT resume the original slow permutation cell.


In [26]:
# =====================================================================
# DIAGNOSIS PERMUTATION VALIDITY DIAGNOSTIC
#
# Question:
# Do the fixed diagnosis-stratified outer folds themselves create
# below-0.5 AUROC under globally permuted labels?
#
# This fits NO Q model. It assigns every test participant only the
# corresponding outer-training-set prevalence.
# =====================================================================

def intercept_only_fixed_fold_null_auc(permutation):
    data = fast_primary_frame("diagnosis").copy()

    participant_ids = (
        data["participant_id"]
        .astype(str)
        .to_numpy()
    )

    original_y = (
        data["y"]
        .to_numpy(float)
    )

    rng = np.random.default_rng(
        BASE_SEED
        + 10_000_000
        + int(permutation)
    )

    permuted_y = rng.permutation(
        original_y
    )

    label_lookup = dict(
        zip(
            participant_ids,
            permuted_y,
        )
    )

    all_ids = set(participant_ids)

    repeat_aucs = []
    prevalence_rows = []

    for repeat in range(
        1,
        PERMUTATION_REPEATS + 1,
    ):
        y_repeat = []
        p_repeat = []

        for outer_fold in range(
            1,
            OUTER_FOLDS + 1,
        ):
            test_ids = set(
                split_manifest.loc[
                    split_manifest["repeat"].eq(repeat)
                    & split_manifest["outer_fold"].eq(
                        outer_fold
                    ),
                    "participant_id",
                ]
                .astype(str)
            )

            test_ids &= all_ids
            train_ids = (
                all_ids
                - test_ids
            )

            y_train = np.asarray(
                [
                    label_lookup[pid]
                    for pid in train_ids
                ],
                dtype=float,
            )

            y_test = np.asarray(
                [
                    label_lookup[pid]
                    for pid in test_ids
                ],
                dtype=float,
            )

            train_prevalence = float(
                np.mean(y_train)
            )

            test_prevalence = float(
                np.mean(y_test)
            )

            # Intercept-only prediction:
            # every participant in this held-out fold receives
            # its corresponding training-set prevalence.
            p_test = np.full(
                len(y_test),
                train_prevalence,
                dtype=float,
            )

            y_repeat.extend(
                y_test.tolist()
            )

            p_repeat.extend(
                p_test.tolist()
            )

            prevalence_rows.append({
                "permutation": int(permutation),
                "repeat": int(repeat),
                "outer_fold": int(outer_fold),
                "train_prevalence": train_prevalence,
                "test_prevalence": test_prevalence,
                "prevalence_difference_test_minus_train": (
                    test_prevalence
                    - train_prevalence
                ),
            })

        y_repeat = np.asarray(
            y_repeat,
            dtype=int,
        )

        p_repeat = np.asarray(
            p_repeat,
            dtype=float,
        )

        repeat_aucs.append(
            roc_auc_score(
                y_repeat,
                p_repeat,
            )
        )

    return (
        float(
            np.mean(
                repeat_aucs
            )
        ),
        repeat_aucs,
        prevalence_rows,
    )


diagnostic_rows = []
prevalence_rows = []

for permutation in range(1, 9):
    (
        intercept_auc,
        repeat_aucs,
        local_prevalence,
    ) = intercept_only_fixed_fold_null_auc(
        permutation
    )

    fast_metric = float(
        dx_fast_benchmark.loc[
            dx_fast_benchmark[
                "permutation"
            ].eq(permutation),
            "null_metric",
        ].iloc[0]
    )

    diagnostic_rows.append({
        "permutation": permutation,
        "CoreQ_null_AUROC": fast_metric,
        "intercept_only_fixed_fold_AUROC": (
            intercept_auc
        ),
        "intercept_repeat_min": float(
            np.min(repeat_aucs)
        ),
        "intercept_repeat_max": float(
            np.max(repeat_aucs)
        ),
    })

    prevalence_rows.extend(
        local_prevalence
    )


diagnostic = pd.DataFrame(
    diagnostic_rows
)

prevalence_diagnostic = pd.DataFrame(
    prevalence_rows
)

display(diagnostic)


print()
print(
    "Mean Core-Q null AUROC:",
    f"{diagnostic['CoreQ_null_AUROC'].mean():.6f}",
)

print(
    "Mean intercept-only fixed-fold AUROC:",
    f"{diagnostic['intercept_only_fixed_fold_AUROC'].mean():.6f}",
)


# Fold-level inverse-prevalence relationship.
prevalence_corr = float(
    prevalence_diagnostic[
        [
            "train_prevalence",
            "test_prevalence",
        ]
    ]
    .corr()
    .iloc[0, 1]
)

print(
    "Correlation between train and test "
    "permuted prevalence across folds:",
    f"{prevalence_corr:.6f}",
)


atomic_write_csv(
    diagnostic,
    FAST_PERM_AUDIT
    / "diagnosis_fixed_fold_null_diagnostic.csv",
)

atomic_write_csv(
    prevalence_diagnostic,
    FAST_PERM_AUDIT
    / "diagnosis_fixed_fold_prevalence_diagnostic.csv",
)

print()
print(
    "DIAGNOSIS FIXED-FOLD PERMUTATION DIAGNOSTIC: COMPLETE"
)

,permutation,CoreQ_null_AUROC,intercept_only_fixed_fold_AUROC,intercept_repeat_min,intercept_repeat_max
0,1,0.443824,0.424362,0.368287,0.455792
1,2,0.443843,0.433937,0.401371,0.473005
2,3,0.457125,0.422464,0.373418,0.475930
3,4,0.397766,0.421346,0.382048,0.460155
4,5,0.396270,0.409983,0.359513,0.475930
5,6,0.393987,0.415425,0.371356,0.458717
6,7,0.410923,0.424171,0.389912,0.481013
7,8,0.412543,0.434949,0.388569,0.467300



Mean Core-Q null AUROC: 0.419535
Mean intercept-only fixed-fold AUROC: 0.423330
Correlation between train and test permuted prevalence across folds: -0.999934

DIAGNOSIS FIXED-FOLD PERMUTATION DIAGNOSTIC: COMPLETE


In [27]:
%run -i "goal1_diagnosis_permutation_corrected_v1_1.py"

GOAL 1 — CORRECTED DIAGNOSIS PERMUTATION PREFLIGHT
Old fixed-manifest Core-Q null mean (8 draws): 0.419535
Old fixed-manifest intercept-only mean: 0.423330
Old train/test prevalence correlation: -0.999934
Regenerated-stratification intercept-only mean (200 draws): 0.490171
Primary Core-Q AUROC, 10 repeats (preserved): 0.743038
Matched observed AUROC, repeats 1-3 (formal permutation statistic): 0.740410
True-label split reproduction, repeats 1-3: PASS
Corrected engine signature: bb3670a63db23af8

Corrected diagnosis permutation: 4 new draws | target=4 | workers=4 | CV=5 folds x 3 repeats
completed through permutation 4 | checkpoint rows=4 | batch wall=12.68 min


,permutation,null_metric,label_permutation_seed,outer_repeat_seeds,split_manifest_hash,engine,signature
0,1,0.496835,30260826,20260825;20260826;20260827,066bbaa2bc98317ea2fb4e1c27111edeceefd79b619d21...,goal1-diagnosis-permutation-regenerated-strati...,bb3670a63db23af8023c337c9a3d344c53878ec8f5e8d2...
1,2,0.494342,30260827,20260825;20260826;20260827,58d939b82e70cef2d9b9890060f788ba2d9bf6ad3f5db4...,goal1-diagnosis-permutation-regenerated-strati...,bb3670a63db23af8023c337c9a3d344c53878ec8f5e8d2...
2,3,0.532764,30260828,20260825;20260826;20260827,c93522765515eb1291b0bfd70db4fca792dd10840a9c77...,goal1-diagnosis-permutation-regenerated-strati...,bb3670a63db23af8023c337c9a3d344c53878ec8f5e8d2...
3,4,0.428973,30260829,20260825;20260826;20260827,ea1f6365d649fcc8cd458bdf2e185223aad807aa4b8ad7...,goal1-diagnosis-permutation-regenerated-strati...,bb3670a63db23af8023c337c9a3d344c53878ec8f5e8d2...



Observed wall seconds per completed permutation at current 4-worker throughput: 190.19
Naive projected wall time for 1000 at this throughput: 52.83 hours

CORRECTED DIAGNOSIS PERMUTATION BENCHMARK: COMPLETE
Do NOT use the old fixed-manifest null.
Next, inspect this timing benchmark before calling run_corrected_diagnosis_permutations(target=1000, ...).


In [28]:
# ================================================================
# 8-WORKER THROUGHPUT BENCHMARK
# Adds 8 genuine corrected permutations: 5 through 12.
# All results are checkpointed and count toward the final 1000.
# ================================================================

(
    corrected_dx_8worker_test,
    seconds_8worker,
    n_new_8worker,
) = run_corrected_diagnosis_permutations(
    target=12,
    n_jobs=8,
)

display(
    corrected_dx_8worker_test.tail(12)
)

if n_new_8worker > 0:
    throughput_per_hour = (
        n_new_8worker
        / seconds_8worker
        * 3600
    )

    remaining = (
        1000
        - len(
            corrected_dx_8worker_test
        )
    )

    projected_remaining_hours = (
        remaining
        / throughput_per_hour
    )

    print()
    print(
        "New permutations completed:",
        n_new_8worker,
    )

    print(
        "8-worker wall time:",
        f"{seconds_8worker / 60:.2f} min",
    )

    print(
        "Throughput:",
        f"{throughput_per_hour:.2f} permutations/hour",
    )

    print(
        "Completed total:",
        len(corrected_dx_8worker_test),
        "/ 1000",
    )

    print(
        "Projected remaining wall time:",
        f"{projected_remaining_hours:.2f} hours",
    )

print()
print("8-WORKER BENCHMARK: COMPLETE")

Corrected diagnosis permutation: 8 new draws | target=12 | workers=8 | CV=5 folds x 3 repeats
completed through permutation 12 | checkpoint rows=12 | batch wall=12.47 min


,permutation,null_metric,label_permutation_seed,outer_repeat_seeds,split_manifest_hash,engine,signature
0,1,0.496835,30260826,20260825;20260826;20260827,066bbaa2bc98317ea2fb4e1c27111edeceefd79b619d21...,goal1-diagnosis-permutation-regenerated-strati...,bb3670a63db23af8023c337c9a3d344c53878ec8f5e8d2...
1,2,0.494342,30260827,20260825;20260826;20260827,58d939b82e70cef2d9b9890060f788ba2d9bf6ad3f5db4...,goal1-diagnosis-permutation-regenerated-strati...,bb3670a63db23af8023c337c9a3d344c53878ec8f5e8d2...
2,3,0.532764,30260828,20260825;20260826;20260827,c93522765515eb1291b0bfd70db4fca792dd10840a9c77...,goal1-diagnosis-permutation-regenerated-strati...,bb3670a63db23af8023c337c9a3d344c53878ec8f5e8d2...
3,4,0.428973,30260829,20260825;20260826;20260827,ea1f6365d649fcc8cd458bdf2e185223aad807aa4b8ad7...,goal1-diagnosis-permutation-regenerated-strati...,bb3670a63db23af8023c337c9a3d344c53878ec8f5e8d2...
4,5,0.482643,30260830,20260825;20260826;20260827,840bd46c749cedbe8967d69a9cc318f4e943d86ace3f34...,goal1-diagnosis-permutation-regenerated-strati...,bb3670a63db23af8023c337c9a3d344c53878ec8f5e8d2...
5,6,0.455440,30260831,20260825;20260826;20260827,f1587f9a378b974b23a754b0bcaf5f3ce9efbcc7358312...,goal1-diagnosis-permutation-regenerated-strati...,bb3670a63db23af8023c337c9a3d344c53878ec8f5e8d2...
6,7,0.437029,30260832,20260825;20260826;20260827,acaca0a1911df01133dc39496bd1c9bad5f6d8a13201ab...,goal1-diagnosis-permutation-regenerated-strati...,bb3670a63db23af8023c337c9a3d344c53878ec8f5e8d2...
7,8,0.439714,30260833,20260825;20260826;20260827,c39884956f64e175c14d6d03c7eeb1a9cb5190579320b4...,goal1-diagnosis-permutation-regenerated-strati...,bb3670a63db23af8023c337c9a3d344c53878ec8f5e8d2...
8,9,0.504155,30260834,20260825;20260826;20260827,b62658ec95d25d219d8e4d7163c2494accf8b963ecc408...,goal1-diagnosis-permutation-regenerated-strati...,bb3670a63db23af8023c337c9a3d344c53878ec8f5e8d2...
9,10,0.510229,30260835,20260825;20260826;20260827,9f366b028ca9c139353dcf2d4a89acb07698edd6c00298...,goal1-diagnosis-permutation-regenerated-strati...,bb3670a63db23af8023c337c9a3d344c53878ec8f5e8d2...



New permutations completed: 8
8-worker wall time: 12.47 min
Throughput: 38.48 permutations/hour
Completed total: 12 / 1000
Projected remaining wall time: 25.67 hours

8-WORKER BENCHMARK: COMPLETE


In [29]:
# ================================================================
# FINAL THROUGHPUT CHECK — 12 WORKERS
#
# Already complete: permutations 1-12
# This computes only permutations 13-24.
#
# These are REAL final corrected permutations and remain part of
# the final 1000-draw null distribution.
# ================================================================

(
    corrected_dx_12worker_test,
    seconds_12worker,
    n_new_12worker,
) = run_corrected_diagnosis_permutations(
    target=24,
    n_jobs=12,
)

display(
    corrected_dx_12worker_test.tail(24)
)

if n_new_12worker > 0:

    throughput_12 = (
        n_new_12worker
        / seconds_12worker
        * 3600
    )

    remaining_12 = (
        1000
        - len(corrected_dx_12worker_test)
    )

    projected_remaining_hours_12 = (
        remaining_12
        / throughput_12
    )

    print()
    print(
        "New permutations completed:",
        n_new_12worker,
    )

    print(
        "12-worker wall time:",
        f"{seconds_12worker / 60:.2f} min",
    )

    print(
        "12-worker throughput:",
        f"{throughput_12:.2f} permutations/hour",
    )

    print(
        "Completed total:",
        len(corrected_dx_12worker_test),
        "/ 1000",
    )

    print(
        "Projected remaining wall time:",
        f"{projected_remaining_hours_12:.2f} hours",
    )

print()
print("12-WORKER BENCHMARK: COMPLETE")

Corrected diagnosis permutation: 12 new draws | target=24 | workers=12 | CV=5 folds x 3 repeats
completed through permutation 24 | checkpoint rows=24 | batch wall=12.75 min


,permutation,null_metric,label_permutation_seed,outer_repeat_seeds,split_manifest_hash,engine,signature
0,1,0.496835,30260826,20260825;20260826;20260827,066bbaa2bc98317ea2fb4e1c27111edeceefd79b619d21...,goal1-diagnosis-permutation-regenerated-strati...,bb3670a63db23af8023c337c9a3d344c53878ec8f5e8d2...
1,2,0.494342,30260827,20260825;20260826;20260827,58d939b82e70cef2d9b9890060f788ba2d9bf6ad3f5db4...,goal1-diagnosis-permutation-regenerated-strati...,bb3670a63db23af8023c337c9a3d344c53878ec8f5e8d2...
2,3,0.532764,30260828,20260825;20260826;20260827,c93522765515eb1291b0bfd70db4fca792dd10840a9c77...,goal1-diagnosis-permutation-regenerated-strati...,bb3670a63db23af8023c337c9a3d344c53878ec8f5e8d2...
3,4,0.428973,30260829,20260825;20260826;20260827,ea1f6365d649fcc8cd458bdf2e185223aad807aa4b8ad7...,goal1-diagnosis-permutation-regenerated-strati...,bb3670a63db23af8023c337c9a3d344c53878ec8f5e8d2...
4,5,0.482643,30260830,20260825;20260826;20260827,840bd46c749cedbe8967d69a9cc318f4e943d86ace3f34...,goal1-diagnosis-permutation-regenerated-strati...,bb3670a63db23af8023c337c9a3d344c53878ec8f5e8d2...
5,6,0.455440,30260831,20260825;20260826;20260827,f1587f9a378b974b23a754b0bcaf5f3ce9efbcc7358312...,goal1-diagnosis-permutation-regenerated-strati...,bb3670a63db23af8023c337c9a3d344c53878ec8f5e8d2...
6,7,0.437029,30260832,20260825;20260826;20260827,acaca0a1911df01133dc39496bd1c9bad5f6d8a13201ab...,goal1-diagnosis-permutation-regenerated-strati...,bb3670a63db23af8023c337c9a3d344c53878ec8f5e8d2...
7,8,0.439714,30260833,20260825;20260826;20260827,c39884956f64e175c14d6d03c7eeb1a9cb5190579320b4...,goal1-diagnosis-permutation-regenerated-strati...,bb3670a63db23af8023c337c9a3d344c53878ec8f5e8d2...
8,9,0.504155,30260834,20260825;20260826;20260827,b62658ec95d25d219d8e4d7163c2494accf8b963ecc408...,goal1-diagnosis-permutation-regenerated-strati...,bb3670a63db23af8023c337c9a3d344c53878ec8f5e8d2...
9,10,0.510229,30260835,20260825;20260826;20260827,9f366b028ca9c139353dcf2d4a89acb07698edd6c00298...,goal1-diagnosis-permutation-regenerated-strati...,bb3670a63db23af8023c337c9a3d344c53878ec8f5e8d2...



New permutations completed: 12
12-worker wall time: 12.75 min
12-worker throughput: 56.48 permutations/hour
Completed total: 24 / 1000
Projected remaining wall time: 17.28 hours

12-WORKER BENCHMARK: COMPLETE


In [30]:
# ================================================================
# FINAL CORRECTED DIAGNOSIS PERMUTATION RUN
# ================================================================

(
    corrected_dx_final,
    corrected_final_seconds,
    corrected_final_n_new,
) = run_corrected_diagnosis_permutations(
    target=1000,
    n_jobs=12,
)

print()
print(
    "New permutations computed in this run:",
    corrected_final_n_new,
)

print(
    "Final-run wall time:",
    f"{corrected_final_seconds / 3600:.2f} hours",
)

print(
    "Checkpoint rows:",
    len(corrected_dx_final),
)

print(
    "Unique permutations:",
    corrected_dx_final["permutation"].nunique(),
)

Corrected diagnosis permutation: 976 new draws | target=1000 | workers=12 | CV=5 folds x 3 repeats
completed through permutation 48 | checkpoint rows=48 | batch wall=25.49 min
completed through permutation 72 | checkpoint rows=72 | batch wall=25.45 min
completed through permutation 96 | checkpoint rows=96 | batch wall=26.05 min
completed through permutation 120 | checkpoint rows=120 | batch wall=24.68 min
completed through permutation 144 | checkpoint rows=144 | batch wall=22.93 min
completed through permutation 168 | checkpoint rows=168 | batch wall=22.97 min
completed through permutation 192 | checkpoint rows=192 | batch wall=22.88 min
completed through permutation 216 | checkpoint rows=216 | batch wall=23.15 min
completed through permutation 240 | checkpoint rows=240 | batch wall=23.25 min
completed through permutation 264 | checkpoint rows=264 | batch wall=23.43 min
completed through permutation 288 | checkpoint rows=288 | batch wall=23.28 min
completed through permutation 312 | ch

NameError: name 'dtype' is not defined

In [31]:
# ================================================================
# RECOVERY STEP 1 — VERIFY THE SAVED CORRECTED NULL CHECKPOINT
# ================================================================

import numpy as np
import pandas as pd

checkpoint_now = pd.read_csv(
    CORRECTED_CHECKPOINT
)

checkpoint_now = (
    checkpoint_now
    .sort_values("permutation")
    .reset_index(drop=True)
)

print(
    "Checkpoint rows:",
    len(checkpoint_now),
)

print(
    "Unique permutations:",
    checkpoint_now[
        "permutation"
    ].nunique(),
)

print(
    "First permutation:",
    int(
        checkpoint_now[
            "permutation"
        ].min()
    ),
)

print(
    "Last permutation:",
    int(
        checkpoint_now[
            "permutation"
        ].max()
    ),
)

print(
    "Finite null metrics:",
    bool(
        np.isfinite(
            checkpoint_now[
                "null_metric"
            ]
        ).all()
    ),
)


expected_so_far = set(
    range(
        1,
        int(
            checkpoint_now[
                "permutation"
            ].max()
        ) + 1,
    )
)

observed_so_far = set(
    checkpoint_now[
        "permutation"
    ].astype(int)
)

missing_so_far = sorted(
    expected_so_far
    - observed_so_far
)


if checkpoint_now[
    "permutation"
].duplicated().any():

    raise RuntimeError(
        "Duplicate permutation IDs detected."
    )


if missing_so_far:

    raise RuntimeError(
        f"Missing checkpoint IDs: "
        f"{missing_so_far[:20]}"
    )


if not np.isfinite(
    checkpoint_now[
        "null_metric"
    ]
).all():

    raise RuntimeError(
        "Non-finite null metric in checkpoint."
    )


print()
print(
    "CORRECTED NULL CHECKPOINT: PASS"
)

Checkpoint rows: 360
Unique permutations: 360
First permutation: 1
Last permutation: 360
Finite null metrics: True

CORRECTED NULL CHECKPOINT: PASS


In [32]:
# ================================================================
# RECOVERY STEP 2 — TERMINATE ONLY THE FAILED LOKY WORKER POOL
# ================================================================

import gc

from joblib.externals.loky import (
    get_reusable_executor,
)


try:
    executor = (
        get_reusable_executor()
    )

    executor.shutdown(
        wait=True,
        kill_workers=True,
    )

    print(
        "Old loky worker pool terminated."
    )

except Exception as exc:

    print(
        "Worker-pool cleanup returned:",
        type(exc).__name__,
        str(exc),
    )


gc.collect()

print(
    "NOTEBOOK 10 KERNEL REMAINS ACTIVE."
)

Old loky worker pool terminated.
NOTEBOOK 10 KERNEL REMAINS ACTIVE.


In [33]:
# ================================================================
# RECOVERY STEP 3 — RE-RUN THE FAILED RANGE WITH FRESH WORKERS
#
# Existing checkpoint: 1–360
# target=384 computes ONLY 361–384.
#
# These are real final permutations.
# ================================================================

(
    recovery_test,
    recovery_seconds,
    recovery_n_new,
) = run_corrected_diagnosis_permutations(
    target=384,
    n_jobs=8,
)


print()
print(
    "New permutations:",
    recovery_n_new,
)

print(
    "Checkpoint rows:",
    len(recovery_test),
)

print(
    "Last permutation:",
    int(
        recovery_test[
            "permutation"
        ].max()
    ),
)

print(
    "Recovery batch wall time:",
    f"{recovery_seconds / 60:.2f} min",
)


if len(recovery_test) != 384:
    raise RuntimeError(
        "Recovery did not reach permutation 384."
    )


if (
    recovery_test[
        "permutation"
    ].nunique()
    != 384
):
    raise RuntimeError(
        "Recovery checkpoint IDs are not unique."
    )


print()
print(
    "PERMUTATIONS 361–384 RECOVERY: PASS"
)

Corrected diagnosis permutation: 24 new draws | target=384 | workers=8 | CV=5 folds x 3 repeats
completed through permutation 376 | checkpoint rows=376 | batch wall=23.36 min
completed through permutation 384 | checkpoint rows=384 | batch wall=11.34 min

New permutations: 24
Checkpoint rows: 384
Last permutation: 384
Recovery batch wall time: 34.70 min

PERMUTATIONS 361–384 RECOVERY: PASS


In [34]:
# ================================================================
# RECOVERY STEP 4 — RESILIENT COMPLETION DRIVER
#
# Scientific computation is unchanged.
# This wrapper only:
#   - resumes from the atomic checkpoint,
#   - recreates workers after a transient infrastructure failure,
#   - retries the SAME deterministic permutation IDs.
#
# It never skips a failed permutation.
# ================================================================

import gc
import time
import traceback

from joblib.externals.loky import (
    get_reusable_executor,
)


FINAL_TARGET = 1000
RECOVERY_WORKERS = 8
MAX_CONSECUTIVE_FAILURES = 3

consecutive_failures = 0


while True:

    current = pd.read_csv(
        CORRECTED_CHECKPOINT
    )

    current = (
        current
        .sort_values(
            "permutation"
        )
        .drop_duplicates(
            subset=[
                "permutation"
            ],
            keep="last",
        )
        .reset_index(
            drop=True
        )
    )

    completed = int(
        current[
            "permutation"
        ].nunique()
    )

    print()
    print(
        "=" * 68
    )

    print(
        f"Checkpoint status: "
        f"{completed}/{FINAL_TARGET}"
    )

    print(
        "=" * 68
    )


    if completed >= FINAL_TARGET:

        print(
            "Target reached."
        )

        break


    try:

        (
            _result,
            _seconds,
            _n_new,
        ) = (
            run_corrected_diagnosis_permutations(
                target=FINAL_TARGET,
                n_jobs=RECOVERY_WORKERS,
            )
        )

        consecutive_failures = 0


    except Exception as exc:

        consecutive_failures += 1

        print()
        print(
            "TRANSIENT PARALLEL RUN FAILURE"
        )

        print(
            "Exception:",
            type(exc).__name__,
            str(exc),
        )

        # Read the atomic checkpoint again.
        current_after_error = (
            pd.read_csv(
                CORRECTED_CHECKPOINT
            )
        )

        completed_after_error = int(
            current_after_error[
                "permutation"
            ].nunique()
        )

        print(
            "Checkpoint safely contains:",
            completed_after_error,
            "permutations",
        )


        if (
            consecutive_failures
            >= MAX_CONSECUTIVE_FAILURES
        ):

            raise RuntimeError(
                "Three consecutive failures occurred "
                "without successful recovery. "
                "Stop here and inspect rather than "
                "continuing automatically."
            ) from exc


        # Kill only this notebook's loky workers.
        try:

            executor = (
                get_reusable_executor()
            )

            executor.shutdown(
                wait=True,
                kill_workers=True,
            )

        except Exception:
            pass


        gc.collect()

        print(
            "Fresh worker pool will be created."
        )

        print(
            "Retrying the SAME missing permutation IDs "
            "in 15 seconds..."
        )

        time.sleep(15)

        continue


Checkpoint status: 384/1000
Corrected diagnosis permutation: 616 new draws | target=1000 | workers=8 | CV=5 folds x 3 repeats
completed through permutation 400 | checkpoint rows=400 | batch wall=22.62 min
completed through permutation 416 | checkpoint rows=416 | batch wall=23.25 min
completed through permutation 432 | checkpoint rows=432 | batch wall=22.07 min
completed through permutation 448 | checkpoint rows=448 | batch wall=22.82 min
completed through permutation 464 | checkpoint rows=464 | batch wall=23.23 min
completed through permutation 480 | checkpoint rows=480 | batch wall=23.69 min
completed through permutation 496 | checkpoint rows=496 | batch wall=23.75 min
completed through permutation 512 | checkpoint rows=512 | batch wall=24.94 min
completed through permutation 528 | checkpoint rows=528 | batch wall=26.17 min
completed through permutation 544 | checkpoint rows=544 | batch wall=24.43 min
completed through permutation 560 | checkpoint rows=560 | batch wall=23.69 min

TRA

In [36]:
# ================================================================
# GOAL 1 — FINALIZE CORRECTED DIAGNOSIS PERMUTATION NULL
#
# HARD GATES:
#   - exactly 1,000 permutations
#   - IDs exactly 1..1000
#   - no duplicates / missing IDs
#   - finite AUROCs
#   - correct deterministic label seeds
#   - one engine / one signature
#   - matched 3-repeat observed AUROC
#
# The invalid old fixed-stratified diagnosis null is NOT used.
# ================================================================

from datetime import datetime, timezone
import json
import shutil
import numpy as np
import pandas as pd


# ------------------------------------------------
# 1. Reload from durable checkpoint
# ------------------------------------------------

final_dx_null = safe_read_csv(
    CORRECTED_CHECKPOINT,
    required_columns=[
        "permutation",
        "null_metric",
        "label_permutation_seed",
        "outer_repeat_seeds",
        "split_manifest_hash",
        "engine",
        "signature",
    ],
)

final_dx_null = (
    final_dx_null
    .sort_values("permutation")
    .reset_index(drop=True)
)


# ------------------------------------------------
# 2. Completeness
# ------------------------------------------------

expected_ids = np.arange(
    1,
    1001,
    dtype=int,
)

observed_ids = (
    final_dx_null[
        "permutation"
    ]
    .astype(int)
    .to_numpy()
)

if len(final_dx_null) != 1000:
    raise RuntimeError(
        f"Expected 1000 rows; found {len(final_dx_null)}."
    )

if final_dx_null[
    "permutation"
].nunique() != 1000:
    raise RuntimeError(
        "Permutation IDs are not unique."
    )

if not np.array_equal(
    observed_ids,
    expected_ids,
):
    missing = sorted(
        set(expected_ids)
        - set(observed_ids)
    )

    extra = sorted(
        set(observed_ids)
        - set(expected_ids)
    )

    raise RuntimeError(
        "Permutation ID mismatch. "
        f"Missing={missing[:20]} | "
        f"Extra={extra[:20]}"
    )


# ------------------------------------------------
# 3. Metric integrity
# ------------------------------------------------

null_values = pd.to_numeric(
    final_dx_null[
        "null_metric"
    ],
    errors="coerce",
).to_numpy(float)

if not np.isfinite(
    null_values
).all():
    raise RuntimeError(
        "Non-finite diagnosis null AUROC found."
    )

if np.any(
    (null_values < 0)
    | (null_values > 1)
):
    raise RuntimeError(
        "Diagnosis null AUROC outside [0,1]."
    )


# ------------------------------------------------
# 4. Engine/signature integrity
# ------------------------------------------------

engines = set(
    final_dx_null[
        "engine"
    ].astype(str)
)

signatures = set(
    final_dx_null[
        "signature"
    ].astype(str)
)

if engines != {
    CORRECTED_ENGINE
}:
    raise RuntimeError(
        f"Unexpected engines: {engines}"
    )

if signatures != {
    CORRECTED_SIGNATURE
}:
    raise RuntimeError(
        "Corrected diagnosis signature mismatch."
    )


# ------------------------------------------------
# 5. Deterministic label-seed integrity
# ------------------------------------------------

expected_label_seeds = (
    BASE_SEED
    + 10_000_000
    + expected_ids
)

actual_label_seeds = (
    pd.to_numeric(
        final_dx_null[
            "label_permutation_seed"
        ],
        errors="raise",
    )
    .astype(int)
    .to_numpy()
)

if not np.array_equal(
    actual_label_seeds,
    expected_label_seeds,
):
    raise RuntimeError(
        "Diagnosis label permutation seeds "
        "do not match the frozen deterministic rule."
    )


# ------------------------------------------------
# 6. Outer repeat seed integrity
# ------------------------------------------------

expected_outer_seed_string = ";".join(
    str(
        BASE_SEED + r
    )
    for r in range(
        CORRECTED_REPEATS
    )
)

if set(
    final_dx_null[
        "outer_repeat_seeds"
    ].astype(str)
) != {
    expected_outer_seed_string
}:
    raise RuntimeError(
        "Outer-repeat seed provenance mismatch."
    )


# ------------------------------------------------
# 7. Split-manifest hash integrity
# ------------------------------------------------

if final_dx_null[
    "split_manifest_hash"
].isna().any():
    raise RuntimeError(
        "Missing split-manifest hash."
    )

# Every independently shuffled label vector should normally
# generate its own manifest. Hash collisions here would be
# scientifically suspicious.
if final_dx_null[
    "split_manifest_hash"
].nunique() != 1000:
    raise RuntimeError(
        "Expected 1000 unique regenerated "
        "split-manifest hashes."
    )


# ------------------------------------------------
# 8. Formal empirical P value
#
# IMPORTANT:
# matched observed statistic = first 3 true-label repeats,
# because the corrected null uses exactly 3 repeats.
# ------------------------------------------------

observed_dx_3rep = float(
    CORRECTED_OBSERVED_AUROC
)

if not np.isfinite(
    observed_dx_3rep
):
    raise RuntimeError(
        "Matched observed diagnosis AUROC is non-finite."
    )

extreme_dx = int(
    np.sum(
        null_values
        >= observed_dx_3rep
    )
)

empirical_p_dx = (
    1 + extreme_dx
) / (
    1 + len(null_values)
)


# ------------------------------------------------
# 9. Useful null diagnostics
# ------------------------------------------------

dx_null_summary = {
    "created_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),

    "status": "PASS",

    "task": "diagnosis",

    "metric": "AUROC",

    "primary_10_repeat_AUROC": float(
        PRIMARY_10_REPEAT_AUROC
    ),

    "matched_3_repeat_observed_AUROC": (
        observed_dx_3rep
    ),

    "permutation_repeats": int(
        CORRECTED_REPEATS
    ),

    "n_permutations": int(
        len(null_values)
    ),

    "extreme_null_draws": (
        extreme_dx
    ),

    "empirical_p": float(
        empirical_p_dx
    ),

    "null_mean": float(
        np.mean(null_values)
    ),

    "null_sd": float(
        np.std(
            null_values,
            ddof=1,
        )
    ),

    "null_median": float(
        np.median(null_values)
    ),

    "null_q025": float(
        np.quantile(
            null_values,
            0.025,
        )
    ),

    "null_q975": float(
        np.quantile(
            null_values,
            0.975,
        )
    ),

    "engine": (
        CORRECTED_ENGINE
    ),

    "signature": (
        CORRECTED_SIGNATURE
    ),

    "old_fixed_manifest_diagnosis_null_used": False,

    "outer_stratification_regenerated_after_permutation": True,

    "inner_stratification_regenerated_after_permutation": True,

    "fold_safe_QCHAN_recomputed": True,

    "preprocessing_and_tuning_recomputed": True,
}


# ------------------------------------------------
# 10. Produce canonical corrected table
# ------------------------------------------------

dx_perm_corrected = (
    final_dx_null.copy()
)

dx_perm_corrected[
    "task"
] = "diagnosis"

dx_perm_corrected[
    "metric"
] = "AUROC"

dx_perm_corrected[
    "observed_metric"
] = observed_dx_3rep

dx_perm_corrected[
    "empirical_p"
] = empirical_p_dx

dx_perm_corrected[
    "permutation_repeats"
] = CORRECTED_REPEATS

dx_perm_corrected[
    "run_mode"
] = RUN_MODE

dx_perm_corrected[
    "permutation_design"
] = (
    "participant-level label permutation; "
    "outer and inner diagnosis stratification "
    "regenerated after each permutation"
)


# ------------------------------------------------
# 11. Preserve any old canonical diagnosis-null file
#     before replacing it.
# ------------------------------------------------

canonical_dx_path = (
    TABLES
    / "diagnosis_coreq_permutation_null.csv"
)

if canonical_dx_path.exists():

    archived_dx_path = (
        CORRECTED_AUDIT
        / (
            "INVALID_OLD_"
            "diagnosis_coreq_permutation_null.csv"
        )
    )

    if not archived_dx_path.exists():
        shutil.copy2(
            canonical_dx_path,
            archived_dx_path,
        )


# ------------------------------------------------
# 12. Write authoritative corrected outputs
# ------------------------------------------------

atomic_write_csv(
    dx_perm_corrected,
    CORRECTED_TABLES
    / (
        "diagnosis_coreq_permutation_"
        "null_corrected.csv"
    ),
)

atomic_write_csv(
    dx_perm_corrected,
    canonical_dx_path,
)

atomic_write_json(
    dx_null_summary,
    CORRECTED_TABLES
    / (
        "diagnosis_permutation_"
        "final_summary.json"
    ),
)


print(
    "=" * 76
)

print(
    "CORRECTED DIAGNOSIS PERMUTATION NULL: FINAL PASS"
)

print(
    "=" * 76
)

print(
    "Permutations:",
    len(null_values),
)

print(
    "Matched observed AUROC "
    "(3 repeats):",
    f"{observed_dx_3rep:.6f}",
)

print(
    "Primary AUROC "
    "(10 repeats; unchanged):",
    f"{PRIMARY_10_REPEAT_AUROC:.6f}",
)

print(
    "Null mean:",
    f"{np.mean(null_values):.6f}",
)

print(
    "Null SD:",
    f"{np.std(null_values, ddof=1):.6f}",
)

print(
    "95% null interval:",
    (
        f"[{np.quantile(null_values, 0.025):.6f}, "
        f"{np.quantile(null_values, 0.975):.6f}]"
    ),
)

print(
    "Extreme null draws:",
    extreme_dx,
)

print(
    "Empirical P:",
    f"{empirical_p_dx:.6g}",
)

print(
    "Old fixed-manifest diagnosis null used:",
    False,
)

CORRECTED DIAGNOSIS PERMUTATION NULL: FINAL PASS
Permutations: 1000
Matched observed AUROC (3 repeats): 0.740410
Primary AUROC (10 repeats; unchanged): 0.743038
Null mean: 0.487495
Null SD: 0.038132
95% null interval: [0.429441, 0.572887]
Extreme null draws: 0
Empirical P: 0.000999001
Old fixed-manifest diagnosis null used: False


In [37]:
# ================================================================
# GOAL 1 — CHECK SEVERITY PERMUTATION STATUS
# ================================================================

severity_checkpoint = (
    CHECKPOINTS
    / "severity__coreq_permutations.csv"
)

severity_canonical = (
    TABLES
    / "severity_coreq_permutation_null.csv"
)


print(
    "Severity checkpoint exists:",
    severity_checkpoint.exists(),
)

if severity_checkpoint.exists():

    sev_checkpoint = pd.read_csv(
        severity_checkpoint
    )

    print(
        "Severity checkpoint rows:",
        len(sev_checkpoint),
    )

    print(
        "Severity unique permutation IDs:",
        sev_checkpoint[
            "permutation"
        ].nunique()
        if "permutation" in sev_checkpoint
        else "<missing column>",
    )

    if len(
        sev_checkpoint
    ):

        print(
            "Highest severity permutation ID:",
            int(
                pd.to_numeric(
                    sev_checkpoint[
                        "permutation"
                    ],
                    errors="coerce",
                ).max()
            ),
        )


print()

print(
    "Canonical severity permutation table exists:",
    severity_canonical.exists(),
)

if severity_canonical.exists():

    sev_canonical = pd.read_csv(
        severity_canonical
    )

    print(
        "Canonical severity rows:",
        len(sev_canonical),
    )

    if (
        len(sev_canonical) == 1000
        and {
            "null_metric",
            "observed_metric",
            "empirical_p",
        }.issubset(
            sev_canonical.columns
        )
    ):

        print(
            "Severity empirical P:",
            sev_canonical[
                "empirical_p"
            ].iloc[0],
        )

        print()
        print(
            "SEVERITY PERMUTATION NULL: "
            "ALREADY COMPLETE"
        )

    else:

        print()
        print(
            "SEVERITY PERMUTATION NULL: "
            "NOT YET FINAL"
        )

else:

    print()
    print(
        "SEVERITY PERMUTATION NULL: "
        "NOT YET FINAL"
    )

Severity checkpoint exists: False

Canonical severity permutation table exists: False

SEVERITY PERMUTATION NULL: NOT YET FINAL


In [38]:
# ================================================================
# GOAL 1 — SEVERITY PERMUTATION NULL
# Restart-safe fixed-split 5x3 implementation
#
# SCIENCE:
#   - 145 primary ALS participants
#   - one <=60-day severity index pair / participant
#   - participant-level bulbar-score permutation
#   - fixed master participant folds
#   - 5 outer folds x 3 repeats
#   - fold-safe QCHAN rebuilt
#   - preprocessing/tuning/ridge fitting fully rerun
#   - observed statistic uses the SAME first 3 repeats
#   - 1000 permutations
#   - lower MAE is favorable
# ================================================================

from __future__ import annotations

import gc
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from joblib import Parallel, delayed, parallel_backend
from threadpoolctl import threadpool_limits


# ------------------------------------------------
# Frozen severity permutation configuration
# ------------------------------------------------

SEV_PERM_ENGINE = (
    "goal1-severity-permutation-fixed-split-v1.0.0"
)

SEV_PERM_REPEATS = 3

SEV_PERM_ROOT = (
    OUT
    / "severity_permutation_v1_0"
)

SEV_PERM_CHECKPOINT = (
    SEV_PERM_ROOT
    / "severity_permutation_checkpoint.csv"
)

SEV_PERM_TABLES = (
    SEV_PERM_ROOT
    / "tables"
)

SEV_PERM_AUDIT = (
    SEV_PERM_ROOT
    / "audit"
)

for d in [
    SEV_PERM_ROOT,
    SEV_PERM_TABLES,
    SEV_PERM_AUDIT,
]:
    d.mkdir(
        parents=True,
        exist_ok=True,
    )


# ------------------------------------------------
# Hard context gates
# ------------------------------------------------

required_globals = [
    "RUN_MODE",
    "BASE_SEED",
    "OUTER_FOLDS",
    "INNER_FOLDS",
    "N_PERMUTATIONS",
    "PRIMARY_MODEL_SPECS",
    "run_nested_cv",
    "repeat_metric_table",
    "split_manifest",
    "model_checkpoint_paths",
    "safe_read_csv",
    "atomic_write_csv",
    "atomic_write_json",
    "stable_hash",
]

missing = [
    name
    for name in required_globals
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Notebook 10 runtime state is incomplete. "
        f"Missing: {missing}"
    )

if RUN_MODE != "FINAL":
    raise RuntimeError(
        f"Expected FINAL mode; observed {RUN_MODE!r}."
    )

if int(N_PERMUTATIONS) != 1000:
    raise RuntimeError(
        "Expected N_PERMUTATIONS=1000."
    )

if int(OUTER_FOLDS) != 5:
    raise RuntimeError(
        "Expected five outer folds."
    )

if int(INNER_FOLDS) != 5:
    raise RuntimeError(
        "Expected five inner folds."
    )


# ------------------------------------------------
# Exact primary severity population
# ------------------------------------------------

if "fast_primary_frame" in globals():

    sev_perm_frame = (
        fast_primary_frame(
            "severity"
        ).copy()
    )

elif "sev_primary" in globals():

    sev_perm_frame = (
        sev_primary.copy()
    )

else:

    raise RuntimeError(
        "Neither fast_primary_frame('severity') "
        "nor sev_primary exists."
    )


sev_perm_frame[
    "participant_id"
] = (
    sev_perm_frame[
        "participant_id"
    ]
    .astype(str)
    .str.strip()
)

sev_perm_frame = (
    sev_perm_frame
    .sort_values(
        "participant_id"
    )
    .reset_index(
        drop=True
    )
)


if len(sev_perm_frame) != 145:
    raise RuntimeError(
        "Expected exactly 145 primary "
        f"severity participants; found "
        f"{len(sev_perm_frame)}."
    )

if not sev_perm_frame[
    "participant_id"
].is_unique:
    raise RuntimeError(
        "Primary severity population is not "
        "one row per participant."
    )

sev_y = pd.to_numeric(
    sev_perm_frame["y"],
    errors="raise",
).to_numpy(float)

if not np.isfinite(
    sev_y
).all():
    raise RuntimeError(
        "Non-finite severity outcome."
    )

if not (
    (sev_y >= 0)
    & (sev_y <= 12)
).all():
    raise RuntimeError(
        "Severity outcome outside 0–12."
    )


# ------------------------------------------------
# Exact Core-Q ridge specification
# ------------------------------------------------

SEV_COREQ_SPEC = (
    PRIMARY_MODEL_SPECS[
        "Core-Q"
    ]
)

if (
    SEV_COREQ_SPEC.get(
        "kind"
    )
    != "ridge"
):
    raise RuntimeError(
        "Primary Core-Q severity model "
        "is unexpectedly not ridge."
    )


# ------------------------------------------------
# Fixed master split provenance
#
# Severity score never generated this split.
# We use exactly repeats 1–3.
# ------------------------------------------------

sev_split3 = (
    split_manifest.loc[
        split_manifest[
            "repeat"
        ].between(
            1,
            SEV_PERM_REPEATS,
        )
    ]
    .copy()
)

sev_split3[
    "participant_id"
] = (
    sev_split3[
        "participant_id"
    ]
    .astype(str)
    .str.strip()
)

expected_split_rows = (
    224
    * SEV_PERM_REPEATS
)

if len(
    sev_split3
) != expected_split_rows:

    raise RuntimeError(
        "Fixed split manifest does not "
        "contain the expected first "
        "three repeats."
    )

if not (
    sev_split3.groupby(
        [
            "participant_id",
            "repeat",
        ]
    )
    .size()
    .eq(1)
    .all()
):
    raise RuntimeError(
        "Fixed split assignment is not unique."
    )


SEV_SPLIT_HASH = stable_hash(
    sev_split3[
        [
            "participant_id",
            "repeat",
            "outer_fold",
        ]
    ]
    .sort_values(
        [
            "repeat",
            "outer_fold",
            "participant_id",
        ]
    )
    .to_dict(
        "records"
    )
)


# ------------------------------------------------
# Observed Core-Q severity OOF
# ------------------------------------------------

sev_oof_path = (
    model_checkpoint_paths(
        "severity",
        "Core-Q",
    )[
        "oof"
    ]
)

sev_observed_oof = safe_read_csv(
    sev_oof_path,
    required_columns=[
        "participant_id",
        "logical_recording_id",
        "y",
        "prediction",
        "repeat",
        "outer_fold",
        "model",
        "task",
    ],
)

if (
    sev_observed_oof[
        "participant_id"
    ].nunique()
    != 145
):
    raise RuntimeError(
        "Observed Core-Q severity OOF "
        "does not contain 145 participants."
    )

if (
    sev_observed_oof[
        "repeat"
    ].nunique()
    != 10
):
    raise RuntimeError(
        "Observed severity OOF must contain "
        "10 primary repeats."
    )


# Matched observed statistic for formal permutation test.
sev_observed3 = (
    sev_observed_oof.loc[
        sev_observed_oof[
            "repeat"
        ].between(
            1,
            SEV_PERM_REPEATS,
        )
    ]
    .copy()
)

SEV_OBSERVED_MAE_3REP = float(
    repeat_metric_table(
        sev_observed3,
        "severity",
    )[
        "MAE"
    ].mean()
)


# Keep primary 10-repeat estimate separate.
SEV_PRIMARY_MAE_10REP = float(
    repeat_metric_table(
        sev_observed_oof,
        "severity",
    )[
        "MAE"
    ].mean()
)


# ------------------------------------------------
# Provenance signature
# ------------------------------------------------

SEV_PERM_SIGNATURE = stable_hash(
    {
        "engine": SEV_PERM_ENGINE,
        "base_seed": BASE_SEED,
        "outer_folds": OUTER_FOLDS,
        "outer_repeats_for_permutation": (
            SEV_PERM_REPEATS
        ),
        "inner_folds": INNER_FOLDS,
        "n_permutations": N_PERMUTATIONS,
        "population_n": 145,
        "primary_model": (
            "Core-Q ridge"
        ),
        "outcome": (
            "ALSFRS-R bulbar score 0-12"
        ),
        "outer_rule": (
            "fixed Phase-0 master split; "
            "first three repeats"
        ),
        "inner_rule": (
            "severity KFold participant-grouped "
            "inner tuning independent of y"
        ),
        "observed_statistic": (
            "mean participant-level MAE "
            "across true-score repeats 1-3"
        ),
        "null_statistic": (
            "mean participant-level MAE "
            "across identical fixed repeats 1-3"
        ),
        "split_hash": (
            SEV_SPLIT_HASH
        ),
    }
)


atomic_write_json(
    {
        "created_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "status": (
            "READY"
        ),
        "engine": (
            SEV_PERM_ENGINE
        ),
        "signature": (
            SEV_PERM_SIGNATURE
        ),
        "population_n": 145,
        "permutation_count": 1000,
        "permutation_repeats": (
            SEV_PERM_REPEATS
        ),
        "fixed_split_manifest": True,
        "split_manifest_sha256": (
            SEV_SPLIT_HASH
        ),
        "severity_used_to_construct_split": False,
        "fold_safe_qchan_recomputed": True,
        "preprocessing_recomputed": True,
        "inner_tuning_recomputed": True,
        "ridge_refit": True,
        "primary_10_repeat_MAE": (
            SEV_PRIMARY_MAE_10REP
        ),
        "matched_3_repeat_observed_MAE": (
            SEV_OBSERVED_MAE_3REP
        ),
    },
    SEV_PERM_AUDIT
    / "severity_permutation_contract.json",
)


print(
    "=" * 72
)

print(
    "SEVERITY PERMUTATION PREFLIGHT: PASS"
)

print(
    "=" * 72
)

print(
    "Participants:",
    len(sev_perm_frame),
)

print(
    "Primary 10-repeat MAE:",
    f"{SEV_PRIMARY_MAE_10REP:.6f}",
)

print(
    "Matched 3-repeat observed MAE:",
    f"{SEV_OBSERVED_MAE_3REP:.6f}",
)

print(
    "Fixed split hash:",
    SEV_SPLIT_HASH,
)

print(
    "Formal null design:",
    "5 folds x 3 repeats x 1000 permutations",
)

SEVERITY PERMUTATION PREFLIGHT: PASS
Participants: 145
Primary 10-repeat MAE: 1.685235
Matched 3-repeat observed MAE: 1.680926
Fixed split hash: 649220757c2e3a049062e31c76dfa4b14b7b1c85fd5c9f036edff69f955249b0
Formal null design: 5 folds x 3 repeats x 1000 permutations


In [39]:
# ================================================================
# SEVERITY PERMUTATION WORKER + RESTART-SAFE PARALLEL RUNNER
# ================================================================

def _single_severity_permutation(
    permutation: int,
):
    permutation = int(
        permutation
    )

    permuted = (
        sev_perm_frame.copy()
    )

    label_seed = int(
        BASE_SEED
        + 20_000_000
        + permutation
    )

    rng = np.random.default_rng(
        label_seed
    )

    permuted["y"] = (
        rng.permutation(
            permuted[
                "y"
            ].to_numpy(float)
        )
    )

    # IMPORTANT:
    # split_manifest remains fixed.
    # Unlike diagnosis, severity y never
    # determined outer or inner fold allocation.
    with threadpool_limits(
        limits=1
    ):

        null_oof, _, _, _ = (
            run_nested_cv(
                permuted,
                (
                    "Core-Q severity "
                    f"permutation {permutation}"
                ),
                SEV_COREQ_SPEC,
                "severity",
                repeats=SEV_PERM_REPEATS,
                record_splits=False,
            )
        )

    metric_table = (
        repeat_metric_table(
            null_oof,
            "severity",
        )
    )

    null_mae = float(
        metric_table[
            "MAE"
        ].mean()
    )

    if not np.isfinite(
        null_mae
    ):
        raise RuntimeError(
            f"Permutation {permutation}: "
            "non-finite MAE."
        )

    return {
        "permutation": (
            permutation
        ),
        "null_metric": (
            null_mae
        ),
        "label_permutation_seed": (
            label_seed
        ),
        "outer_repeat_seeds": (
            ";".join(
                str(
                    BASE_SEED + r
                )
                for r in range(
                    SEV_PERM_REPEATS
                )
            )
        ),
        "split_manifest_hash": (
            SEV_SPLIT_HASH
        ),
        "engine": (
            SEV_PERM_ENGINE
        ),
        "signature": (
            SEV_PERM_SIGNATURE
        ),
    }


def run_severity_permutations(
    *,
    target: int,
    n_jobs: int = 8,
):
    target = int(target)
    n_jobs = int(n_jobs)

    if not (
        1
        <= target
        <= 1000
    ):
        raise ValueError(
            "target must be 1..1000"
        )

    if n_jobs < 1:
        raise ValueError(
            "n_jobs must be >= 1."
        )

    required = [
        "permutation",
        "null_metric",
        "label_permutation_seed",
        "outer_repeat_seeds",
        "split_manifest_hash",
        "engine",
        "signature",
    ]

    if (
        SEV_PERM_CHECKPOINT
        .exists()
    ):

        existing = safe_read_csv(
            SEV_PERM_CHECKPOINT,
            required_columns=required,
            allow_empty=True,
        )

        if len(existing):

            if set(
                existing[
                    "signature"
                ].astype(str)
            ) != {
                SEV_PERM_SIGNATURE
            }:

                raise RuntimeError(
                    "Existing severity permutation "
                    "checkpoint has incompatible signature."
                )

    else:

        existing = pd.DataFrame(
            columns=required
        )


    done = set(
        pd.to_numeric(
            existing.get(
                "permutation",
                pd.Series(
                    dtype=float
                ),
            ),
            errors="coerce",
        )
        .dropna()
        .astype(int)
    )


    todo = [
        p
        for p in range(
            1,
            target + 1,
        )
        if p not in done
    ]


    if not todo:

        print(
            f"Severity permutations "
            f"1-{target} already complete."
        )

        return (
            existing
            .sort_values(
                "permutation"
            )
            .reset_index(
                drop=True
            ),
            0.0,
            0,
        )


    print(
        f"Severity permutation: "
        f"{len(todo)} new draws | "
        f"target={target} | "
        f"workers={n_jobs} | "
        f"CV=5 folds x "
        f"{SEV_PERM_REPEATS} repeats"
    )


    working = (
        existing.copy()
    )

    total_start = (
        time.perf_counter()
    )


    # Small batches mean a transient worker
    # failure loses little unsaved computation.
    batch_size = max(
        n_jobs * 2,
        8,
    )


    for start in range(
        0,
        len(todo),
        batch_size,
    ):

        batch = todo[
            start:
            start + batch_size
        ]

        batch_start = (
            time.perf_counter()
        )


        with parallel_backend(
            "loky",
            inner_max_num_threads=1,
        ):

            result = Parallel(
                n_jobs=n_jobs,
                batch_size=1,
                verbose=0,
            )(
                delayed(
                    _single_severity_permutation
                )(p)
                for p in batch
            )


        working = (
            pd.concat(
                [
                    working,
                    pd.DataFrame(
                        result
                    ),
                ],
                ignore_index=True,
            )
            .sort_values(
                "permutation"
            )
            .drop_duplicates(
                "permutation",
                keep="last",
            )
            .reset_index(
                drop=True
            )
        )


        atomic_write_csv(
            working,
            SEV_PERM_CHECKPOINT,
            required_columns=required,
        )


        elapsed_batch = (
            time.perf_counter()
            - batch_start
        )


        print(
            "completed through "
            f"permutation "
            f"{int(working['permutation'].max())} "
            f"| checkpoint rows="
            f"{len(working)} "
            f"| batch wall="
            f"{elapsed_batch / 60:.2f} min"
        )


    elapsed = (
        time.perf_counter()
        - total_start
    )


    return (
        working,
        elapsed,
        len(todo),
    )


print(
    "SEVERITY PARALLEL PERMUTATION ENGINE: READY"
)

SEVERITY PARALLEL PERMUTATION ENGINE: READY


In [41]:
sev_benchmark, sev_seconds, sev_n_new = (
    run_severity_permutations(
        target=16,
        n_jobs=8,
    )
)

display(
    sev_benchmark
)

print()
print(
    "New severity permutations:",
    sev_n_new,
)

print(
    "Wall time:",
    f"{sev_seconds / 60:.2f} min",
)

if sev_n_new:
    throughput = (
        sev_n_new
        / sev_seconds
        * 3600
    )

    print(
        "Throughput:",
        f"{throughput:.2f} permutations/hour",
    )

    print(
        "Projected remaining:",
        f"{(1000-len(sev_benchmark))/throughput:.2f} hours",
    )

print()
print(
    "SEVERITY 8-WORKER BENCHMARK: COMPLETE"
)

Severity permutation: 16 new draws | target=16 | workers=8 | CV=5 folds x 3 repeats
completed through permutation 16 | checkpoint rows=16 | batch wall=11.19 min


,permutation,null_metric,label_permutation_seed,outer_repeat_seeds,split_manifest_hash,engine,signature
0,1,2.028207,40260826,20260825;20260826;20260827,649220757c2e3a049062e31c76dfa4b14b7b1c85fd5c9f...,goal1-severity-permutation-fixed-split-v1.0.0,76da846a72ff3984fd24fa89ed1181106cc870f7aa8f53...
1,2,2.054999,40260827,20260825;20260826;20260827,649220757c2e3a049062e31c76dfa4b14b7b1c85fd5c9f...,goal1-severity-permutation-fixed-split-v1.0.0,76da846a72ff3984fd24fa89ed1181106cc870f7aa8f53...
2,3,2.030329,40260828,20260825;20260826;20260827,649220757c2e3a049062e31c76dfa4b14b7b1c85fd5c9f...,goal1-severity-permutation-fixed-split-v1.0.0,76da846a72ff3984fd24fa89ed1181106cc870f7aa8f53...
3,4,2.027707,40260829,20260825;20260826;20260827,649220757c2e3a049062e31c76dfa4b14b7b1c85fd5c9f...,goal1-severity-permutation-fixed-split-v1.0.0,76da846a72ff3984fd24fa89ed1181106cc870f7aa8f53...
4,5,2.083959,40260830,20260825;20260826;20260827,649220757c2e3a049062e31c76dfa4b14b7b1c85fd5c9f...,goal1-severity-permutation-fixed-split-v1.0.0,76da846a72ff3984fd24fa89ed1181106cc870f7aa8f53...
5,6,2.042023,40260831,20260825;20260826;20260827,649220757c2e3a049062e31c76dfa4b14b7b1c85fd5c9f...,goal1-severity-permutation-fixed-split-v1.0.0,76da846a72ff3984fd24fa89ed1181106cc870f7aa8f53...
6,7,2.025544,40260832,20260825;20260826;20260827,649220757c2e3a049062e31c76dfa4b14b7b1c85fd5c9f...,goal1-severity-permutation-fixed-split-v1.0.0,76da846a72ff3984fd24fa89ed1181106cc870f7aa8f53...
7,8,2.015346,40260833,20260825;20260826;20260827,649220757c2e3a049062e31c76dfa4b14b7b1c85fd5c9f...,goal1-severity-permutation-fixed-split-v1.0.0,76da846a72ff3984fd24fa89ed1181106cc870f7aa8f53...
8,9,2.038786,40260834,20260825;20260826;20260827,649220757c2e3a049062e31c76dfa4b14b7b1c85fd5c9f...,goal1-severity-permutation-fixed-split-v1.0.0,76da846a72ff3984fd24fa89ed1181106cc870f7aa8f53...
9,10,2.06626,40260835,20260825;20260826;20260827,649220757c2e3a049062e31c76dfa4b14b7b1c85fd5c9f...,goal1-severity-permutation-fixed-split-v1.0.0,76da846a72ff3984fd24fa89ed1181106cc870f7aa8f53...



New severity permutations: 16
Wall time: 11.19 min
Throughput: 85.76 permutations/hour
Projected remaining: 11.47 hours

SEVERITY 8-WORKER BENCHMARK: COMPLETE


In [42]:
# ================================================================
# SEVERITY — RESILIENT COMPLETION TO 1000
# ================================================================

import gc
import time

from joblib.externals.loky import (
    get_reusable_executor,
)


FINAL_TARGET = 1000
SEV_WORKERS = 8
MAX_CONSECUTIVE_FAILURES = 3

consecutive_failures = 0


while True:

    if SEV_PERM_CHECKPOINT.exists():

        current = pd.read_csv(
            SEV_PERM_CHECKPOINT
        )

        current = (
            current
            .sort_values(
                "permutation"
            )
            .drop_duplicates(
                "permutation",
                keep="last",
            )
            .reset_index(
                drop=True
            )
        )

        completed = int(
            current[
                "permutation"
            ].nunique()
        )

    else:

        completed = 0


    print()
    print(
        "=" * 68
    )

    print(
        "Severity checkpoint status:",
        f"{completed}/{FINAL_TARGET}",
    )

    print(
        "=" * 68
    )


    if completed >= FINAL_TARGET:

        print(
            "Severity target reached."
        )

        break


    try:

        (
            _result,
            _seconds,
            _n_new,
        ) = (
            run_severity_permutations(
                target=FINAL_TARGET,
                n_jobs=SEV_WORKERS,
            )
        )

        consecutive_failures = 0


    except Exception as exc:

        consecutive_failures += 1

        print()
        print(
            "TRANSIENT SEVERITY PARALLEL FAILURE"
        )

        print(
            "Exception:",
            type(exc).__name__,
            str(exc),
        )


        if (
            SEV_PERM_CHECKPOINT
            .exists()
        ):

            safe_now = pd.read_csv(
                SEV_PERM_CHECKPOINT
            )

            completed_now = int(
                safe_now[
                    "permutation"
                ].nunique()
            )

        else:

            completed_now = 0


        print(
            "Checkpoint safely contains:",
            completed_now,
            "permutations",
        )


        if (
            consecutive_failures
            >= MAX_CONSECUTIVE_FAILURES
        ):

            raise RuntimeError(
                "Three consecutive severity "
                "worker failures occurred. "
                "Stop and inspect."
            ) from exc


        try:

            executor = (
                get_reusable_executor()
            )

            executor.shutdown(
                wait=True,
                kill_workers=True,
            )

        except Exception:
            pass


        gc.collect()

        print(
            "Fresh worker pool will be created."
        )

        print(
            "Retrying SAME missing permutation IDs "
            "in 15 seconds..."
        )

        time.sleep(15)


Severity checkpoint status: 16/1000
Severity permutation: 984 new draws | target=1000 | workers=8 | CV=5 folds x 3 repeats
completed through permutation 32 | checkpoint rows=32 | batch wall=11.29 min
completed through permutation 48 | checkpoint rows=48 | batch wall=11.08 min
completed through permutation 64 | checkpoint rows=64 | batch wall=10.94 min
completed through permutation 80 | checkpoint rows=80 | batch wall=11.34 min
completed through permutation 96 | checkpoint rows=96 | batch wall=11.16 min
completed through permutation 112 | checkpoint rows=112 | batch wall=10.98 min
completed through permutation 128 | checkpoint rows=128 | batch wall=10.86 min
completed through permutation 144 | checkpoint rows=144 | batch wall=10.91 min
completed through permutation 160 | checkpoint rows=160 | batch wall=11.28 min
completed through permutation 176 | checkpoint rows=176 | batch wall=11.16 min
completed through permutation 192 | checkpoint rows=192 | batch wall=10.97 min
completed through

## 13. Interpretation guardrail

Only after a **FINAL** run seals successfully may the numerical results be transferred to the manuscript.

If the primary Core-Q model exceeds the participant-level permutation null, the permitted conclusion is:

> Recording-quality characteristics carry reproducible clinical information in unseen participants.

Goal 1 does not establish that this information is technical in origin, spurious, a causal acquisition effect, or a shortcut used by an acoustic model.